# Indic speaker-attributed ASR — diarization + ASR, end to end

99 Indic YouTube clips (12.26 h, 9 scripts, 2–8 speakers). Every stage of the
pipeline, in order, in one notebook:

| stage | what | compute |
|---|---|---|
| 1 | audio: YouTube → 16 kHz mono WAV, trimmed sample-exact | network |
| 2 | reference parsing: spreadsheet → RTTM + speaker-attributed text | CPU |
| 3 | diarization: pyannote 3.1, Sortformer 4spk-v1 (offline + streaming) | **GPU** |
| 4a | ASR: IndicConformer-600M (ONNX, CTC), Whisper large-v3 | **GPU** |
| 4b | language-ID fallback, word → speaker attribution | CPU |
| 5 | speaker relabelling: rule baseline (CPU), Qwen2.5-7B (**GPU**) | mixed |
| 4c/5b | cpWER / WDER, projection back to RTTM, DER / JER | CPU |
| 6 | results table: baseline vs improved, per model, per video | CPU |

## How to run

1. **Runtime → Change runtime type → T4 GPU.**
2. Put the `indic-speaker-asr` data folder in **My Drive** (or set `DRIVE_DIR` below).
3. Add a Colab secret **`HF_TOKEN`** (key icon on the left), switch on **notebook access**, and
   run the setup cell — if the access dialog is missed, the cell reports a `TimeoutException`
   and both gated models are skipped; re-run that cell to recover. The token's account must
   have accepted the conditions of **three** gated repos:
   `pyannote/speaker-diarization-3.1`, `pyannote/segmentation-3.0`, and
   `ai4bharat/indic-conformer-600m-multilingual`.
4. **Runtime → Run all.** To run again after a failure, use **Runtime → Disconnect and
   delete runtime** first, so the run starts from a clean machine.

## The one switch: `FULL_GPU_RUN`

A full GPU pass is over five T4 hours (Whisper 2.5 h, the LLM ~40 min per condition,
pyannote 50 min), which is longer than a free Colab session reliably lasts. So:

- **`FULL_GPU_RUN = False`** (default, ~45 min). Each GPU stage **runs live on the
  first `SMOKE_CLIPS` clips** and is compared with the reference run's output for those
  clips; then the reference run's **full 99-clip GPU outputs are loaded from Drive**.
  Every CPU stage recomputes on all 99 clips, and the final results table is
  **asserted identical** to the committed one.
- **`FULL_GPU_RUN = True`**: the same cells run every GPU stage on all 99 clips, and
  the results are compared (not asserted) against the committed table.

Stage 1 cannot run on Colab: YouTube bot-gates cloud IPs, even with cookies. Audio was
extracted from a home connection with the same script, and this notebook verifies it
(format, exact sample count, not silent) instead of downloading it.

The cached GPU outputs come from the per-stage Kaggle runs whose notebooks, with their
original outputs, are in `notebooks/`; each cached directory is byte-identical to the
corresponding Kaggle output dataset.

## How the notebook is built

- **Every stage runs as a subprocess** of a script written by the `%%writefile` cell at
  the start of its section — the same `pipeline/*.py` files as in the repository. The
  notebook kernel itself imports only the standard library.
- **Each GPU model gets its own Python environment** (a venv layered over Colab's
  packages). NeMo, pyannote.audio, faster-whisper and onnxruntime-gpu pin conflicting
  versions of numpy and cuDNN; isolated, none of them can break another or the scoring
  stack.

In [ ]:
FULL_GPU_RUN = False     # True: every GPU stage on all 99 clips (5+ T4 hours)
SMOKE_CLIPS  = 3         # clips each GPU stage runs live when FULL_GPU_RUN is False
DRIVE_DIR    = "/content/drive/MyDrive/indic-speaker-asr"

In [ ]:
# Standard library only: see "How the notebook is built" above.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    ON_COLAB = True
except ImportError:                      # any other Jupyter: point ASR_DRIVE at the folder
    ON_COLAB = False
    DRIVE_DIR = os.environ.get("ASR_DRIVE", DRIVE_DIR)

DRIVE = Path(DRIVE_DIR)
WORKDIR = Path("/content/indic-speaker-asr") if ON_COLAB else Path(os.environ.get("ASR_WORKDIR", "asr_run")).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)                        # scripts are written here and run with --data data
DATA = WORKDIR / "data"
DATA.mkdir(exist_ok=True)
CACHE = DRIVE / "cache"                  # the reference run's GPU outputs
EXPECTED = DRIVE / "expected"            # the committed results tables

for p in (DRIVE / "youtube_segments_final.xlsx", DRIVE / "manifest.jsonl", DRIVE / "wav", CACHE, EXPECTED):
    assert p.exists(), f"missing {p} -- is the indic-speaker-asr folder in My Drive (DRIVE_DIR)?"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if ON_COLAB and not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as exc:             # secret missing, or notebook access not granted
        # A TimeoutException means the "grant access" dialog was never answered --
        # easy to miss under Run all, and it silently costs two GPU stages.
        print(f"HF_TOKEN not available ({type(exc).__name__}). Both gated models will be skipped:")
        print("  pyannote/speaker-diarization-3.1  (accept its conditions, and segmentation-3.0)")
        print("  ai4bharat/indic-conformer-600m-multilingual")
        print("Add the secret with the key icon, switch on notebook access, then RE-RUN THIS CELL.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

HAVE_GPU = (os.environ.get("ASR_GPU", "1") != "0"     # ASR_GPU=0: CPU half only
            and shutil.which("nvidia-smi") is not None
            and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0)
if FULL_GPU_RUN:
    assert HAVE_GPU, "FULL_GPU_RUN needs a GPU runtime"

REPORT = []

def run(label, args, *, may_fail=False, env=None, python=None):
    # Run one script in a fresh interpreter, streaming its output into the cell.
    print("$ python", " ".join(map(str, args)), flush=True)
    t0 = time.time()
    e = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
         "TQDM_DISABLE": "1", **(env or {})}
    p = subprocess.Popen([str(python or sys.executable)] + [str(a) for a in args], cwd=WORKDIR,
                         env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    mins = (time.time() - t0) / 60
    REPORT.append({"step": label, "status": "ok" if rc == 0 else "FAILED", "exit": rc, "min": mins})
    print(f"[{label}] exit {rc}, {mins:.1f} min", flush=True)
    if rc and not may_fail:
        raise RuntimeError(f"{label} failed (exit {rc}) -- see the log above")
    return rc

def skip(label, why):
    REPORT.append({"step": label, "status": "skipped: " + why, "exit": None, "min": 0.0})
    print(f"[{label}] skipped: {why}")

def gpu_env(name, *packages):
    # A venv for one GPU stack. Colab's own site-packages are appended to its path
    # through a .pth file, so torch and the CUDA wheels are reused rather than
    # downloaded again, while anything pip installs or upgrades here lands in the
    # venv and shadows the system copy for this environment only.
    # Returns the venv's python, or None if the install failed in smoke mode.
    root = WORKDIR / "_venvs" / name
    py = root / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    label = f"install env: {name}"
    t0 = time.time()
    try:
        if not py.exists():
            subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(root)], check=True)
            system_paths = json.loads(subprocess.run(
                [sys.executable, "-c", "import json, site; print(json.dumps(site.getsitepackages() + [site.getusersitepackages()]))"],
                capture_output=True, text=True, check=True).stdout)
            purelib = subprocess.run([str(py), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                                     capture_output=True, text=True, check=True).stdout.strip()
            Path(purelib).mkdir(parents=True, exist_ok=True)
            (Path(purelib) / "_system_site.pth").write_text("\n".join(system_paths) + "\n", encoding="utf-8")
        print(f"[{label}] pip install {' '.join(packages)}", flush=True)
        subprocess.run([str(py), "-m", "pip", "install", "-q", *packages], check=True)
    except subprocess.CalledProcessError as exc:
        REPORT.append({"step": label, "status": "FAILED", "exit": exc.returncode, "min": (time.time() - t0) / 60})
        print(f"[{label}] FAILED (exit {exc.returncode})")
        if FULL_GPU_RUN:
            raise
        return None
    REPORT.append({"step": label, "status": "ok", "exit": 0, "min": (time.time() - t0) / 60})
    return py

def restore(sub):
    # Copy one directory of the reference run's GPU output into data/.
    src, dst = CACHE / sub, DATA / sub
    assert src.is_dir(), f"missing cached output {src}"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n = sum(1 for f in dst.rglob("*") if f.suffix in (".rttm", ".json"))
    print(f"restored {sub}: {n} files")

# A GPU stage that fails in smoke mode is recorded and the notebook carries on,
# because the cached full outputs do not depend on it. In a full run it is fatal.
GPU_MAY_FAIL = not FULL_GPU_RUN
GPU_LIMIT = [] if FULL_GPU_RUN else ["--limit", str(SMOKE_CLIPS)]
GPU_ROOT = "data" if FULL_GPU_RUN else "smoke"   # smoke output never mixes with the full data

print("Colab      :", ON_COLAB, "  Python", sys.version.split()[0])
print("GPU        :", HAVE_GPU)
print("HF_TOKEN   :", bool(HF_TOKEN))
print("mode       :", "FULL GPU RUN (99 clips)" if FULL_GPU_RUN else f"smoke {SMOKE_CLIPS} clips + cached GPU outputs")
print("drive      :", DRIVE)
print("workdir    :", WORKDIR)

In [ ]:
# The CPU scoring stack, into the main environment. pyannote.metrics brings
# pyannote.core; meeteval is cpWER; rapidfuzz is the word alignment behind WDER;
# openpyxl writes the xlsx table. Nothing installs into this environment after here.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyannote.metrics", "meeteval", "rapidfuzz", "openpyxl"], check=True)

### Notebook checks

The verification steps that need numpy, pandas or pyannote live in one small script,
run as subprocesses like the stages themselves: the audio check, the reference hash,
the smoke-vs-reference diarization comparison, and the final comparison with the
committed tables.

In [ ]:
%%writefile nb_checks.py
"""Verification steps for end_to_end.ipynb, run as subprocesses."""
import argparse
import hashlib
import json
import sys
import wave
from pathlib import Path


def audio(args):
    import numpy as np

    recs = {}
    for line in Path(args.manifest).read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            recs[r["clip_id"]] = r                  # append-only: last record per clip wins
    ok = [r for r in recs.values() if r["status"] == "ok"]
    failed = {r["video_id"]: r.get("error_class") for r in recs.values() if r["status"] != "ok"}

    broken, minutes = [], 0.0
    for r in ok:
        with wave.open(str(Path(args.wav_dir) / (r["clip_id"] + ".wav")), "rb") as w:
            fmt, n = (w.getframerate(), w.getnchannels(), w.getsampwidth()), w.getnframes()
            x = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float32) / 32768
        rms_db = float(20 * np.log10(np.sqrt(np.mean(x ** 2)) + 1e-12))
        minutes += n / 16000 / 60
        # The corpus is ~94% speech, so a real clip sits far above -45 dBFS.
        if fmt != (16000, 1, 2) or n != r["expected_samples"] or rms_db < -45:
            broken.append((r["clip_id"], fmt, n - r["expected_samples"], round(rms_db, 1)))

    print(f"{len(ok)} clips, {minutes:.1f} min of audio; not extracted: {failed}")
    for b in broken:
        print("  BROKEN (clip, format, sample delta, rms dBFS):", b)
    if broken or len(ok) != 99 or failed != {"GUVrL5ltiP4": "unavailable"}:
        sys.exit("audio does not match the reference run")
    print("all 99: 16 kHz, mono, 16-bit, exact sample count, not silent")


EXPECTED_REF_SHA256 = "e505186dc1cd00ea79301effbc554dc01951d2aa2ec6d39dc4a739d2a4e36dc1"


def refs(args):
    # Stage 2 is a pure function of the xlsx, so unlike the audio it CAN be checked
    # byte for byte, against the ref/ every later stage consumed. clip_meta.csv is
    # left out: its has_audio column reads the Stage 1 manifest.
    import pandas as pd

    root = Path(args.data) / "ref"
    files = sorted(list((root / "rttm").glob("*.rttm")) + list((root / "segments").glob("*.json")),
                   key=lambda p: (p.parent.name, p.name))
    m = hashlib.sha256()
    for p in files:
        m.update(p.parent.name.encode() + b"/" + p.name.encode() + b"\0")
        m.update(p.read_bytes().replace(b"\r\n", b"\n"))   # the reference digest was taken on Windows
    print(f"this run : {m.hexdigest()}  ({len(files)} files)")
    print(f"expected : {EXPECTED_REF_SHA256}  (200 files)")
    if (m.hexdigest(), len(files)) != (EXPECTED_REF_SHA256, 200):
        sys.exit("ref/ differs from the reference run")
    print("ref/rttm + ref/segments: IDENTICAL to the reference run")

    meta = pd.read_csv(root / "clip_meta.csv")
    print(f"{len(meta)} clips, {meta.duration.sum() / 3600:.2f} h, {int(meta.has_audio.sum())} with audio; "
          f"overlap {meta.overlap_sec.sum() / meta.speech_sec.sum() * 100:.2f}% of speech")


def diar_smoke(args):
    # A diarizer on a GPU is not bit-reproducible, so this reports rather than
    # asserts: DER of the live RTTM, scored against the reference run's RTTM for
    # the same clip -- 0 means identical turns.
    from pyannote.metrics.diarization import DiarizationErrorRate
    from stage3_score import load_rttm

    for system in args.systems:
        live_dir = Path(args.smoke) / "hyp" / system / "rttm"
        for f in sorted(live_dir.glob("*.rttm")) if live_dir.is_dir() else []:
            ref = load_rttm(Path(args.cache) / "hyp" / system / "rttm" / f.name)
            if not ref:
                print(f"{system:18s} {f.stem}  (no reference-run RTTM: out of memory on Kaggle)")
                continue
            der = DiarizationErrorRate(collar=0.0, skip_overlap=False)(ref, load_rttm(f))
            print(f"{system:18s} {f.stem}  DER against the reference run's output: {100 * der:5.2f}%")


def expected(args):
    import pandas as pd

    def canonical(df):
        # Row order follows the order systems were passed on the command line,
        # which differs between this notebook and the Kaggle runs; rows must not.
        keys = [c for c in df.columns if df[c].dtype == object]
        return df.sort_values(keys).reset_index(drop=True)

    ours_dir, exp_dir = Path(args.data) / "results", Path(args.expected)
    diffs = []
    for name in ("diarization_per_clip.csv", "diarization_summary.csv", "asr_per_clip.csv",
                 "asr_summary.csv", "results_per_video.csv"):
        ours, theirs = canonical(pd.read_csv(ours_dir / name)), canonical(pd.read_csv(exp_dir / name))
        try:
            pd.testing.assert_frame_equal(ours, theirs, check_exact=False, rtol=1e-6, atol=1e-6)
            print(f"{name:26s} identical to the committed table  {ours.shape}")
        except AssertionError as exc:
            diffs.append(name)
            print(f"{name:26s} DIFFERS: {str(exc)[:300]}")
    same_md = ((ours_dir / "results_table.md").read_text(encoding="utf-8")
               == (exp_dir / "results_table.md").read_text(encoding="utf-8"))
    print(f"{'results_table.md':26s} {'identical' if same_md else 'DIFFERS'}")
    if diffs or not same_md:
        if args.strict:
            sys.exit("the CPU stages did not reproduce the committed results")
        print("\nDifferences from the committed numbers: GPU non-determinism and model or library "
              "versions in a full GPU run, reported rather than asserted.")
    else:
        print("\nEvery CPU stage reproduced the committed results exactly.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("audio"); a.add_argument("--manifest"); a.add_argument("--wav-dir")
    r = sub.add_parser("refs"); r.add_argument("--data")
    d = sub.add_parser("diar-smoke"); d.add_argument("--smoke"); d.add_argument("--cache")
    d.add_argument("--systems", nargs="+")
    e = sub.add_parser("expected"); e.add_argument("--data"); e.add_argument("--expected")
    e.add_argument("--strict", action="store_true")
    args = ap.parse_args()
    {"audio": audio, "refs": refs, "diar-smoke": diar_smoke, "expected": expected}[args.cmd](args)

## Stage 1 — audio extraction

`stage1_extract.py` downloads each video once with yt-dlp, cuts every window with
ffmpeg **output seeking** (`-ss` after `-i`, which decodes up to the mark and is
sample-exact, where input seeking snaps to a keyframe and drifts by up to a second),
and writes 16 kHz mono 16-bit WAV. The manifest is append-only and resumable.

**It is not re-run here.** YouTube bot-gates Colab and Kaggle IPs; from Kaggle, adding
cookies moved the failure to "The page needs to be reloaded" (yt-dlp #17389), and the
`web_embedded` client then served no audio stream. The reference run, from a home
connection, extracted **99 of 100 clips**; `GUVrL5ltiP4` has been removed from YouTube.

What this notebook does instead is copy that audio from Drive to local disk and
**verify** it. There is no audio checksum, because YouTube can serve a different
encode to a different client; what the trim controls is length, and length is exact.

The command that produced the audio:

    python stage1_extract.py --input youtube_segments_final.xlsx --out data --workers 3

In [ ]:
%%writefile stage1_extract.py
#!/usr/bin/env python3
"""
Stage 1 -- YouTube audio extraction.

Reads the segment table, downloads the audio track for each unique video_id,
cuts each [start_sec, end_sec] window to a 16 kHz mono PCM WAV, and records one
line per clip in an append-only JSONL manifest.

Design notes:
  * Network work is keyed on video_id, cutting is keyed on clip_id, so a video
    contributing several windows is downloaded exactly once.
  * The trim uses ffmpeg OUTPUT seeking (-ss/-t placed after -i). ffmpeg decodes
    from zero and discards samples up to the mark, which is sample-exact.
    Input seeking (-ss before -i) snaps to a keyframe and can drift ~1s.
  * Every cut is verified against round((end-start) * 16000) samples. The
    verification is the deliverable, not the absence of a traceback.
  * The manifest is append-only. A killed session loses at most its last line.

Usage:
    python stage1_extract.py --input youtube_segments_final.xlsx --out /kaggle/working/data
    python stage1_extract.py --input ... --out ... --limit 10        # dev subset
    python stage1_extract.py --input ... --out ... --retry-permanent # force retry
    python stage1_extract.py --input ... --out ... --report-only     # summary only
"""

from __future__ import annotations

import argparse
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import wave
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

TARGET_SR = 16_000
TARGET_CHANNELS = 1
SAMPLE_TOLERANCE = 16          # +/- 1 ms at 16 kHz; anything more is real drift
REQUIRED_COLUMNS = [
    "video_id", "youtube_link", "start_sec", "end_sec",
    "diarization_segments", "asr_segments",
]

# Which YouTube player clients yt-dlp should try. This is the single most
# version-sensitive knob in the whole stage -- what defeats bot-gating changes
# every few weeks. Override without editing code:
#     export YTDLP_PLAYER_CLIENTS="tv,web_safari,ios"
# Set to "" to let yt-dlp use its own defaults.
PLAYER_CLIENTS = os.environ.get("YTDLP_PLAYER_CLIENTS", "default,tv,web_safari")

# Optional Netscape-format cookie jar; the reliable answer to bot-gating.
#     export YTDLP_COOKIES=/kaggle/input/yt-cookies/cookies.txt
COOKIES_FILE = os.environ.get("YTDLP_COOKIES", "")

# How to invoke yt-dlp. Prefer the console script; fall back to the module, which
# is what you get when pip installs into a Scripts/bin dir that is not on PATH.
def _resolve_ytdlp() -> list[str]:
    override = os.environ.get("YTDLP_CMD")
    if override:
        return override.split()
    if shutil.which("yt-dlp"):
        return ["yt-dlp"]
    try:
        probe = subprocess.run([sys.executable, "-m", "yt_dlp", "--version"],
                               capture_output=True, text=True, timeout=60)
        if probe.returncode == 0:
            return [sys.executable, "-m", "yt_dlp"]
    except Exception:
        pass
    return []


# Failure classes we will NOT retry on a rerun -- the video is simply gone.
PERMANENT_FAILURES = {"unavailable", "private", "removed", "age_gated", "no_audio"}

# Populated by preflight().
YTDLP_CMD: list[str] = []


# --------------------------------------------------------------------------
# Manifest records
# --------------------------------------------------------------------------

@dataclass
class ClipRecord:
    clip_id: str
    video_id: str
    youtube_link: str
    start_sec: float
    end_sec: float
    status: str                      # ok | short_source | failed
    wav_path: str | None = None
    n_samples: int | None = None
    expected_samples: int | None = None
    sample_delta: int | None = None
    sample_rate: int | None = None
    duration_sec: float | None = None
    error_class: str | None = None
    error_msg: str | None = None
    ts: str = ""

    def __post_init__(self):
        if not self.ts:
            self.ts = datetime.now(timezone.utc).isoformat(timespec="seconds")


class Manifest:
    """Append-only JSONL manifest with an in-memory index for resume."""

    def __init__(self, path: Path):
        self.path = path
        self._lock = threading.Lock()
        self.records: dict[str, dict] = {}
        if path.exists():
            with path.open("r", encoding="utf-8") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        # Truncated final line from a killed session. Expected.
                        continue
                    self.records[rec["clip_id"]] = rec

    def append(self, rec: ClipRecord) -> None:
        with self._lock:
            with self.path.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(asdict(rec), ensure_ascii=False) + "\n")
                fh.flush()
                os.fsync(fh.fileno())
            self.records[rec.clip_id] = asdict(rec)

    def should_skip(self, clip_id: str, retry_permanent: bool) -> bool:
        """True if this clip is already done (and still verifiably done)."""
        rec = self.records.get(clip_id)
        if rec is None:
            return False

        if rec["status"] in ("ok", "short_source"):
            # Trust but verify: the WAV must still exist with the right length.
            wav = Path(rec["wav_path"]) if rec.get("wav_path") else None
            if wav is None or not wav.exists():
                return False
            try:
                if wav_num_frames(wav) != rec.get("n_samples"):
                    return False
            except Exception:
                return False
            return True

        if rec["status"] == "failed":
            if retry_permanent:
                return False
            return rec.get("error_class") in PERMANENT_FAILURES

        return False


# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------

def clip_id_for(video_id: str, start: float, end: float) -> str:
    """Stable per-window identity. Millisecond precision keeps it filename-safe."""
    return f"{video_id}__{int(round(start * 1000)):09d}_{int(round(end * 1000)):09d}"


def wav_num_frames(path: Path) -> int:
    with wave.open(str(path), "rb") as wf:
        return wf.getnframes()


def wav_info(path: Path) -> tuple[int, int, int]:
    """(n_frames, sample_rate, n_channels)"""
    with wave.open(str(path), "rb") as wf:
        return wf.getnframes(), wf.getframerate(), wf.getnchannels()


def classify_error(text: str) -> str:
    """Map yt-dlp / ffmpeg stderr onto a failure class we can act on."""
    t = (text or "").lower()
    checks = [
        ("bot_gated",   [r"sign in to confirm", r"not a bot", r"confirm you.re not a bot"]),
        ("geo_blocked", [r"not available in your country", r"geo restrict",
                         r"blocked it in your country"]),
        ("private",     [r"private video", r"this video is private"]),
        ("removed",     [r"removed by the uploader", r"account associated with this video has been terminated",
                         r"video has been removed"]),
        ("unavailable", [r"video unavailable", r"is unavailable", r"does not exist",
                         r"has been deleted", r"members-only", r"join this channel"]),
        ("age_gated",   [r"age-restricted", r"inappropriate for some users"]),
        ("no_audio",    [r"requested format is not available", r"no audio",
                         r"only images are available"]),
        ("network",     [r"timed out", r"timeout", r"connection reset",
                         r"temporary failure in name resolution", r"unable to download",
                         r"http error 5\d\d", r"429", r"too many requests"]),
    ]
    for cls, patterns in checks:
        for p in patterns:
            if re.search(p, t):
                return cls
    return "unknown"


def run(cmd: list[str], timeout: int) -> subprocess.CompletedProcess:
    return subprocess.run(
        cmd, capture_output=True, text=True, timeout=timeout,
        encoding="utf-8", errors="replace",
    )


class StageError(Exception):
    def __init__(self, error_class: str, message: str):
        super().__init__(message)
        self.error_class = error_class
        self.message = message


# --------------------------------------------------------------------------
# CSV loading
# --------------------------------------------------------------------------

def read_table(path: Path) -> pd.DataFrame:
    """Load the segment table from .xlsx/.xls or .csv/.tsv, dispatching on suffix."""
    suffix = path.suffix.lower()
    if suffix in (".xlsx", ".xlsm", ".xls"):
        try:
            return pd.read_excel(path)
        except ImportError as exc:
            raise SystemExit(f"Reading {suffix} needs openpyxl: pip install openpyxl  ({exc})")
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)


def load_table(path: Path, limit: int | None, only_ids: set[str] | None) -> pd.DataFrame:
    df = read_table(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise SystemExit(
            f"CSV is missing required columns: {missing}\n"
            f"Columns present: {list(df.columns)}\n"
            "Stage 1 was written against the expected segment-table schema; "
            "if the real CSV differs, fix the mapping before going further."
        )

    n_before = len(df)
    df["start_sec"] = pd.to_numeric(df["start_sec"], errors="coerce")
    df["end_sec"] = pd.to_numeric(df["end_sec"], errors="coerce")

    bad = df["start_sec"].isna() | df["end_sec"].isna() | (df["end_sec"] <= df["start_sec"])
    if bad.any():
        print(f"[warn] dropping {int(bad.sum())} row(s) with unusable start/end:", file=sys.stderr)
        for _, r in df[bad].head(10).iterrows():
            print(f"        {r['video_id']}  start={r['start_sec']}  end={r['end_sec']}", file=sys.stderr)
        df = df[~bad].copy()

    df["clip_id"] = [
        clip_id_for(str(r.video_id), float(r.start_sec), float(r.end_sec))
        for r in df.itertuples()
    ]

    dupes = int(df["clip_id"].duplicated().sum())
    if dupes:
        print(f"[warn] {dupes} duplicate clip_id row(s); keeping first of each", file=sys.stderr)
        df = df.drop_duplicates(subset="clip_id", keep="first").copy()

    if only_ids:
        df = df[df["video_id"].astype(str).isin(only_ids)].copy()
    if limit:
        df = df.head(limit).copy()

    print(f"[csv ] {n_before} rows in -> {len(df)} clips, "
          f"{df['video_id'].nunique()} unique videos, "
          f"{df['end_sec'].sub(df['start_sec']).sum() / 3600:.2f}h of audio requested "
          f"(~{df['end_sec'].sub(df['start_sec']).sum() * TARGET_SR * 2 / 1e9:.2f} GB of WAV)")
    return df


# --------------------------------------------------------------------------
# Download
# --------------------------------------------------------------------------

def download_audio(video_id: str, url: str, raw_dir: Path, timeout: int) -> Path:
    """Fetch the best audio-only stream for one video. Returns the raw file path."""
    existing = sorted(raw_dir.glob(f"{video_id}.*"))
    if existing:
        return existing[0]

    out_tmpl = str(raw_dir / "%(id)s.%(ext)s")
    cmd = [
        *YTDLP_CMD,
        "--no-playlist",
        "--no-progress",
        "--no-warnings",
        "-f", "bestaudio/best",
        "--retries", "3",
        "--fragment-retries", "5",
        "--socket-timeout", "30",
        "--sleep-requests", "1",
        "-o", out_tmpl,
        # --print implies --simulate in current yt-dlp, so --no-simulate is
        # required to both download AND report the resulting path.
        "--no-simulate",
        "--print", "after_move:filepath",
    ]
    if PLAYER_CLIENTS:
        cmd += ["--extractor-args", f"youtube:player_client={PLAYER_CLIENTS}"]
    if COOKIES_FILE:
        cmd += ["--cookies", COOKIES_FILE]
    cmd.append(url)

    try:
        proc = run(cmd, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise StageError("network", f"yt-dlp timed out after {timeout}s")

    if proc.returncode != 0:
        blob = (proc.stderr or "") + "\n" + (proc.stdout or "")
        raise StageError(classify_error(blob), blob.strip()[-600:])

    # Preferred: the path yt-dlp printed. Fallback: glob, in case --print
    # semantics have shifted again.
    path = None
    for line in reversed((proc.stdout or "").splitlines()):
        cand = Path(line.strip())
        if line.strip() and cand.exists():
            path = cand
            break
    if path is None:
        found = sorted(raw_dir.glob(f"{video_id}.*"))
        path = found[0] if found else None
    if path is None:
        raise StageError("unknown", "yt-dlp exited 0 but produced no file")
    return path


# --------------------------------------------------------------------------
# Cut + verify
# --------------------------------------------------------------------------

def cut_clip(raw: Path, start: float, end: float, out_wav: Path, timeout: int) -> tuple[int, int, int]:
    """Cut [start, end) to 16 kHz mono PCM WAV. Returns (frames, sr, channels)."""
    duration = end - start
    tmp = out_wav.with_suffix(".tmp.wav")
    cmd = [
        "ffmpeg", "-nostdin", "-y", "-loglevel", "error",
        "-i", str(raw),
        # -ss/-t AFTER -i == output seeking == decode-and-discard == sample exact.
        "-ss", f"{start:.6f}",
        "-t", f"{duration:.6f}",
        "-map", "0:a:0",
        "-vn",
        "-ac", str(TARGET_CHANNELS),
        "-ar", str(TARGET_SR),
        "-c:a", "pcm_s16le",
        str(tmp),
    ]
    try:
        proc = run(cmd, timeout=timeout)
    except subprocess.TimeoutExpired:
        tmp.unlink(missing_ok=True)
        raise StageError("ffmpeg", f"ffmpeg timed out after {timeout}s")

    if proc.returncode != 0 or not tmp.exists():
        tmp.unlink(missing_ok=True)
        raise StageError("ffmpeg", (proc.stderr or "ffmpeg produced no output").strip()[-600:])

    try:
        frames, sr, ch = wav_info(tmp)
    except Exception as exc:
        tmp.unlink(missing_ok=True)
        raise StageError("ffmpeg", f"output WAV unreadable: {exc}")

    # Atomic publish: downstream stages never observe a half-written WAV.
    os.replace(tmp, out_wav)
    return frames, sr, ch


def process_clip(row, raw: Path, wav_dir: Path, timeout: int) -> ClipRecord:
    start, end = float(row.start_sec), float(row.end_sec)
    expected = int(round((end - start) * TARGET_SR))
    out_wav = wav_dir / f"{row.clip_id}.wav"

    frames, sr, ch = cut_clip(raw, start, end, out_wav, timeout)
    delta = frames - expected

    if sr != TARGET_SR or ch != TARGET_CHANNELS:
        raise StageError("ffmpeg", f"wrong format: sr={sr} ch={ch}")

    # A short result almost always means end_sec runs past the true video
    # duration. Keep the audio, but mark it so Stage 3+ can exclude or
    # renormalise rather than silently scoring against a truncated clip.
    status = "ok" if abs(delta) <= SAMPLE_TOLERANCE else "short_source"

    return ClipRecord(
        clip_id=row.clip_id,
        video_id=str(row.video_id),
        youtube_link=str(row.youtube_link),
        start_sec=start,
        end_sec=end,
        status=status,
        wav_path=str(out_wav),
        n_samples=frames,
        expected_samples=expected,
        sample_delta=delta,
        sample_rate=sr,
        duration_sec=round(frames / sr, 6),
        error_class=None if status == "ok" else "short_source",
        error_msg=None if status == "ok" else f"{delta:+d} samples vs expected {expected}",
    )


# --------------------------------------------------------------------------
# Per-video worker
# --------------------------------------------------------------------------

def process_video(video_id, rows, raw_dir, wav_dir, manifest, args) -> list[ClipRecord]:
    """Download one video once, then cut every window the CSV asks for from it."""
    out: list[ClipRecord] = []
    raw = None
    try:
        raw = download_audio(video_id, str(rows[0].youtube_link), raw_dir, args.download_timeout)
    except StageError as exc:
        for row in rows:
            rec = ClipRecord(
                clip_id=row.clip_id, video_id=str(video_id),
                youtube_link=str(row.youtube_link),
                start_sec=float(row.start_sec), end_sec=float(row.end_sec),
                status="failed", error_class=exc.error_class, error_msg=exc.message,
            )
            manifest.append(rec)
            out.append(rec)
        return out

    for row in rows:
        try:
            rec = process_clip(row, raw, wav_dir, args.ffmpeg_timeout)
        except StageError as exc:
            rec = ClipRecord(
                clip_id=row.clip_id, video_id=str(video_id),
                youtube_link=str(row.youtube_link),
                start_sec=float(row.start_sec), end_sec=float(row.end_sec),
                status="failed", error_class=exc.error_class, error_msg=exc.message,
            )
        manifest.append(rec)
        out.append(rec)

    if not args.keep_raw and raw is not None:
        raw.unlink(missing_ok=True)
    return out


# --------------------------------------------------------------------------
# Reporting
# --------------------------------------------------------------------------

def print_summary(manifest: Manifest, df: pd.DataFrame) -> None:
    recs = [manifest.records[c] for c in df["clip_id"] if c in manifest.records]
    status_counts = Counter(r["status"] for r in recs)
    total = len(df)

    print("\n" + "=" * 68)
    print("STAGE 1 SUMMARY")
    print("=" * 68)
    print(f"  clips requested   : {total}")
    for st in ("ok", "short_source", "failed"):
        n = status_counts.get(st, 0)
        pct = f"  ({n / total * 100:5.1f}%)" if total else ""
        print(f"  {st:<18}: {n:>4}{pct}")
    missing = total - len(recs)
    if missing:
        print(f"  {'not attempted':<18}: {missing:>4}")

    fails = [r for r in recs if r["status"] == "failed"]
    if fails:
        print("\n  failure classes:")
        for cls, n in Counter(r["error_class"] for r in fails).most_common():
            tag = "permanent" if cls in PERMANENT_FAILURES else "retryable"
            print(f"    {cls:<14} {n:>4}   ({tag})")

    good = [r for r in recs if r["status"] in ("ok", "short_source")]
    if good:
        deltas = [abs(r["sample_delta"]) for r in good]
        exact = sum(1 for d in deltas if d <= SAMPLE_TOLERANCE)
        secs = sum(r["duration_sec"] for r in good)
        print(f"\n  trim accuracy     : {exact}/{len(good)} clips sample-exact "
              f"(|delta| <= {SAMPLE_TOLERANCE} samples = 1 ms)")
        print(f"  worst |delta|     : {max(deltas)} samples "
              f"({max(deltas) / TARGET_SR * 1000:.1f} ms)")
        print(f"  audio on disk     : {secs / 60:.1f} min across {len(good)} clips")

    drifted = [r for r in good if abs(r["sample_delta"]) > SAMPLE_TOLERANCE]
    if drifted:
        print(f"\n  [!] {len(drifted)} clip(s) off-length. Inspect before trusting Stage 3:")
        for r in sorted(drifted, key=lambda x: -abs(x["sample_delta"]))[:8]:
            print(f"      {r['clip_id']}  {r['sample_delta']:+d} samples "
                  f"({r['sample_delta'] / TARGET_SR:+.3f}s)")
    print("=" * 68)


# --------------------------------------------------------------------------

def preflight() -> None:
    global YTDLP_CMD
    YTDLP_CMD = _resolve_ytdlp()
    if not YTDLP_CMD:
        raise SystemExit(
            "yt-dlp not found (neither on PATH nor as an importable module).\n"
            "  pip install -qU yt-dlp\n"
            "If it is installed but not on PATH, set YTDLP_CMD, e.g.\n"
            '  export YTDLP_CMD="python -m yt_dlp"'
        )
    if shutil.which("ffmpeg") is None:
        raise SystemExit(
            "'ffmpeg' not found on PATH. On Kaggle it is preinstalled; "
            "locally, install it and reopen the shell."
        )
    for label, cmd in (("yt-dlp", YTDLP_CMD + ["--version"]),
                       ("ffmpeg", ["ffmpeg", "-version"])):
        try:
            print(f"[env ] {label}: {run(cmd, timeout=60).stdout.splitlines()[0]}")
        except Exception:
            pass
    if COOKIES_FILE:
        print(f"[env ] cookies: {COOKIES_FILE} "
              f"({'found' if Path(COOKIES_FILE).exists() else 'MISSING'})")
    print(f"[env ] player_clients: {PLAYER_CLIENTS or '(yt-dlp default)'}")


def main() -> int:
    ap = argparse.ArgumentParser(description="Stage 1: extract 16 kHz mono WAV clips from YouTube.")
    ap.add_argument("--input", "--csv", dest="input", required=True, type=Path,
                    help="segment table: .xlsx, .csv or .tsv")
    ap.add_argument("--out", required=True, type=Path, help="output root (wav/, raw/, manifest.jsonl)")
    ap.add_argument("--workers", type=int, default=3,
                    help="parallel video downloads; >4 raises bot-gating risk")
    ap.add_argument("--limit", type=int, default=None, help="first N clips only (dev subset)")
    ap.add_argument("--only-ids", default=None, help="comma-separated video_ids")
    ap.add_argument("--keep-raw", action="store_true", help="do not delete source audio after cutting")
    ap.add_argument("--retry-permanent", action="store_true",
                    help="also retry failures classed as permanent")
    ap.add_argument("--download-timeout", type=int, default=900)
    ap.add_argument("--ffmpeg-timeout", type=int, default=600)
    ap.add_argument("--report-only", action="store_true",
                    help="print summary from manifest, do nothing else")
    args = ap.parse_args()

    out_root: Path = args.out
    wav_dir, raw_dir = out_root / "wav", out_root / "raw"
    for d in (out_root, wav_dir, raw_dir):
        d.mkdir(parents=True, exist_ok=True)

    only_ids = set(args.only_ids.split(",")) if args.only_ids else None
    df = load_table(args.input, args.limit, only_ids)
    manifest = Manifest(out_root / "manifest.jsonl")

    if args.report_only:
        print_summary(manifest, df)
        return 0

    preflight()

    by_video: dict[str, list] = defaultdict(list)
    n_skipped = 0
    for row in df.itertuples():
        if manifest.should_skip(row.clip_id, args.retry_permanent):
            n_skipped += 1
            continue
        by_video[str(row.video_id)].append(row)

    if n_skipped:
        print(f"[run ] resuming: {n_skipped} clip(s) already done, skipping")
    if not by_video:
        print("[run ] nothing to do.")
        print_summary(manifest, df)
        return 0

    n_todo = sum(len(v) for v in by_video.values())
    print(f"[run ] {n_todo} clip(s) across {len(by_video)} video(s), {args.workers} worker(s)\n")

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {
            pool.submit(process_video, vid, rows, raw_dir, wav_dir, manifest, args): vid
            for vid, rows in by_video.items()
        }
        for fut in as_completed(futures):
            vid = futures[fut]
            try:
                recs = fut.result()
            except Exception as exc:                      # worker crash, not a clip failure
                print(f"[ERR ] {vid}: worker crashed: {exc!r}", file=sys.stderr)
                continue
            done += len(recs)
            for r in recs:
                if r.status == "ok":
                    print(f"[ ok ] {r.clip_id}  {r.duration_sec:.3f}s  ({r.sample_delta:+d} samp)")
                elif r.status == "short_source":
                    print(f"[shrt] {r.clip_id}  {r.error_msg}")
                else:
                    tail = (r.error_msg or "").splitlines()
                    print(f"[FAIL] {r.clip_id}  [{r.error_class}] "
                          f"{(tail[-1] if tail else '')[:110]}")
            print(f"       ---- {done}/{n_todo} clips, {time.time() - t0:.0f}s elapsed")

    print_summary(manifest, df)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# Local disk, not the Drive mount: every GPU stage reads all 99 files, and the
# Drive FUSE mount is far slower for that than /content.
AUDIO = WORKDIR / "wav" if ON_COLAB else DRIVE / "wav"
if ON_COLAB and len(list(AUDIO.glob("*.wav"))) != 99:
    shutil.copytree(DRIVE / "wav", AUDIO, dirs_exist_ok=True)
shutil.copy(DRIVE / "manifest.jsonl", DATA / "manifest.jsonl")
print(AUDIO, len(list(AUDIO.glob("*.wav"))), "wavs")

run("stage 1: verify audio", ["nb_checks.py", "audio", "--manifest", DATA / "manifest.jsonl", "--wav-dir", AUDIO])

## Stage 2 — reference parsing

A pure function of the spreadsheet, so unlike the audio it is checked **byte for
byte** against the `ref/` every later stage consumed. What it decides, not just
formats:

- the two label columns are joined by index, and a turn-count or timestamp
  disagreement stops the run rather than giving one speaker's words to another;
- turns are clipped to the window (88 truncated, 38 outside it dropped);
- overlap counts only time with two or more *distinct* speakers.

**Ground truth never enters the pipeline.** `ref/` is read by the scorers and by
diagnostics labelled *oracle*, never by a model or a correction step.

In [ ]:
%%writefile stage2_parse_refs.py
#!/usr/bin/env python3
"""
Stage 2 -- Parse ground-truth labels into scoring-ready references.

Reads the segment table and turns the two label columns into:

    ref/rttm/<clip_id>.rttm    reference diarization, one file per clip
    ref/all.rttm               all clips concatenated (some scorers want one file)
    ref/segments/<clip_id>.json  speaker <-> text join, one record per turn
    ref/clip_meta.csv          per-clip conditions for the Stage 6 breakdown
    ref/stage2_report.json     anomaly counts for the writeup

Design notes:
  * `diarization_segments` and `asr_segments` are promised to share boundaries in
    the same order. That is ASSERTED here, not assumed -- if the promise ever
    breaks we fail loudly rather than silently misattributing text to speakers.
  * Reference turns are clipped to [0, end_sec - start_sec]. The task is to
    use only that window, so labels running past it are truncated rather than
    padding the audio. Truncation hits reference and hypothesis identically and
    introduces no differential bias.
  * Overlap is computed SPEAKER-AWARE: only regions with >=2 distinct speakers
    count. Counting all interval pairs (including adjacent same-speaker turns)
    overstates it.
  * Text is stored RAW. The reference uses a gloss convention -- native script
    followed by the English/numeric source, e.g. "कॉफी(coffee)", "एक(1)" -- which
    touches ~16% of tokens. Normalising that is a scoring decision and belongs in
    Stage 4, where it can be applied to reference and hypothesis alike.
  * Idempotent rather than checkpointed: it rewrites its output tree each run and
    completes in seconds, so a killed run leaves no half-state.

Usage:
    python stage2_parse_refs.py --input youtube_segments_final.xlsx --out data
    python stage2_parse_refs.py --input ... --out data --manifest data/manifest.jsonl
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import shutil
import sys
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

# --------------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "video_id", "youtube_link", "start_sec", "end_sec",
    "diarization_segments", "asr_segments",
]

# "Speaker A [12.34-56.78]"
DIAR_RE = re.compile(r"^\s*(?P<spk>.+?)\s*\[\s*(?P<a>-?\d+(?:\.\d+)?)\s*-\s*(?P<b>-?\d+(?:\.\d+)?)\s*\]\s*$")
# "[12.34-56.78] some text"
ASR_RE = re.compile(r"^\s*\[\s*(?P<a>-?\d+(?:\.\d+)?)\s*-\s*(?P<b>-?\d+(?:\.\d+)?)\s*\]\s*(?P<text>.*)$", re.S)

MIN_SEG_DUR = 0.01          # segments shorter than this after clipping are noise
TIME_TOL = 1e-6             # tolerance when asserting diar/asr timestamps agree

# Unicode script -> language label. Devanagari covers both Hindi and Marathi and
# cannot be split by script alone; kept as one bucket and flagged as such.
SCRIPT_TO_LANG = {
    "DEVANAGARI": "Devanagari (hi/mr)",
    "BENGALI": "Bengali",
    "GUJARATI": "Gujarati",
    "GURMUKHI": "Punjabi",
    "KANNADA": "Kannada",
    "MALAYALAM": "Malayalam",
    "ORIYA": "Odia",
    "TAMIL": "Tamil",
    "TELUGU": "Telugu",
}


@dataclass
class Anomalies:
    inverted: list = field(default_factory=list)          # end < start
    clipped_tail: list = field(default_factory=list)      # truncated at window end
    dropped_outside: list = field(default_factory=list)   # entirely past window
    dropped_tiny: list = field(default_factory=list)      # < MIN_SEG_DUR after clip
    empty_text: int = 0
    empty_clips: list = field(default_factory=list)       # no usable turns left


# --------------------------------------------------------------------------
# Parsing
# --------------------------------------------------------------------------

def split_turns(blob: str) -> list[str]:
    """Split the pipe-delimited turn list. Asserts no stray '|' inside text."""
    return [p for p in str(blob).split("|")]


def parse_diarization(blob: str, clip_id: str) -> list[tuple[str, float, float]]:
    out = []
    for i, part in enumerate(split_turns(blob)):
        if not part.strip():
            continue
        m = DIAR_RE.match(part)
        if not m:
            raise ValueError(f"{clip_id}: unparseable diarization turn {i}: {part!r}")
        out.append((m.group("spk").strip(), float(m.group("a")), float(m.group("b"))))
    return out


def parse_asr(blob: str, clip_id: str) -> list[tuple[float, float, str]]:
    out = []
    for i, part in enumerate(split_turns(blob)):
        if not part.strip():
            continue
        m = ASR_RE.match(part)
        if not m:
            raise ValueError(f"{clip_id}: unparseable asr turn {i}: {part!r}")
        out.append((float(m.group("a")), float(m.group("b")), m.group("text").strip()))
    return out


def normalise_speaker(label: str) -> str:
    """RTTM is whitespace-delimited, so a space in the speaker name corrupts it."""
    return re.sub(r"\s+", "_", label.strip())


# --------------------------------------------------------------------------
# Clipping + geometry
# --------------------------------------------------------------------------

def clip_turns(turns, duration: float, clip_id: str, anom: Anomalies):
    """Clip reference turns to [0, duration], recording every adjustment."""
    kept = []
    for spk, a, b, text in turns:
        if b < a:
            anom.inverted.append({"clip_id": clip_id, "speaker": spk, "start": a, "end": b})
            continue
        if a >= duration:
            anom.dropped_outside.append({"clip_id": clip_id, "speaker": spk,
                                         "start": a, "end": b, "duration": duration})
            continue
        na, nb = max(0.0, a), min(duration, b)
        if nb - a != b - a or na != a:
            anom.clipped_tail.append({"clip_id": clip_id, "speaker": spk,
                                      "orig": [a, b], "clipped": [na, nb],
                                      "lost_sec": round((b - nb) + (na - a), 3)})
        if nb - na < MIN_SEG_DUR:
            anom.dropped_tiny.append({"clip_id": clip_id, "speaker": spk,
                                      "start": na, "end": nb})
            continue
        kept.append((spk, na, nb, text))
    return kept


def speaker_aware_overlap(turns) -> float:
    """Seconds where >=2 DISTINCT speakers are simultaneously active.

    Builds a sweep over boundary points and counts distinct speakers in each
    elementary interval. Same-speaker adjacent or nested turns do not count.
    """
    if not turns:
        return 0.0
    points = sorted({t for _, a, b, _ in turns for t in (a, b)})
    total = 0.0
    for lo, hi in zip(points, points[1:]):
        if hi <= lo:
            continue
        mid = (lo + hi) / 2.0
        active = {spk for spk, a, b, _ in turns if a <= mid < b}
        if len(active) >= 2:
            total += hi - lo
    return total


def union_speech(turns) -> float:
    """Seconds with at least one speaker active (overlap counted once)."""
    iv = sorted((a, b) for _, a, b, _ in turns)
    total, cur_a, cur_b = 0.0, None, None
    for a, b in iv:
        if cur_a is None:
            cur_a, cur_b = a, b
        elif a <= cur_b:
            cur_b = max(cur_b, b)
        else:
            total += cur_b - cur_a
            cur_a, cur_b = a, b
    if cur_a is not None:
        total += cur_b - cur_a
    return total


def detect_language(text: str) -> str:
    """Majority Unicode script of the reference text, as a language proxy.

    Bracketed timestamps and parenthetical English glosses are stripped first --
    otherwise the Latin glosses skew the vote on every clip.
    """
    body = re.sub(r"\[[^\]]*\]|\([^)]*\)", " ", text)
    counts = Counter()
    for ch in body:
        if not ch.isalpha():
            continue
        try:
            script = unicodedata.name(ch).split()[0]
        except ValueError:
            continue
        counts[script] += 1
    if not counts:
        return "unknown"
    top = counts.most_common(1)[0][0]
    return SCRIPT_TO_LANG.get(top, top.title())


# --------------------------------------------------------------------------
# Writers
# --------------------------------------------------------------------------

def rttm_lines(clip_id: str, turns) -> list[str]:
    """SPEAKER <file> <chan> <onset> <dur> <NA> <NA> <spk> <NA> <NA>"""
    lines = []
    for spk, a, b, _ in turns:
        lines.append(
            f"SPEAKER {clip_id} 1 {a:.3f} {b - a:.3f} "
            f"<NA> <NA> {normalise_speaker(spk)} <NA> <NA>"
        )
    return lines


# --------------------------------------------------------------------------

def load_manifest(path: Path) -> dict[str, dict]:
    if not path or not path.exists():
        return {}
    recs = {}
    with path.open("r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            recs[r["clip_id"]] = r
    return recs


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(path)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)


def clip_id_for(video_id: str, start: float, end: float) -> str:
    """Must match Stage 1 exactly, or the audio and references will not join."""
    return f"{video_id}__{int(round(start * 1000)):09d}_{int(round(end * 1000)):09d}"


def main() -> int:
    ap = argparse.ArgumentParser(description="Stage 2: parse ground truth into RTTM + aligned segments.")
    ap.add_argument("--input", "--csv", dest="input", required=True, type=Path)
    ap.add_argument("--out", required=True, type=Path, help="root containing ref/ (usually Stage 1's --out)")
    ap.add_argument("--manifest", type=Path, default=None,
                    help="Stage 1 manifest.jsonl, to mark which clips have audio")
    args = ap.parse_args()

    df = read_table(args.input)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise SystemExit(f"missing columns: {missing}")

    ref_root = args.out / "ref"
    if ref_root.exists():
        shutil.rmtree(ref_root)          # idempotent: no half-state from a killed run
    (ref_root / "rttm").mkdir(parents=True)
    (ref_root / "segments").mkdir(parents=True)

    manifest = load_manifest(args.manifest) if args.manifest else {}
    anom = Anomalies()
    all_rttm: list[str] = []
    meta_rows: list[dict] = []
    pipe_violations = 0

    for _, row in df.iterrows():
        video_id = str(row.video_id)
        start, end = float(row.start_sec), float(row.end_sec)
        duration = end - start
        clip_id = clip_id_for(video_id, start, end)

        diar = parse_diarization(row.diarization_segments, clip_id)
        asr = parse_asr(row.asr_segments, clip_id)

        # The index-join is only safe if the two columns really do correspond.
        # Assert it; a mismatch means the file changed shape under us.
        if len(diar) != len(asr):
            raise SystemExit(
                f"{clip_id}: diarization has {len(diar)} turns but asr has {len(asr)}. "
                "The index-join assumption is broken -- stop and re-audit the input."
            )
        for i, ((_, da, db), (aa, ab, _)) in enumerate(zip(diar, asr)):
            if abs(da - aa) > TIME_TOL or abs(db - ab) > TIME_TOL:
                raise SystemExit(
                    f"{clip_id}: turn {i} timestamps disagree between columns: "
                    f"diar [{da}-{db}] vs asr [{aa}-{ab}]."
                )

        turns = [(spk, a, b, text) for (spk, a, b), (_, _, text) in zip(diar, asr)]
        for _, _, _, text in turns:
            if "|" in text:
                pipe_violations += 1
            if not text.strip():
                anom.empty_text += 1

        turns = clip_turns(turns, duration, clip_id, anom)
        if not turns:
            anom.empty_clips.append(clip_id)

        # --- artifacts -----------------------------------------------------
        lines = rttm_lines(clip_id, turns)
        (ref_root / "rttm" / f"{clip_id}.rttm").write_text(
            "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        all_rttm.extend(lines)

        seg_records = [
            {"index": i, "speaker": normalise_speaker(spk),
             "start": round(a, 3), "end": round(b, 3), "text": text}
            for i, (spk, a, b, text) in enumerate(turns)
        ]
        (ref_root / "segments" / f"{clip_id}.json").write_text(
            json.dumps({"clip_id": clip_id, "video_id": video_id,
                        "start_sec": start, "end_sec": end, "duration": duration,
                        "segments": seg_records}, ensure_ascii=False, indent=1),
            encoding="utf-8")

        # --- per-clip conditions for Stage 6 -------------------------------
        speakers = sorted({normalise_speaker(s) for s, _, _, _ in turns})
        overlap = speaker_aware_overlap(turns)
        speech = union_speech(turns)
        all_text = " ".join(t for _, _, _, t in turns)
        n_words = len(re.findall(r"\S+", re.sub(r"\([^)]*\)", "", all_text)))
        rec = manifest.get(clip_id)

        meta_rows.append({
            "clip_id": clip_id,
            "video_id": video_id,
            "start_sec": start,
            "end_sec": end,
            "duration": round(duration, 3),
            "n_speakers": len(speakers),
            "n_segments": len(turns),
            "speech_sec": round(speech, 3),
            "speech_frac": round(speech / duration, 4) if duration else 0.0,
            "overlap_sec": round(overlap, 3),
            "overlap_frac_of_speech": round(overlap / speech, 4) if speech else 0.0,
            "overlap_frac_of_clip": round(overlap / duration, 4) if duration else 0.0,
            "language": detect_language(all_text),
            "n_ref_words": n_words,
            "has_audio": bool(rec and rec.get("status") in ("ok", "short_source")),
        })

    (ref_root / "all.rttm").write_text("\n".join(all_rttm) + "\n", encoding="utf-8")

    meta = pd.DataFrame(meta_rows)
    meta.to_csv(ref_root / "clip_meta.csv", index=False, encoding="utf-8")

    report = {
        "n_clips": len(meta),
        "n_turns_kept": int(meta.n_segments.sum()),
        "inverted_dropped": len(anom.inverted),
        "dropped_outside_window": len(anom.dropped_outside),
        "dropped_tiny_after_clip": len(anom.dropped_tiny),
        "clipped_at_window_end": len(anom.clipped_tail),
        "total_sec_lost_to_clipping": round(sum(a["lost_sec"] for a in anom.clipped_tail), 3),
        "empty_text_turns": anom.empty_text,
        "clips_with_no_turns": anom.empty_clips,
        "pipe_in_text_violations": pipe_violations,
        "details": {
            "inverted": anom.inverted,
            "dropped_outside": anom.dropped_outside[:50],
            "dropped_tiny": anom.dropped_tiny[:50],
        },
    }
    (ref_root / "stage2_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    # --- summary -----------------------------------------------------------
    print("=" * 70)
    print("STAGE 2 SUMMARY")
    print("=" * 70)
    print(f"  clips parsed          : {len(meta)}")
    print(f"  reference turns kept  : {int(meta.n_segments.sum())}")
    print(f"  turns clipped at end  : {len(anom.clipped_tail)} "
          f"({report['total_sec_lost_to_clipping']:.1f}s lost total)")
    print(f"  turns dropped         : {len(anom.inverted)} inverted, "
          f"{len(anom.dropped_outside)} outside window, "
          f"{len(anom.dropped_tiny)} sub-{MIN_SEG_DUR}s")
    if pipe_violations:
        print(f"  [!] '|' found inside {pipe_violations} transcript(s) -- split may be wrong")
    if anom.empty_clips:
        print(f"  [!] clips left with NO turns: {anom.empty_clips}")

    print(f"\n  total speech (union)  : {meta.speech_sec.sum() / 3600:.2f}h "
          f"of {meta.duration.sum() / 3600:.2f}h audio "
          f"({meta.speech_sec.sum() / meta.duration.sum() * 100:.1f}%)")
    print(f"  OVERLAP (speaker-aware): {meta.overlap_sec.sum() / 60:.1f} min = "
          f"{meta.overlap_sec.sum() / meta.speech_sec.sum() * 100:.2f}% of speech")
    print(f"  clips containing overlap: {int((meta.overlap_sec > 0).sum())}/{len(meta)}")

    print("\n  speaker-count distribution:")
    for k, v in sorted(meta.n_speakers.value_counts().items()):
        print(f"    {k} speakers : {v:>3} clips")
    print("\n  language distribution:")
    for k, v in meta.language.value_counts().items():
        print(f"    {k:<22}: {v:>3} clips")
    if "has_audio" in meta:
        print(f"\n  clips with audio ready : {int(meta.has_audio.sum())}/{len(meta)}")
    print(f"\n  wrote -> {ref_root}")
    print("=" * 70)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
run("stage 2: parse references",
    ["stage2_parse_refs.py", "--input", DRIVE / "youtube_segments_final.xlsx",
     "--out", "data", "--manifest", "data/manifest.jsonl"])
run("stage 2: verify references", ["nb_checks.py", "refs", "--data", "data"])

## Stage 3 — baseline diarization

- **pyannote 3.1** (`pyannote/speaker-diarization-3.1`, gated: needs `HF_TOKEN`).
- **Sortformer 4spk-v1**, offline and streaming. Offline attention is O(T²) and ran
  out of memory on the 25 longest clips on a T4; those count as total miss, so its row
  measures memory, not the model. The streaming mode (speaker cache + FIFO) fits.

**Metric policy:** `collar = 0`, **overlap scored**, UEM = the whole clip. Published
numbers usually use a 0.25 s collar and skip overlap; with that policy pyannote would
report 20.58 here instead of 27.34.

In [ ]:
%%writefile stage3_diarize.py
#!/usr/bin/env python3
"""
Stage 3a -- Baseline diarization inference.

Runs one diarization system over the extracted WAVs and writes hypothesis RTTMs.
Inference only; scoring lives in stage3_score.py so that a metric bug never costs
another GPU run.

    data/hyp/<system>/rttm/<clip_id>.rttm
    data/hyp/<system>/manifest.jsonl

Systems:
    pyannote31    pyannote/speaker-diarization-3.1   (gated; needs HF token)
    community1    pyannote/speaker-diarization-community-1  (pyannote.audio 4.x)
    sortformer    nvidia/diar_sortformer_4spk-v1     (NeMo; HARD CAP 4 speakers)

Resumability: a clip is skipped when its RTTM exists and the manifest records it
ok. Killed sessions resume at the next clip.

Usage:
    python stage3_diarize.py --system pyannote31 --data data --hf-token $HF_TOKEN
    python stage3_diarize.py --system sortformer --data data --limit 5
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import threading
import time
import wave
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path

# --------------------------------------------------------------------------

SUPPORTED = ("pyannote31", "community1", "sortformer", "sortformer_stream")

MODEL_IDS = {
    "pyannote31": "pyannote/speaker-diarization-3.1",
    "community1": "pyannote/speaker-diarization-community-1",
    "sortformer": "nvidia/diar_sortformer_4spk-v1",
    # Same checkpoint as `sortformer`, run through NeMo's streaming path. Kept as
    # a separate system so the offline and streaming hypotheses can be scored
    # side by side on the clips where both succeeded.
    "sortformer_stream": "nvidia/diar_sortformer_4spk-v1",
}

# Sortformer is architecturally limited to 4 speakers. Recorded so the Stage 6
# breakdown can separate "model got it wrong" from "model could not represent it".
SPEAKER_CAP = {"sortformer": 4, "sortformer_stream": 4}

# Sortformer attends over the whole session, so peak VRAM grows as O(duration^2):
# a 913 s clip asked for 7.8 GiB and an 1822 s clip for 30.9 GiB on a 14.6 GiB T4.
# Streaming mode bounds that by processing fixed-length chunks and carrying speaker
# identity forward in a speaker cache + FIFO queue, so labels stay consistent
# across chunk boundaries without any stitching on our side.
STREAMING_ATTRS = ("chunk_len", "chunk_left_context", "chunk_right_context",
                   "fifo_len", "spkcache_len", "spkcache_update_period")


@dataclass
class HypRecord:
    clip_id: str
    system: str
    status: str                       # ok | failed
    n_speakers_pred: int | None = None
    n_segments: int | None = None
    speech_sec: float | None = None
    runtime_sec: float | None = None
    rtf: float | None = None          # runtime / audio duration
    error: str | None = None
    ts: str = ""

    def __post_init__(self):
        if not self.ts:
            self.ts = datetime.now(timezone.utc).isoformat(timespec="seconds")


class Manifest:
    def __init__(self, path: Path):
        self.path = path
        self._lock = threading.Lock()
        self.records: dict[str, dict] = {}
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                self.records[r["clip_id"]] = r

    def append(self, rec: HypRecord) -> None:
        with self._lock:
            with self.path.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(asdict(rec), ensure_ascii=False) + "\n")
                fh.flush()
                os.fsync(fh.fileno())
            self.records[rec.clip_id] = asdict(rec)

    def done(self, clip_id: str, rttm_dir: Path) -> bool:
        r = self.records.get(clip_id)
        if not r or r["status"] != "ok":
            return False
        return (rttm_dir / f"{clip_id}.rttm").exists()


# --------------------------------------------------------------------------

def wav_duration(path: Path) -> float:
    with wave.open(str(path), "rb") as wf:
        return wf.getnframes() / wf.getframerate()


def write_rttm(path: Path, clip_id: str, turns: list[tuple[float, float, str]]) -> None:
    """turns = [(start, end, speaker)]. Written atomically."""
    lines = [
        f"SPEAKER {clip_id} 1 {s:.3f} {e - s:.3f} <NA> <NA> {str(spk).replace(' ', '_')} <NA> <NA>"
        for s, e, spk in turns if e > s
    ]
    tmp = path.with_suffix(".tmp")
    tmp.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    os.replace(tmp, path)


# --------------------------------------------------------------------------
# Backends
# --------------------------------------------------------------------------

class PyannoteBackend:
    """pyannote 3.1 / community-1. Same API surface, different checkpoint."""

    def __init__(self, system: str, hf_token: str | None, device: str):
        import pyannote.audio
        from pyannote.audio import Pipeline
        import torch

        model_id = MODEL_IDS[system]
        print(f"[env ] pyannote.audio {pyannote.audio.__version__}")

        # The auth kwarg was renamed in pyannote.audio 4.0:
        #   3.x -> use_auth_token=...     4.x -> token=...
        # Try in that order rather than pinning a version, so the same script
        # runs on whichever the environment happens to give us.
        last_err = None
        self.pipeline = None
        attempts = ([{"token": hf_token}, {"use_auth_token": hf_token}]
                    if hf_token else [{}])
        for kwargs in attempts:
            try:
                self.pipeline = Pipeline.from_pretrained(model_id, **kwargs)
                break
            except TypeError as exc:
                last_err = exc                       # wrong kwarg name; try the other
                continue
        if self.pipeline is None and last_err is not None:
            raise SystemExit(
                f"could not call Pipeline.from_pretrained for {model_id}: {last_err}\n"
                "Neither token= nor use_auth_token= was accepted -- check the "
                "pyannote.audio version printed above."
            )
        if self.pipeline is None:
            raise SystemExit(
                f"Pipeline.from_pretrained returned None for {model_id}.\n"
                "That almost always means the HF token is missing/invalid, or you have not\n"
                "accepted the user conditions. For 3.1 you must accept BOTH:\n"
                "  https://hf.co/pyannote/speaker-diarization-3.1\n"
                "  https://hf.co/pyannote/segmentation-3.0"
            )
        self.pipeline.to(torch.device(device))

    def __call__(self, wav: Path, clip_id: str):
        out = self.pipeline(str(wav))

        # 3.x returns an Annotation directly. 4.x returns a DiarizeOutput with
        # BOTH .speaker_diarization (overlaps preserved) and
        # .exclusive_speaker_diarization (overlaps stripped for transcription).
        # We must take the former: we score with skip_overlap=False, so the
        # exclusive variant would silently flatten every overlap region and show
        # up as a large, entirely artificial miss rate.
        if hasattr(out, "itertracks"):
            ann = out
        elif hasattr(out, "speaker_diarization"):
            ann = out.speaker_diarization
        else:
            attrs = [a for a in dir(out) if not a.startswith("_")]
            raise RuntimeError(
                f"{clip_id}: cannot get an Annotation from {type(out).__name__}; "
                f"available attributes: {attrs}"
            )
        return [(seg.start, seg.end, label) for seg, _, label in ann.itertracks(yield_label=True)]


class SortformerBackend:
    """NeMo Sortformer.

    NOTE: NeMo's diarize() return shape has changed across releases. This handles
    the shapes seen in the wild and fails loudly with the actual repr if it meets
    something new -- better than silently writing an empty RTTM.
    """

    def __init__(self, system: str, hf_token: str | None, device: str,
                 streaming: bool = False, overrides: dict | None = None):
        from nemo.collections.asr.models import SortformerEncLabelModel
        import torch

        self.torch = torch
        self.model = SortformerEncLabelModel.from_pretrained(MODEL_IDS[system])
        self.model.eval()
        self.model.to(torch.device(device))

        sm = self.model.sortformer_modules
        if streaming:
            # NeMo exposes streaming as a model flag, not a diarize() argument:
            # _diarize_forward dispatches on it. The chunk parameters ship with the
            # checkpoint; we only override what was asked for on the command line.
            self.model.streaming_mode = True
            for key, val in (overrides or {}).items():
                if val is not None:
                    setattr(sm, key, val)

        # One output frame = window_stride * subsampling_factor seconds, so the
        # chunk parameters are reported in seconds too -- frames are meaningless
        # to read in a log.
        stride = float(self.model.cfg.get("preprocessor", {}).get("window_stride", 0.01))
        sub = float(getattr(sm, "subsampling_factor", 8) or 8)
        frame_sec = stride * sub
        print(f"[env ] streaming_mode={getattr(self.model, 'streaming_mode', None)}"
              f"  frame={frame_sec:.3f}s")
        for key in STREAMING_ATTRS:
            val = getattr(sm, key, None)
            if val is None:
                continue
            secs = f"  ({val * frame_sec:.1f}s)" if isinstance(val, (int, float)) else ""
            print(f"[env ]   {key} = {val}{secs}")

    @staticmethod
    def _parse(pred, clip_id: str):
        # Unwrap a per-file list wrapper.
        if isinstance(pred, list) and len(pred) == 1 and isinstance(pred[0], list):
            pred = pred[0]
        turns = []
        for item in pred:
            if isinstance(item, str):
                # "start end speaker_N" or an RTTM-ish line
                parts = item.split()
                if len(parts) >= 3 and parts[0] == "SPEAKER":
                    turns.append((float(parts[3]), float(parts[3]) + float(parts[4]), parts[7]))
                elif len(parts) >= 3:
                    turns.append((float(parts[0]), float(parts[1]), parts[2]))
                else:
                    raise ValueError(f"{clip_id}: unrecognised sortformer string: {item!r}")
            elif isinstance(item, (list, tuple)) and len(item) >= 3:
                turns.append((float(item[0]), float(item[1]), str(item[2])))
            elif isinstance(item, dict):
                s = item.get("start", item.get("begin"))
                e = item.get("end", item.get("stop"))
                spk = item.get("speaker", item.get("label"))
                if s is None or e is None or spk is None:
                    raise ValueError(f"{clip_id}: unrecognised sortformer dict: {item!r}")
                turns.append((float(s), float(e), str(spk)))
            else:
                raise ValueError(f"{clip_id}: unrecognised sortformer item: {type(item)} {item!r}")
        return turns

    def __call__(self, wav: Path, clip_id: str):
        pred = self.model.diarize(audio=[str(wav)], batch_size=1)
        return self._parse(pred, clip_id)


def build_backend(system: str, hf_token: str | None, device: str,
                  overrides: dict | None = None):
    if system in ("pyannote31", "community1"):
        return PyannoteBackend(system, hf_token, device)
    if system in ("sortformer", "sortformer_stream"):
        return SortformerBackend(system, hf_token, device,
                                 streaming=(system == "sortformer_stream"),
                                 overrides=overrides)
    raise SystemExit(f"unknown system {system!r}; expected one of {SUPPORTED}")


# --------------------------------------------------------------------------

def main() -> int:
    ap = argparse.ArgumentParser(description="Stage 3a: run one diarization system over the clips.")
    ap.add_argument("--system", required=True, choices=SUPPORTED)
    ap.add_argument("--data", required=True, type=Path,
                    help="writable output root (holds ref/, hyp/, results/)")
    ap.add_argument("--wav-dir", type=Path, default=None,
                    help="where the WAVs live (default: <data>/wav). Point this at a "
                         "read-only Kaggle input mount so Save Version does not "
                         "re-snapshot 1.4 GB of audio on every commit.")
    ap.add_argument("--hf-token", default=os.environ.get("HF_TOKEN", ""),
                    help="HuggingFace token (or set HF_TOKEN)")
    ap.add_argument("--device", default=None, help="cuda / cpu (default: auto)")
    ap.add_argument("--limit", type=int, default=None, help="first N clips only")
    ap.add_argument("--max-duration", type=float, default=None,
                    help="skip clips longer than this many seconds (OOM guard)")
    for key in STREAMING_ATTRS:
        ap.add_argument(f"--{key.replace('_', '-')}", type=int, default=None,
                        help=f"sortformer_stream: override {key} "
                             "(frames; default = the checkpoint's own value)")
    args = ap.parse_args()
    overrides = {key: getattr(args, key) for key in STREAMING_ATTRS}

    import torch
    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")

    wav_dir = args.wav_dir or (args.data / "wav")
    out_dir = args.data / "hyp" / args.system
    rttm_dir = out_dir / "rttm"
    rttm_dir.mkdir(parents=True, exist_ok=True)

    wavs = sorted(wav_dir.glob("*.wav"))
    if not wavs:
        raise SystemExit(f"no WAVs in {wav_dir} -- run Stage 1 first, "
                         "or pass --wav-dir if the audio lives elsewhere")
    if args.limit:
        wavs = wavs[:args.limit]

    manifest = Manifest(out_dir / "manifest.jsonl")
    todo = [w for w in wavs if not manifest.done(w.stem, rttm_dir)]

    print(f"[env ] system={args.system}  model={MODEL_IDS[args.system]}")
    print(f"[env ] wav_dir={wav_dir}")
    print(f"[env ] device={device}"
          + (f"  gpu={torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
    if args.system in SPEAKER_CAP:
        print(f"[warn] {args.system} is capped at {SPEAKER_CAP[args.system]} speakers; "
              "clips above that cannot be solved and should be reported separately")
    print(f"[run ] {len(todo)} of {len(wavs)} clips to do "
          f"({len(wavs) - len(todo)} already done)\n")
    if not todo:
        print("nothing to do.")
        return 0

    backend = build_backend(args.system, args.hf_token or None, device, overrides)
    t_start = time.time()
    total_audio = 0.0

    for i, wav in enumerate(todo, 1):
        clip_id = wav.stem
        dur = wav_duration(wav)

        if args.max_duration and dur > args.max_duration:
            rec = HypRecord(clip_id, args.system, "failed",
                            error=f"skipped: {dur:.0f}s exceeds --max-duration")
            manifest.append(rec)
            print(f"[skip] {clip_id}  {dur:.0f}s > {args.max_duration:.0f}s")
            continue

        t0 = time.time()
        try:
            turns = backend(wav, clip_id)
            elapsed = time.time() - t0
            write_rttm(rttm_dir / f"{clip_id}.rttm", clip_id, turns)
            speakers = {spk for _, _, spk in turns}
            speech = sum(e - s for s, e, _ in turns if e > s)
            rec = HypRecord(clip_id, args.system, "ok",
                            n_speakers_pred=len(speakers), n_segments=len(turns),
                            speech_sec=round(speech, 3), runtime_sec=round(elapsed, 2),
                            rtf=round(elapsed / dur, 4) if dur else None)
            total_audio += dur
            print(f"[ ok ] {clip_id[:44]:<44} {dur:6.0f}s  "
                  f"{len(speakers)}spk {len(turns):4d}seg  {elapsed:6.1f}s "
                  f"(rtf {elapsed / dur:.3f})")
        except Exception as exc:
            # Never let one bad clip end the run -- record and continue.
            rec = HypRecord(clip_id, args.system, "failed",
                            runtime_sec=round(time.time() - t0, 2),
                            error=f"{type(exc).__name__}: {exc}"[:600])
            print(f"[FAIL] {clip_id[:44]:<44} {type(exc).__name__}: {str(exc)[:90]}",
                  file=sys.stderr)
        manifest.append(rec)

        # Several Sortformer failures asked for only ~1.7 GiB on a 14.6 GiB card:
        # that is fragmentation, not model size. Release between clips so one long
        # clip does not poison the ones after it.
        if device == "cuda":
            torch.cuda.empty_cache()

        if i % 10 == 0 or i == len(todo):
            el = time.time() - t_start
            print(f"       ---- {i}/{len(todo)} clips, {el / 60:.1f} min elapsed, "
                  f"{(len(todo) - i) * el / i / 60:.1f} min left (est)")

    ok = [r for r in manifest.records.values() if r["status"] == "ok"]
    fail = [r for r in manifest.records.values() if r["status"] != "ok"]
    print("\n" + "=" * 70)
    print(f"STAGE 3a SUMMARY -- {args.system}")
    print("=" * 70)
    print(f"  ok / failed        : {len(ok)} / {len(fail)}")
    if ok:
        rtfs = [r["rtf"] for r in ok if r.get("rtf")]
        print(f"  mean RTF           : {sum(rtfs) / len(rtfs):.4f} "
              f"({1 / (sum(rtfs) / len(rtfs)):.0f}x realtime)")
        preds = [r["n_speakers_pred"] for r in ok]
        from collections import Counter
        print(f"  predicted speakers : {sorted(Counter(preds).items())}")
    for r in fail[:10]:
        print(f"  [FAIL] {r['clip_id']}: {(r.get('error') or '')[:100]}")
    print(f"  wrote -> {rttm_dir}")
    print("=" * 70)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile stage3_score.py
#!/usr/bin/env python3
"""
Stage 3b -- Diarization scoring.

Scores hypothesis RTTMs against the Stage 2 references and writes per-clip and
per-system results. CPU-only and instant, so metric bugs cost seconds rather
than another GPU run.

    data/results/diarization_per_clip.csv
    data/results/diarization_summary.csv
    data/results/diarization_summary.md

Metric policy (deliberately unforgiving):
  * collar = 0.0        -- no boundary forgiveness
  * skip_overlap = False -- overlapping speech IS scored
  * an explicit UEM of [0, clip_duration] per clip, so the scoring region is the
    whole clip rather than the extent of reference-union-hypothesis. Without it,
    false alarms in leading/trailing silence are counted inconsistently.

Corpus numbers are DURATION-WEIGHTED (sum of errors / sum of reference speech),
not the mean of per-clip rates. Both are reported, because the unweighted mean
is the one people publish by accident: it lets a 50s clip outweigh a 30min one.

Usage:
    python stage3_score.py --data data --systems pyannote31 sortformer
"""

from __future__ import annotations

import argparse
import sys
from collections import Counter
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------

def load_rttm(path: Path):
    """Parse an RTTM file into a pyannote Annotation.

    Hand-rolled rather than using pyannote.database.util.load_rttm, which has
    moved between releases; the format is ten whitespace-separated fields.
    """
    from pyannote.core import Annotation, Segment

    ann = Annotation(uri=path.stem)
    if not path.exists():
        return ann
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        p = line.split()
        if len(p) < 8 or p[0] != "SPEAKER":
            continue
        start, dur, spk = float(p[3]), float(p[4]), p[7]
        if dur <= 0:
            continue
        ann[Segment(start, start + dur)] = spk
    return ann


def subtract_timeline(a, b):
    """a minus b, as a new Timeline. pyannote.core's extrude() has moved across
    releases, so this is done explicitly with interval arithmetic."""
    from pyannote.core import Segment, Timeline

    cuts = []
    for seg in a:
        pieces = [(seg.start, seg.end)]
        for rem in b:
            nxt = []
            for lo, hi in pieces:
                if rem.end <= lo or rem.start >= hi:
                    nxt.append((lo, hi))
                    continue
                if rem.start > lo:
                    nxt.append((lo, rem.start))
                if rem.end < hi:
                    nxt.append((rem.end, hi))
            pieces = nxt
        cuts.extend(pieces)
    return Timeline([Segment(lo, hi) for lo, hi in cuts if hi - lo > 1e-6])


def load_pairs(system: str, data: Path, meta: pd.DataFrame) -> list[tuple]:
    """Load (meta_row, reference, hypothesis) once, for reuse across configs.

    The diagnostic runs six scoring passes; re-parsing ~200 RTTM files each time
    dominated the runtime.
    """
    ref_dir = data / "ref" / "rttm"
    hyp_dir = data / "hyp" / system / "rttm"
    pairs = []
    for _, m in meta.iterrows():
        if not m.has_audio:
            continue
        ref = load_rttm(ref_dir / f"{m.clip_id}.rttm")
        if not ref:
            continue
        pairs.append((m, ref, load_rttm(hyp_dir / f"{m.clip_id}.rttm")))
    return pairs


def score_pass(pairs: list[tuple], collar: float, skip_overlap: bool,
               region: str = "full") -> dict:
    """One scoring configuration over the whole corpus.

    region: 'full'    -- the whole clip
            'overlap' -- only where >=2 reference speakers are active
            'single'  -- reference speech with the overlap regions removed

    NOTE on the region split: DER applies an optimal speaker mapping, and that
    mapping is recomputed per scoring region. So the overlap/single numbers are
    a decomposition of *difficulty*, not a strictly additive decomposition of
    the full-clip error. Reported as such.
    """
    from pyannote.core import Segment, Timeline
    from pyannote.metrics.diarization import DiarizationErrorRate

    der = DiarizationErrorRate(collar=collar, skip_overlap=skip_overlap)

    err = tot = 0.0
    n = 0
    for m, ref, hyp in pairs:
        if region == "full":
            uem = Timeline([Segment(0.0, float(m.duration))], uri=m.clip_id)
        elif region == "overlap":
            uem = ref.get_overlap()
        elif region == "single":
            uem = subtract_timeline(ref.get_timeline().support(), ref.get_overlap())
        else:
            raise ValueError(region)

        if not list(uem):
            continue
        d = der(ref, hyp, uem=uem, detailed=True)
        if d["total"] <= 0:
            continue
        err += d["missed detection"] + d["false alarm"] + d["confusion"]
        tot += d["total"]
        n += 1

    return {"der": err / tot if tot else float("nan"),
            "err_sec": err, "ref_sec": tot, "n_clips": n}


def run_diagnostics(system: str, data: Path, meta: pd.DataFrame) -> None:
    """Quantify how much of the error comes from the metric policy vs the audio."""
    configs = [
        ("strict (reported)",      0.00, False, "full"),
        ("overlap not scored",     0.00, True,  "full"),
        ("collar 0.25",            0.25, False, "full"),
        ("collar 0.25 + no overlap", 0.25, True, "full"),
    ]
    pairs = load_pairs(system, data, meta)
    print("\n" + "=" * 78)
    print(f"DIAGNOSTIC -- {system}: metric policy sensitivity  ({len(pairs)} clips)")
    print("=" * 78)
    print(f"  {'configuration':<26}{'DER':>9}{'err (s)':>12}{'scored ref (s)':>16}")
    print("  " + "-" * 62)
    base = None
    for label, collar, skip, region in configs:
        r = score_pass(pairs, collar, skip, region)
        if base is None:
            base = r["der"]
        print(f"  {label:<26}{r['der']*100:>8.2f}%{r['err_sec']:>12.0f}{r['ref_sec']:>16.0f}")
    print("  " + "-" * 62)
    print("  The last row is roughly the configuration most published DER numbers use.")
    print("  Denominators differ between rows, so these are not subtractable.")

    print(f"\n  Error concentration by reference region:")
    print("  " + "-" * 62)
    for label, region in (("overlapped speech", "overlap"), ("single-speaker speech", "single")):
        r = score_pass(pairs, 0.0, False, region)
        print(f"  {label:<26}{r['der']*100:>8.2f}%{r['err_sec']:>12.0f}"
              f"{r['ref_sec']:>16.0f}  (n={r['n_clips']})")
    print("  " + "-" * 62)
    print("  Speaker mapping is recomputed per region, so these show where the")
    print("  difficulty concentrates -- they do not sum to the full-clip DER.")
    print("=" * 78)


def score_system(system: str, data: Path, meta: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    from pyannote.core import Segment, Timeline
    from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate

    ref_dir = data / "ref" / "rttm"
    hyp_dir = data / "hyp" / system / "rttm"
    if not hyp_dir.exists():
        raise SystemExit(f"no hypotheses for {system!r} at {hyp_dir} -- run stage3_diarize.py first")

    der = DiarizationErrorRate(collar=0.0, skip_overlap=False)
    jer = JaccardErrorRate(collar=0.0, skip_overlap=False)

    rows = []
    for _, m in meta.iterrows():
        clip_id = m.clip_id
        if not m.has_audio:
            continue                      # no audio -> no hypothesis is possible
        ref = load_rttm(ref_dir / f"{clip_id}.rttm")
        if not ref:
            continue

        hyp_path = hyp_dir / f"{clip_id}.rttm"
        # A missing/empty hypothesis is a legitimate total miss, NOT a skip.
        # Dropping it would flatter the system by removing its hardest clips.
        hyp = load_rttm(hyp_path)

        # Score the whole clip, explicitly.
        uem = Timeline([Segment(0.0, float(m.duration))], uri=clip_id)

        d = der(ref, hyp, uem=uem, detailed=True)
        j = jer(ref, hyp, uem=uem)

        n_ref_spk = len(ref.labels())
        n_hyp_spk = len(hyp.labels())
        total = d["total"]
        rows.append({
            "clip_id": clip_id,
            "system": system,
            "hyp_missing": not hyp_path.exists(),
            "der": d[der.name] if der.name in d else (d["missed detection"] + d["false alarm"]
                                                     + d["confusion"]) / total if total else 0.0,
            "miss": d["missed detection"] / total if total else 0.0,
            "false_alarm": d["false alarm"] / total if total else 0.0,
            "confusion": d["confusion"] / total if total else 0.0,
            "jer": float(j),
            "ref_speech_sec": total,
            "err_miss_sec": d["missed detection"],
            "err_fa_sec": d["false alarm"],
            "err_conf_sec": d["confusion"],
            "n_ref_speakers": n_ref_spk,
            "n_hyp_speakers": n_hyp_spk,
            "spk_count_correct": int(n_ref_spk == n_hyp_spk),
            "spk_count_err": n_hyp_spk - n_ref_spk,
            "duration": float(m.duration),
            "overlap_frac_of_speech": float(m.overlap_frac_of_speech),
            "language": m.language,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise SystemExit(f"{system}: nothing scored")

    tot = df.ref_speech_sec.sum()
    summary = {
        "system": system,
        "n_clips": len(df),
        "n_hyp_missing": int(df.hyp_missing.sum()),
        # Duration-weighted -- the number to quote.
        "DER": (df.err_miss_sec.sum() + df.err_fa_sec.sum() + df.err_conf_sec.sum()) / tot,
        "miss": df.err_miss_sec.sum() / tot,
        "false_alarm": df.err_fa_sec.sum() / tot,
        "confusion": df.err_conf_sec.sum() / tot,
        # pyannote's own accumulated values, as a cross-check on the arithmetic.
        "DER_pyannote_accum": abs(der),
        "JER_pyannote_accum": abs(jer),
        # Unweighted, for contrast only.
        "DER_macro": df.der.mean(),
        "JER_macro": df.jer.mean(),
        "spk_count_acc": df.spk_count_correct.mean(),
        "spk_count_mae": df.spk_count_err.abs().mean(),
        "spk_count_bias": df.spk_count_err.mean(),
    }
    return df, summary


# --------------------------------------------------------------------------

def main() -> int:
    ap = argparse.ArgumentParser(description="Stage 3b: score diarization hypotheses.")
    ap.add_argument("--data", required=True, type=Path)
    ap.add_argument("--systems", nargs="+", required=True)
    ap.add_argument("--diagnostic", action="store_true",
                    help="also report metric-policy sensitivity and where error concentrates")
    args = ap.parse_args()

    meta = pd.read_csv(args.data / "ref" / "clip_meta.csv")
    out_dir = args.data / "results"
    out_dir.mkdir(parents=True, exist_ok=True)

    per_clip, summaries = [], []
    for system in args.systems:
        df, s = score_system(system, args.data, meta)
        per_clip.append(df)
        summaries.append(s)
        print(f"[ok] scored {system}: {len(df)} clips")

    per_clip_df = pd.concat(per_clip, ignore_index=True)
    per_clip_df.to_csv(out_dir / "diarization_per_clip.csv", index=False, encoding="utf-8")
    sm = pd.DataFrame(summaries)
    sm.to_csv(out_dir / "diarization_summary.csv", index=False, encoding="utf-8")

    # ---- report ----------------------------------------------------------
    print("\n" + "=" * 78)
    print("STAGE 3 -- BASELINE DIARIZATION   (collar=0.0, skip_overlap=False, UEM=full clip)")
    print("=" * 78)
    hdr = (f"{'system':<14}{'DER':>8}{'miss':>8}{'FA':>8}{'conf':>8}"
           f"{'JER':>8}{'spk acc':>9}{'spk MAE':>9}")
    print(hdr)
    print("-" * 78)
    for s in summaries:
        print(f"{s['system']:<14}{s['DER']*100:>7.2f}%{s['miss']*100:>7.2f}%"
              f"{s['false_alarm']*100:>7.2f}%{s['confusion']*100:>7.2f}%"
              f"{s['JER_pyannote_accum']*100:>7.2f}%{s['spk_count_acc']*100:>8.1f}%"
              f"{s['spk_count_mae']:>9.2f}")
    print("-" * 78)
    print("DER/miss/FA/conf are duration-weighted. Macro (per-clip mean) for contrast:")
    for s in summaries:
        print(f"  {s['system']:<14} DER_macro {s['DER_macro']*100:6.2f}%   "
              f"JER_macro {s['JER_macro']*100:6.2f}%   "
              f"(weighted DER {s['DER']*100:.2f}%)")
        if abs(s["DER"] - s["DER_pyannote_accum"]) > 1e-6:
            print(f"    [!] hand-computed DER {s['DER']*100:.4f}% != pyannote accumulated "
                  f"{s['DER_pyannote_accum']*100:.4f}% -- investigate")
        if s["n_hyp_missing"]:
            print(f"    [!] {s['n_hyp_missing']} clip(s) had NO hypothesis (scored as total miss)")

    # ---- per-condition breakdown (previews Stage 6) ----------------------
    print("\n" + "-" * 78)
    print("DER by reference speaker count (duration-weighted within each bucket):")
    print("-" * 78)
    piv = per_clip_df.copy()
    piv["err"] = piv.err_miss_sec + piv.err_fa_sec + piv.err_conf_sec
    g = (piv.groupby(["system", "n_ref_speakers"])
            .apply(lambda x: pd.Series({"DER": x.err.sum() / x.ref_speech_sec.sum(),
                                        "n": len(x)}), include_groups=False)
            .reset_index())
    for system in args.systems:
        sub = g[g.system == system]
        cells = "  ".join(f"{int(r.n_ref_speakers)}spk:{r.DER*100:5.1f}%(n={int(r.n)})"
                          for _, r in sub.iterrows())
        print(f"  {system:<14}{cells}")

    print("\nDER by overlap tercile:")
    print("-" * 78)
    piv["ov_bucket"] = pd.qcut(piv.overlap_frac_of_speech, 3,
                               labels=["low", "mid", "high"], duplicates="drop")
    g2 = (piv.groupby(["system", "ov_bucket"], observed=True)
             .apply(lambda x: pd.Series({"DER": x.err.sum() / x.ref_speech_sec.sum(),
                                         "n": len(x)}), include_groups=False)
             .reset_index())
    for system in args.systems:
        sub = g2[g2.system == system]
        cells = "  ".join(f"{r.ov_bucket}:{r.DER*100:5.1f}%(n={int(r.n)})" for _, r in sub.iterrows())
        print(f"  {system:<14}{cells}")

    # ---- markdown for the writeup ---------------------------------------
    md = ["# Baseline diarization", "",
          "`collar=0.0`, `skip_overlap=False`, UEM = full clip. "
          "DER components are duration-weighted.", "",
          "| System | DER | Miss | FA | Conf | JER | Spk acc | Spk MAE |",
          "|---|---|---|---|---|---|---|---|"]
    for s in summaries:
        md.append(f"| {s['system']} | {s['DER']*100:.2f}% | {s['miss']*100:.2f}% | "
                  f"{s['false_alarm']*100:.2f}% | {s['confusion']*100:.2f}% | "
                  f"{s['JER_pyannote_accum']*100:.2f}% | {s['spk_count_acc']*100:.1f}% | "
                  f"{s['spk_count_mae']:.2f} |")
    (out_dir / "diarization_summary.md").write_text("\n".join(md) + "\n", encoding="utf-8")

    if args.diagnostic:
        for system in args.systems:
            run_diagnostics(system, args.data, meta)

    print(f"\nwrote -> {out_dir}")
    print("=" * 78)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
DIAR_SYSTEMS = ("pyannote31", "sortformer", "sortformer_stream")
envs = {}
if HAVE_GPU:
    envs["pyannote31"] = gpu_env("pyannote", "pyannote.audio")
    envs["sortformer"] = envs["sortformer_stream"] = gpu_env("nemo", "nemo_toolkit[asr]")

for system in DIAR_SYSTEMS:
    label = f"stage 3: diarize {system}"
    if not HAVE_GPU:
        skip(label, "no GPU")
    elif envs[system] is None:
        skip(label, "environment install failed")
    elif system == "pyannote31" and not HF_TOKEN:
        skip(label, "no HF_TOKEN (pyannote/speaker-diarization-3.1 is gated)")
    else:
        run(label, ["stage3_diarize.py", "--system", system, "--data", GPU_ROOT,
                    "--wav-dir", AUDIO] + GPU_LIMIT, python=envs[system], may_fail=GPU_MAY_FAIL)

In [ ]:
if not FULL_GPU_RUN:
    if (WORKDIR / "smoke" / "hyp").is_dir():
        run("check: diarization smoke vs reference run",
            ["nb_checks.py", "diar-smoke", "--smoke", WORKDIR / "smoke", "--cache", CACHE,
             "--systems", *DIAR_SYSTEMS], may_fail=True)
    for system in DIAR_SYSTEMS:
        restore(f"hyp/{system}")

In [ ]:
# --diagnostic adds the metric-policy sensitivity table (collar, overlap) and the
# overlapped vs single-speaker error split quoted in the writeup.
run("stage 3: score diarizers",
    ["stage3_score.py", "--data", "data", "--diagnostic", "--systems", *DIAR_SYSTEMS])

## Stage 4a — ASR: transcribe the whole clip once, then assign words to speakers

ASR never sees a speaker label. Words with timestamps from the whole clip are
assigned to diarizer turns in Stage 4b, so every diarizer and every Stage 5
correction is scored on **identical words**, and a cpWER difference between them is
purely labelling. It also means 99 ASR calls instead of 12,809, and Whisper never has
to transcribe a sub-second fragment.

- **Whisper large-v3** via faster-whisper, `temperature=0.0`. The default temperature
  fallback samples **unseeded**: one clip gave 87, 84 and 100 words on three runs, and
  211 every time at temperature 0.
- **IndicConformer-600M** (AI4Bharat) via ONNX, CTC branch. Its CTC head is
  *multisoftmax*: one softmax per language, so logits from different languages are on
  incomparable scales. The decode picks the clip's language by frame vote and argmaxes
  within it (WER 93.66 → 78.82). `indicconformer_free`, the naive global argmax, is
  kept as a scored ablation.

**Separate environments are not optional here.** On Kaggle, installing `onnxruntime-gpu`
next to faster-whisper replaced the cuDNN that CTranslate2 needs, and Whisper fell
back to CPU with no error. Each runs in its own venv. `onnxruntime-gpu` is pinned to
1.20.2: newer releases are built for CUDA 13.

In [ ]:
%%writefile stage4_asr.py
#!/usr/bin/env python3
"""
Stage 4 -- ASR (GPU-bound). Audio in, words with timestamps out.

This stage sees audio ONLY. It knows nothing about speakers, and nothing about
the diarization hypotheses from Stage 3. Speaker attribution is a separate
CPU-only stage (stage4_attribute.py), which means one ASR run is reused across
every diarization system and every Stage 5 correction -- so a cpWER delta
between them is attributable to the labelling, never to the ASR having seen a
different slice of audio.

    python stage4_asr.py --system whisper        --data data --wav-dir WAV
    python stage4_asr.py --system indicconformer --data data --wav-dir WAV

Writes data/asr/<system>/words/<clip_id>.json, one record per clip, and appends
to data/asr/<system>/manifest.jsonl. Re-running skips clips already marked ok,
so a dead Kaggle session costs only the clip that was in flight.
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
import wave
from pathlib import Path

SUPPORTED = ("whisper", "indicconformer", "indicconformer_free")

WHISPER_MODEL = "large-v3"
INDIC_REPO = "ai4bharat/indic-conformer-600m-multilingual"

# A Conformer encoder attends over the whole input, so peak memory grows as
# O(T^2) -- the same trap that OOM'd offline Sortformer on the long clips in
# Stage 3. We chunk instead of discovering the limit at clip 74 of 99.
# CTC is a local, monotonic alignment, so chunks stitch by concatenation; there
# is no speaker identity to carry across boundaries the way Sortformer needed.
CHUNK_SEC = 30.0

OVERLAP_SEC = 2.0

SAMPLE_RATE = 16000

# Tokens per language block in the aggregate tokenizer (22 x 256 = 5632).
BLOCK_SIZE = 256

# SentencePiece word-boundary marker.
WORD_MARK = "▁"


# --------------------------------------------------------------------------
# manifest -- append-only, fsync'd, so a killed session leaves a readable file
# --------------------------------------------------------------------------

class Manifest:
    def __init__(self, path: Path):
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.records: dict[str, dict] = {}
        if self.path.exists():
            for line in self.path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue  # truncated final line from a hard kill
                self.records[rec["clip_id"]] = rec

    def done(self, clip_id: str) -> bool:
        # Only "ok" counts. A failed clip is retried on the next run rather than
        # silently treated as complete -- an empty hypothesis scores as a perfect
        # miss and looks like a model result instead of a crash.
        return self.records.get(clip_id, {}).get("status") == "ok"

    def append(self, rec: dict) -> None:
        self.records[rec["clip_id"]] = rec
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()
            os.fsync(fh.fileno())


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)  # atomic: a reader never sees a half-written file


def read_wav(path: Path):
    """16 kHz mono PCM -> float32 numpy in [-1, 1]. Stage 1 guarantees the format."""
    import numpy as np

    with wave.open(str(path), "rb") as wf:
        if wf.getframerate() != SAMPLE_RATE or wf.getnchannels() != 1:
            raise ValueError(f"{path.name}: expected 16k mono, got "
                             f"{wf.getframerate()} Hz / {wf.getnchannels()} ch")
        raw = wf.readframes(wf.getnframes())
    return np.frombuffer(raw, dtype="int16").astype("float32") / 32768.0


# --------------------------------------------------------------------------
# mel frontend
# --------------------------------------------------------------------------

class MelFrontend:
    """
    NeMo's AudioToMelSpectrogramPreprocessor, reimplemented explicitly.

    Used only when AI4Bharat's shipped TorchScript frontend refuses to run. That
    graph was serialized against a torch whose `torch.stft` accepted an implicit
    `return_complex`; torch >= 2.1 raises instead, and the call is baked into the
    serialized code where it cannot be patched.

    Constants are not guesses. n_fft/hop/win are read straight off the failing
    call in the traceback (512 / 160 / 400 = 32 ms FFT, 10 ms hop, 25 ms window at
    16 kHz), 80 mel bins is forced by the encoder's declared input shape, and the
    rest are NeMo's Conformer defaults: Slaney-normalised mel bank, pre-emphasis
    0.97, power spectrum, log with an additive 2^-24 guard, per-feature mean/var
    normalisation over time.

    A wrong frontend degrades WER without ever raising, so the check that this is
    right is the decode itself: correct features give fluent native script, and
    subtly wrong ones give word salad in the right script.
    """

    N_FFT, HOP, WIN, N_MELS = 512, 160, 400, 80
    PREEMPH = 0.97
    LOG_GUARD = 2.0 ** -24

    def __init__(self, device: str):
        import librosa
        import torch

        self.torch = torch
        self.device = torch.device(device)
        fb = librosa.filters.mel(sr=SAMPLE_RATE, n_fft=self.N_FFT,
                                 n_mels=self.N_MELS, fmin=0.0,
                                 fmax=SAMPLE_RATE / 2, norm="slaney", htk=False)
        self.fb = torch.from_numpy(fb).float().to(self.device)
        # periodic=False matches NeMo's FilterbankFeatures, not torch's default.
        self.window = torch.hann_window(self.WIN, periodic=False).to(self.device)

    def __call__(self, sig, length):
        torch = self.torch
        x = sig.to(self.device)
        x = torch.cat([x[:, :1], x[:, 1:] - self.PREEMPH * x[:, :-1]], dim=1)
        spec = torch.stft(x, n_fft=self.N_FFT, hop_length=self.HOP,
                          win_length=self.WIN, window=self.window,
                          center=True, pad_mode="reflect", return_complex=True)
        mel = torch.matmul(self.fb, spec.abs().pow(2.0))
        mel = torch.log(mel + self.LOG_GUARD)
        mean = mel.mean(dim=2, keepdim=True)
        std = mel.std(dim=2, keepdim=True).clamp_min(1e-5)
        mel = (mel - mean) / std
        n_frames = torch.div(length.to(self.device), self.HOP,
                             rounding_mode="floor") + 1
        return mel, n_frames.to(torch.int64)


# --------------------------------------------------------------------------
# backend: IndicConformer via ONNX, CTC branch
# --------------------------------------------------------------------------

class IndicConformerBackend:
    """
    AI4Bharat IndicConformer 600M, run through its exported ONNX graphs.

    Why ONNX rather than NeMo: the .nemo checkpoint declares
    `tokenizer.type: multilingual` (stock NeMo dispatches its aggregate
    tokenizer only on `agg`) and `multisoftmax: True` on both the RNNT and CTC
    decoders, which upstream RNNTDecoder/ConvASRDecoder do not accept. Loading
    it needs AI4Bharat's NeMo fork, which pins an older Python and torch than
    Kaggle provides, and which cannot coexist with the NeMo that Stage 3 needs
    for Sortformer. The ONNX export bakes those fork features into the graph, so
    it needs no fork at all.

    Why the CTC branch rather than RNNT: the RNNT joint ships one output head per
    language (joint_post_net_<lang>.onnx), so it would need a language decision
    per clip -- and the only cheap source of that decision is either another
    model or the reference transcript's script, the latter being ground truth
    leaking into the pipeline. The CTC head is a single 1024 -> 5632 projection
    over the whole aggregate vocabulary, so it is language-agnostic. Because the
    vocabulary is 22 per-language blocks concatenated in order, the argmax index
    also identifies the language as a byproduct. CTC gives frame-level alignment
    directly, which is what the word timestamps are built from.
    """

    name = "indicconformer"

    def __init__(self, device: str, lang_lock: bool = True):
        # lang_lock=False reproduces the unrestricted global argmax, which is
        # wrong for a multisoftmax head but is the ablation that demonstrates it.
        self.lang_lock = lang_lock
        import numpy as np
        import onnxruntime as ort
        import torch
        from huggingface_hub import snapshot_download

        self.np = np
        self.torch = torch
        self.device = device

        # snapshot, not hf_hub_download: encoder.onnx stores its weights as
        # external data resolved by relative path. Fetching the graph alone
        # loads without error and emits plausible garbage.
        local = Path(snapshot_download(INDIC_REPO,
                                       allow_patterns=["assets/*", "*.json"]))
        assets = local / "assets"

        providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                     if device == "cuda" else ["CPUExecutionProvider"])
        self.enc = ort.InferenceSession(str(assets / "encoder.onnx"),
                                        providers=providers)
        self.ctc = ort.InferenceSession(str(assets / "ctc_decoder.onnx"),
                                        providers=providers)
        print(f"[env ] onnxruntime providers: {self.enc.get_providers()}")
        if device == "cuda" and "CUDAExecutionProvider" not in self.enc.get_providers():
            print("[warn] CUDA requested but onnxruntime fell back to CPU; "
                  "install onnxruntime-gpu matching the CUDA runtime")

        self.pre = self._load_frontend(assets / "preprocessor.ts")
        self._load_vocab(assets / "vocab.json")

    def _load_frontend(self, ts_path: Path):
        """
        Prefer AI4Bharat's own TorchScript frontend, and prove it runs before
        accepting it -- a one-second probe turns a per-clip failure into a startup
        decision, and picks a fallback once rather than 99 times.

        CPU is tried as well as the model device, because the graph bakes its Hann
        window in as a TorchScript constant which `.to(cuda)` does not move: the
        stft then sees a CUDA signal against a CPU window and raises. Mel
        extraction is a few milliseconds either way, so running the frontend on
        CPU costs nothing and keeps the authoritative feature pipeline rather than
        substituting my own reconstruction of it.
        """
        torch = self.torch
        devices = [self.device] + (["cpu"] if self.device != "cpu" else [])
        for dev in devices:
            try:
                ts = torch.jit.load(str(ts_path), map_location=torch.device(dev))
                ts.eval()
                ts.to(torch.device(dev))
                probe = torch.zeros(1, SAMPLE_RATE, device=dev)
                plen = torch.tensor([SAMPLE_RATE], dtype=torch.int64, device=dev)
                with torch.no_grad():
                    feats, _ = ts(probe, plen)
                if feats.shape[1] != MelFrontend.N_MELS:
                    raise RuntimeError(f"frontend emitted {feats.shape[1]} bins, "
                                       f"encoder wants {MelFrontend.N_MELS}")
                print(f"[env ] frontend: AI4Bharat TorchScript on {dev}")
                self.pre_device = dev
                return ts
            except Exception as exc:  # noqa: BLE001
                print(f"[env ] frontend: TorchScript on {dev} failed -- "
                      f"{type(exc).__name__}: {str(exc).strip().splitlines()[-1][:160]}")

        print("[warn] falling back to a reimplemented frontend; verify the decode "
              "is real words, not same-script noise")
        self.pre_device = self.device
        return MelFrontend(self.device)

    def _load_vocab(self, path: Path) -> None:
        """
        vocab.json is {lang: <that language's tokens>}. The aggregate tokenizer
        concatenates the blocks in key order, so global id = block offset + local
        id, and blank is the final index.
        """
        raw = json.loads(path.read_text(encoding="utf-8"))
        self.id2tok: list[str] = []
        self.id2lang: list[str] = []
        for lang, block in raw.items():
            if isinstance(block, dict):
                items = list(block.items())
                if all(str(v).lstrip("-").isdigit() for _, v in items):
                    toks = [k for k, _ in sorted(items, key=lambda kv: int(kv[1]))]
                else:
                    toks = [v for _, v in sorted(items, key=lambda kv: int(kv[0]))]
            else:
                toks = list(block)
            # vocab.json ships 257 entries per language while the aggregate is
            # 22 x 256 = 5632, so exactly one entry per block is surplus. It is
            # the *trailing* one: <unk> stays at local index 0. Verified on a
            # Marathi clip -- block[:256] decodes "namaskar mi gaurav joshi ani
            # mi amol kadkar ..." matching Whisper, while dropping the leading
            # <unk> shifts every token by one inside its block and yields
            # mixed-script salad with the right scripts but wrong characters.
            toks = toks[:BLOCK_SIZE]
            self.id2tok.extend(toks)
            self.id2lang.extend([lang] * len(toks))

        self.blank_id = len(self.id2tok)
        out_dim = self.ctc.get_outputs()[0].shape[-1]
        if isinstance(out_dim, int) and out_dim != self.blank_id + 1:
            raise SystemExit(f"vocab/graph mismatch: built {self.blank_id} tokens "
                             f"but the CTC head emits {out_dim} classes")
        print(f"[env ] vocab {self.blank_id} tokens over {len(set(self.id2lang))} "
              f"languages, blank id {self.blank_id}")

    def _forward(self, pcm):
        """One chunk of audio -> per-block scores, n_frames, sec/frame.

        Returns (best_idx, best_val, blank_val, n, frame_sec) where best_idx and
        best_val are [n, 22]: the winning token within each language block and
        its score. The global argmax is NOT taken here, because it is not a
        meaningful operation on this head -- see _decode_ids.
        """
        torch = self.torch
        # The frontend may live on a different device than the model (see
        # _load_frontend); ONNX takes numpy either way, so this costs one copy.
        dev = torch.device(self.pre_device)
        sig = torch.from_numpy(pcm).unsqueeze(0).to(dev)
        length = torch.tensor([pcm.shape[0]], dtype=torch.int64, device=dev)

        with torch.no_grad():
            feats, feat_len = self.pre(sig, length)

        enc_out, enc_len = self.enc.run(
            None,
            {"audio_signal": feats.cpu().numpy().astype("float32"),
             "length": feat_len.cpu().numpy().astype("int64")},
        )
        (logprobs,) = self.ctc.run(None, {"encoder_output": enc_out})

        import numpy as np

        n = int(enc_len[0])
        lp = logprobs[0, :n]
        n_lang = self.blank_id // BLOCK_SIZE
        per_block = lp[:, :self.blank_id].reshape(n, n_lang, BLOCK_SIZE)
        best_idx = per_block.argmax(-1).astype("int32")
        best_val = per_block.max(-1).astype("float32")
        blank_val = lp[:, self.blank_id].astype("float32")

        # Derived, not assumed: the subsampling factor is whatever makes the
        # encoder's frame count match the audio we actually fed it.
        frame_sec = (pcm.shape[0] / SAMPLE_RATE) / max(n, 1)
        return best_idx, best_val, blank_val, n, frame_sec

    def _decode_ids(self, best_idx, best_val, blank_val, lang_block: int | None):
        """Per-block scores -> one token id per frame.

        The CTC head was exported with `multisoftmax: True`: during training the
        softmax ran over ONE language's 256-token block, so the model was never
        asked to compare a Kannada logit against a Marathi one. Those scores are
        on incomparable scales, and a global argmax over all 5632 therefore picks
        a different block almost every frame -- which is exactly what we saw:
        phonetically correct words spelled in six scripts at once.

        With `lang_block` set, the argmax is restricted to that block, which is
        the calibrated comparison the head was actually trained to make. With it
        None, the old unrestricted behaviour is reproduced exactly, so the two
        can be scored against each other as an ablation rather than argued about.
        """
        import numpy as np

        if lang_block is None:
            chosen = best_val.argmax(-1)                       # per-frame block
            val = best_val[np.arange(len(chosen)), chosen]
            idx = best_idx[np.arange(len(chosen)), chosen]
        else:
            chosen = np.full(len(best_val), lang_block, dtype="int64")
            val = best_val[:, lang_block]
            idx = best_idx[:, lang_block]

        ids = chosen * BLOCK_SIZE + idx
        return np.where(blank_val >= val, self.blank_id, ids)

    def _tokens_to_words(self, ids, frame_sec: float, offset: float):
        """
        Greedy CTC collapse, then group BPE pieces into words on the
        SentencePiece word-boundary marker.

        Timing caveat, stated plainly: CTC gives the frame at which a token was
        *emitted*, not the interval it covers, and emission tends to lag acoustic
        onset. These boundaries are good to roughly a frame and are not a forced
        alignment. That is adequate for attributing a word to a speaker turn,
        which is all Stage 4 needs of them.
        """
        pieces = []
        prev = -1
        for t, idx in enumerate(ids):
            idx = int(idx)
            if idx != self.blank_id and idx != prev:
                pieces.append((self.id2tok[idx], self.id2lang[idx], t))
            prev = idx

        grouped = []
        cur, cur_langs, cur_start, cur_end = "", [], None, None
        for tok, lang, frame in pieces:
            starts_word = tok.startswith(WORD_MARK)
            text = tok[1:] if starts_word else tok
            if starts_word and cur:
                grouped.append((cur, cur_langs, cur_start, cur_end))
                cur, cur_langs, cur_start = "", [], None
            if cur_start is None:
                cur_start = frame
            cur += text
            cur_langs.append(lang)
            cur_end = frame
        if cur:
            grouped.append((cur, cur_langs, cur_start, cur_end))

        out = []
        for text, langs, f0, f1 in grouped:
            if not text:
                continue
            out.append({
                "w": text,
                "start": round(offset + f0 * frame_sec, 3),
                "end": round(offset + (f1 + 1) * frame_sec, 3),
                "lang": max(set(langs), key=langs.count),
            })
        return out

    def transcribe(self, pcm, duration: float) -> dict:
        n_chunk = int(CHUNK_SEC * SAMPLE_RATE)
        n_step = int((CHUNK_SEC - OVERLAP_SEC) * SAMPLE_RATE)

        import numpy as np

        # Pass 1: score every chunk, keeping only the per-block winners. The
        # language must be decided over the WHOLE clip -- deciding per chunk
        # would let a 30 s stretch of noise switch scripts mid-transcript, and
        # the corpus has one language per clip.
        chunks = []
        pos = 0
        while pos < pcm.shape[0]:
            seg = pcm[pos:pos + n_chunk]
            if seg.shape[0] < SAMPLE_RATE // 10:  # <100 ms tail, nothing to decode
                break
            best_idx, best_val, blank_val, _n, frame_sec = self._forward(seg)
            chunks.append((pos, best_idx, best_val, blank_val, frame_sec))
            if pos + n_chunk >= pcm.shape[0]:
                break
            pos += n_step

        # Language identification, from the model's own logits and nothing else:
        # count the frames each block would win, ignoring frames the blank takes.
        # Validated earlier -- on the Marathi clip this puts mr first at 48% and
        # the Devanagari blocks together at 73%, far from the 4.5% of noise.
        votes = np.zeros(self.blank_id // BLOCK_SIZE, dtype="int64")
        for _pos, _bi, best_val, blank_val, _fs in chunks:
            winner = best_val.argmax(-1)
            speech = best_val[np.arange(len(winner)), winner] > blank_val
            np.add.at(votes, winner[speech], 1)
        lang_block = int(votes.argmax()) if votes.sum() else 0
        lang_frames = {self.id2lang[b * BLOCK_SIZE]: int(v)
                       for b, v in enumerate(votes) if v}

        # Pass 2: decode each chunk under that decision.
        words = []
        for pos, best_idx, best_val, blank_val, frame_sec in chunks:
            offset = pos / SAMPLE_RATE
            ids = self._decode_ids(best_idx, best_val, blank_val,
                                   lang_block if self.lang_lock else None)
            chunk_words = self._tokens_to_words(ids, frame_sec, offset)

            # Keep only words whose midpoint falls in this chunk's core, so the
            # overlap region is claimed by exactly one chunk and a word split by
            # a cut is recovered whole from the neighbour that saw all of it.
            first = pos == 0
            last = pos + n_chunk >= pcm.shape[0]
            lo = offset if first else offset + OVERLAP_SEC / 2
            hi = offset + CHUNK_SEC - OVERLAP_SEC / 2
            for w in chunk_words:
                mid = (w["start"] + w["end"]) / 2
                if (first or mid >= lo) and (last or mid < hi):
                    words.append(w)

        words.sort(key=lambda w: w["start"])
        langs = [w["lang"] for w in words]
        return {
            "words": words,
            "lang": self.id2lang[lang_block * BLOCK_SIZE] if self.lang_lock
                    else (max(set(langs), key=langs.count) if langs else None),
            "lang_locked": self.lang_lock,
            # Frame votes per block: the model's own language ID, and a useful
            # confidence signal -- a clip whose top two blocks are close is one
            # to look at in the per-condition analysis.
            "lang_frames": dict(sorted(lang_frames.items(), key=lambda kv: -kv[1])),
            "lang_counts": {lg: langs.count(lg) for lg in sorted(set(langs))},
        }


# --------------------------------------------------------------------------
# backend: Whisper large-v3 via faster-whisper
# --------------------------------------------------------------------------

class WhisperBackend:
    """
    faster-whisper rather than WhisperX: WhisperX refines timestamps with
    per-language wav2vec2 alignment models, which do not exist for most of the
    nine Indic scripts in this corpus. Whisper's own cross-attention DTW word
    timestamps are coarser but exist for every language here, and a metric that
    silently degrades for some languages and not others is worse than one that
    is uniformly approximate.
    """

    name = "whisper"

    def __init__(self, device: str):
        from faster_whisper import WhisperModel

        compute = "float16" if device == "cuda" else "int8"
        self.model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute)
        print(f"[env ] faster-whisper {WHISPER_MODEL} ({compute})")

    def transcribe(self, pcm, duration: float) -> dict:
        segments, info = self.model.transcribe(
            pcm,
            word_timestamps=True,
            vad_filter=False,                   # diarization owns speech/non-speech
            condition_on_previous_text=False,   # stops a hallucination loop from
                                                # propagating across the whole clip
            temperature=0.0,                    # see below: greedy, and repeatable
        )
        # faster-whisper defaults to temperature [0, 0.2, ..., 1.0]: a segment
        # that trips the compression-ratio or avg-logprob check is re-decoded at
        # the next temperature, and above zero that samples instead of taking the
        # argmax, unseeded. Two costs, both measured on Tlha36rSd5o (318 ref
        # words). It is not reproducible -- three identical calls returned 87, 84
        # and 100 words. And it is worse: the fallback keeps a sampled draw over
        # the greedy decode it started from, so disabling it returned 211 words,
        # the same 211 every run. A benchmark number that no one can reproduce is
        # not a benchmark number, and here determinism was also the better decode.
        #
        # The fallback was the only guard against a degenerate repeat loop, so
        # `trips` below is now the sole monitor for one. Watch it in the manifest.

        words, trips = [], 0
        for seg in segments:
            # Flag rather than drop: a discarded segment is an invisible deletion
            # that inflates the miss rate for a reason nothing downstream records.
            if seg.compression_ratio > 2.4 or seg.no_speech_prob > 0.6:
                trips += 1
            for w in (seg.words or []):
                text = w.word.strip()
                if text:
                    words.append({"w": text,
                                  "start": round(w.start, 3),
                                  "end": round(w.end, 3)})

        return {
            "words": words,
            "lang": info.language,
            "lang_prob": round(float(info.language_probability), 4),
            "suspect_segments": trips,
        }


def build_backend(system: str, device: str):
    if system == "whisper":
        return WhisperBackend(device)
    if system == "indicconformer":
        return IndicConformerBackend(device, lang_lock=True)
    if system == "indicconformer_free":
        # The ablation: same weights, same words, global argmax across all 22
        # blocks. Kept as a first-class system so the cost of getting this wrong
        # is a measured WER delta in the results table, not a claim.
        return IndicConformerBackend(device, lang_lock=False)
    raise SystemExit(f"unknown system {system!r}; expected one of {SUPPORTED}")


# --------------------------------------------------------------------------

def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--system", required=True, choices=SUPPORTED)
    ap.add_argument("--data", default="data", help="output root")
    ap.add_argument("--wav-dir", required=True, help="16 kHz mono wavs from Stage 1")
    ap.add_argument("--limit", type=int, default=None, help="smoke test: first N clips")
    ap.add_argument("--device", default=None, help="cuda|cpu (default: auto)")
    args = ap.parse_args()

    try:
        import torch
        device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    except ImportError:
        device = args.device or "cpu"
    print(f"[env ] device={device}")

    wav_dir = Path(args.wav_dir)
    wavs = sorted(wav_dir.glob("*.wav"))
    if not wavs:
        raise SystemExit(f"no wavs under {wav_dir}")

    out_root = Path(args.data) / "asr" / args.system
    words_dir = out_root / "words"
    manifest = Manifest(out_root / "manifest.jsonl")

    # Count completion from the manifest before --limit truncates the list;
    # deriving "done" from len(wavs) - len(todo) reports the limit instead of
    # actual progress, which reads as a finished run when nothing is finished.
    pending = [p for p in wavs if not manifest.done(p.stem)]
    todo = pending[:args.limit] if args.limit else pending
    print(f"[plan] {len(wavs)} clips: {len(wavs) - len(pending)} done, "
          f"{len(pending)} pending, running {len(todo)} now")
    if not todo:
        return

    backend = build_backend(args.system, device)

    n_ok = n_fail = 0
    total_audio = total_wall = 0.0
    for i, path in enumerate(todo, 1):
        clip_id = path.stem
        t0 = time.time()
        try:
            pcm = read_wav(path)
            duration = pcm.shape[0] / SAMPLE_RATE
            result = backend.transcribe(pcm, duration)
            wall = time.time() - t0

            write_json(words_dir / f"{clip_id}.json", {
                "clip_id": clip_id,
                "system": args.system,
                "duration": round(duration, 3),
                **result,
            })
            rec = {"clip_id": clip_id, "status": "ok",
                   "n_words": len(result["words"]),
                   "lang": result.get("lang"),
                   "duration": round(duration, 3),
                   "wall_sec": round(wall, 2),
                   "rtf": round(wall / duration, 4) if duration else None}
            n_ok += 1
            total_audio += duration
            total_wall += wall
        except Exception as exc:  # noqa: BLE001 -- one bad clip must not end the run
            wall = time.time() - t0
            rec = {"clip_id": clip_id, "status": "fail",
                   "error": f"{type(exc).__name__}: {exc}",
                   "wall_sec": round(wall, 2)}
            n_fail += 1
            print(f"[fail] {clip_id}: {type(exc).__name__}: {exc}", file=sys.stderr)

        manifest.append(rec)
        print(f"[{i:3d}/{len(todo)}] {clip_id[:40]:40s} {rec['status']:4s} "
              f"{str(rec.get('n_words', '-')):>6} words  rtf={rec.get('rtf', '-')}")

        # Long clips fragment the allocator; releasing between clips stops one
        # from poisoning the clips after it (the Sortformer lesson from Stage 3).
        if device == "cuda":
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass

    print(f"\n[done] ok={n_ok} fail={n_fail}")
    if total_audio:
        print(f"[done] {total_audio / 3600:.2f} h audio in {total_wall / 60:.1f} min "
              f"(mean RTF {total_wall / total_audio:.4f})")


if __name__ == "__main__":
    main()

In [ ]:
label = "stage 4a: ASR whisper"
if not HAVE_GPU:
    skip(label, "no GPU")
elif (WHISPER_PY := gpu_env("whisper", "faster-whisper")) is None:
    skip(label, "environment install failed")
else:
    run(label, ["stage4_asr.py", "--system", "whisper", "--data", GPU_ROOT,
                "--wav-dir", AUDIO] + GPU_LIMIT, python=WHISPER_PY, may_fail=GPU_MAY_FAIL)

In [ ]:
# The ONNX export is a gated repo as well, so without a token this fails mid-download
# with GatedRepoError rather than at the start.
INDIC_PY = gpu_env("indic", "onnxruntime-gpu==1.20.2", "librosa") if (HAVE_GPU and HF_TOKEN) else None
for system in ("indicconformer", "indicconformer_free"):
    label = f"stage 4a: ASR {system}"
    if not HAVE_GPU:
        skip(label, "no GPU")
    elif not HF_TOKEN:
        skip(label, "no HF_TOKEN (ai4bharat/indic-conformer-600m-multilingual is gated)")
    elif INDIC_PY is None:
        skip(label, "environment install failed")
    else:
        # The script prints its onnxruntime providers; CUDAExecutionProvider must be
        # among them, or this is the (slow, correct) CPU fallback.
        run(label, ["stage4_asr.py", "--system", system, "--data", GPU_ROOT,
                    "--wav-dir", AUDIO] + GPU_LIMIT, python=INDIC_PY, may_fail=GPU_MAY_FAIL)

In [ ]:
import difflib

def words(path):
    return [w["w"] for w in json.loads(path.read_text(encoding="utf-8"))["words"]]

if not FULL_GPU_RUN:
    for system in ("whisper", "indicconformer", "indicconformer_free"):
        live_dir = WORKDIR / "smoke" / "asr" / system / "words"
        for f in sorted(live_dir.glob("*.json")) if live_dir.is_dir() else []:
            live, ref = words(f), words(CACHE / "asr" / system / "words" / f.name)
            sim = difflib.SequenceMatcher(None, live, ref, autojunk=False).ratio()
            print(f"{system:20s} {f.stem}  {len(live):5d} words vs {len(ref):5d} in the reference run, "
                  f"{100 * sim:5.1f}% identical sequence")
    for system in ("whisper", "indicconformer", "indicconformer_free"):
        restore(f"asr/{system}")

## Stage 4b — language-ID fallback (the adopted improvement) and attribution

**Language-ID fallback.** On 13 clips IndicConformer's own vote picks Urdu (11) or
Nepali (2), outside the languages the task serves, and the whole clip comes out in the
wrong script: 100% WER whatever was heard. Spoken Hindi and Urdu are nearly identical
and differ mainly in script, so this is real ambiguity, not a decoder bug. The rule:
if IndicConformer's detected language is outside {hi, mr, bn, gu, kn, ml, or, pa, ta,
te}, take Whisper's words for that clip. **No reference is read.** It is written as a
fourth ASR system, `ic_lid_fallback`.

**Attribution rules.** A word goes to the turn it overlaps most (ties to the earlier
turn). A word with no turn is kept and flagged, never dropped: dropping it would
reward a diarizer for missing speech. `--diar ref` is an *oracle* diagnostic.

Expected: the offline `sortformer` conditions exit non-zero, with **25 clips** failing
`no turns in RTTM` — the out-of-memory clips from Stage 3. A clip with no hypothesis
is not a clip where nobody spoke, so it fails loudly instead of being written empty.
The next cell checks that those are the only failures.

In [ ]:
%%writefile stage4_attribute.py
#!/usr/bin/env python3
"""
Stage 4b -- Speaker attribution (CPU-only, seconds per condition).

Takes the words produced by Stage 4a, which never saw a speaker, and labels each
one using ONE diarization hypothesis. Every (asr, diar) pair is a separate
condition written to its own directory, so a cpWER difference between two
conditions is attributable to the thing that differs -- the labelling -- and
never to the ASR having been handed a different slice of audio.

    python stage4_attribute.py --asr whisper --diar pyannote31 sortformer ref

Writes data/attrib/<asr>__<diar>/<clip_id>.json and a manifest per condition.

Attribution rules, all four of which move the score and so are stated here
rather than buried:

  1. MAXIMUM OVERLAP. A word goes to the turn sharing the most time with its
     [start, end] interval. The cheaper rule -- assign by midpoint -- discards
     information exactly where it is most needed, on the long words that
     straddle a turn boundary. Equal overlap breaks toward the earlier turn, so
     the output is deterministic rather than dict-order dependent.

  2. ORPHANS ARE KEPT, NOT DROPPED. A word can land where the diarizer heard
     nothing. Dropping it deletes it from the hypothesis, which shows up in
     cpWER as a deletion and quietly *rewards* a system for missing speech.
     Instead the word is given the nearest turn and flagged `orphan`, so the
     error becomes a substitution when the guess is wrong, and Stage 6 can
     report how much of each system's cpWER came from this rule rather than
     from genuine labelling mistakes.

  3. OVERLAPPED SPEECH needs no special case: rule 1 hands the word to whichever
     simultaneous speaker covers more of it. Contested words are counted so the
     cost of that choice stays visible, and the two ways a word can be contested
     are counted apart -- `n_overlap_words` for genuinely simultaneous speakers,
     `n_boundary_words` for a word crossing between two disjoint turns. Merging
     them would drown the first in the second: the corpus is 7.60% overlapped,
     while every turn change produces boundary words regardless.

  4. `--diar ref` is an ORACLE condition. It attributes with the Stage 2
     reference RTTM, giving a cpWER floor where labelling is perfect by
     construction -- so every other condition reads as "ASR error + what this
     diarizer cost on top". It is a diagnostic only: nothing produced from it is
     fed back into any model, and it must be labelled `oracle` wherever it
     appears in a results table.
"""

from __future__ import annotations

import argparse
import bisect
import json
import os
import sys
from pathlib import Path

ORACLE = "ref"


# --------------------------------------------------------------------------
# manifest -- append-only and fsync'd, matching Stage 4a
# --------------------------------------------------------------------------

class Manifest:
    def __init__(self, path: Path):
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.records: dict[str, dict] = {}
        if self.path.exists():
            for line in self.path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue  # truncated final line from a hard kill
                self.records[rec["clip_id"]] = rec

    def done(self, clip_id: str) -> bool:
        return self.records.get(clip_id, {}).get("status") == "ok"

    def append(self, rec: dict) -> None:
        self.records[rec["clip_id"]] = rec
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()
            os.fsync(fh.fileno())


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)  # atomic: a reader never sees a half-written file


# --------------------------------------------------------------------------
# turns
# --------------------------------------------------------------------------

def load_turns(path: Path) -> list[tuple[float, float, str]]:
    """RTTM -> [(start, end, speaker)] sorted by start.

    Parsed by hand for the same reason Stage 3b does it: the loader moved
    between pyannote releases, and this stage should not need pyannote at all.
    Zero- and negative-duration turns are dropped -- they can never win a
    maximum-overlap comparison, but they would pollute the speaker inventory.
    """
    turns: list[tuple[float, float, str]] = []
    if not path.exists():
        return turns
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        p = line.split()
        if len(p) < 8 or p[0] != "SPEAKER":
            continue
        start, dur, spk = float(p[3]), float(p[4]), p[7]
        if dur <= 0:
            continue
        turns.append((start, start + dur, spk))
    turns.sort(key=lambda t: (t[0], t[1]))
    return turns


def attribute_word(w_start: float, w_end: float, turns, starts, max_dur: float):
    """One word -> (speaker, overlap_seconds, touched_spans).

    `touched_spans` is each intersecting turn clipped to the word, which is what
    lets the caller tell the two ways a word can touch two turns apart:
    simultaneous speech, or a boundary crossing between disjoint turns.

    Only turns that begin within `max_dur` before the word can reach it, so the
    scan starts there instead of at turn zero. Turns may overlap each other, so
    a plain bisect window is not enough on its own -- the longest turn in the
    clip sets how far back to look.
    """
    best_spk, best_ov, spans = None, 0.0, []
    i = bisect.bisect_left(starts, w_start - max_dur)
    for t_start, t_end, spk in turns[i:]:
        if t_start >= w_end:
            break  # sorted by start: nothing later can overlap
        lo, hi = max(w_start, t_start), min(w_end, t_end)
        if hi > lo:
            spans.append((lo, hi))
            # Strict >: ties keep the earlier turn, which the sort fixed.
            if hi - lo > best_ov:
                best_spk, best_ov = spk, hi - lo
    return best_spk, best_ov, spans


def is_simultaneous(spans: list[tuple[float, float]]) -> bool:
    """True if two speakers are active at the same instant inside the word.

    A word crossing the boundary between two disjoint turns also touches two
    turns, and counting that as overlapped speech would make the overlap
    diagnostic mostly boundary noise -- the corpus is only 7.60% overlapped, so
    the distinction is the whole measurement.
    """
    spans = sorted(spans)
    return any(b[0] < a[1] for a, b in zip(spans, spans[1:]))


def nearest_turn(w_start: float, w_end: float, turns) -> str | None:
    """Speaker of the turn with the smallest gap to this word. Ties -> earlier."""
    best_spk, best_gap = None, None
    for t_start, t_end, spk in turns:
        gap = t_start - w_end if t_start > w_end else w_start - t_end
        gap = max(gap, 0.0)
        if best_gap is None or gap < best_gap:
            best_spk, best_gap = spk, gap
    return best_spk


# --------------------------------------------------------------------------

def attribute_clip(words: list[dict], turns) -> dict:
    starts = [t[0] for t in turns]
    max_dur = max((e - s for s, e, _ in turns), default=0.0)

    out, n_orphan, n_overlap, n_boundary = [], 0, 0, 0
    for w in words:
        spk, ov, spans = attribute_word(w["start"], w["end"], turns, starts, max_dur)
        rec = {"w": w["w"], "start": w["start"], "end": w["end"]}
        if "lang" in w:
            rec["lang"] = w["lang"]
        if spk is None:
            # Rule 2: keep it, attributed to the nearest turn, and say so.
            spk = nearest_turn(w["start"], w["end"], turns)
            rec["orphan"] = True
            n_orphan += 1
        elif len(spans) > 1:
            # Both flags mark a contested word, and attribution errors
            # concentrate in them -- but they are contested for different
            # reasons and Stage 6 should be able to separate the two.
            if is_simultaneous(spans):
                rec["overlap"] = len(spans)
                n_overlap += 1
            else:
                rec["boundary"] = True
                n_boundary += 1
        rec["spk"] = spk
        out.append(rec)

    by_speaker: dict[str, list[str]] = {}
    for rec in out:
        by_speaker.setdefault(rec["spk"], []).append(rec["w"])

    return {
        "words": out,
        "by_speaker": {k: " ".join(v) for k, v in sorted(by_speaker.items())},
        "n_words": len(out),
        "n_orphan": n_orphan,
        "n_overlap_words": n_overlap,
        "n_boundary_words": n_boundary,
        "n_turns": len(turns),
        "speakers": sorted({t[2] for t in turns}),
    }


def rttm_dir(data: Path, diar: str) -> Path:
    return data / "ref" / "rttm" if diar == ORACLE else data / "hyp" / diar / "rttm"


def run_condition(asr: str, diar: str, data: Path, limit: int | None) -> bool:
    words_dir = data / "asr" / asr / "words"
    turns_dir = rttm_dir(data, diar)
    if not words_dir.is_dir():
        print(f"[skip] {asr}__{diar}: no words at {words_dir}")
        return False
    if not turns_dir.is_dir():
        print(f"[skip] {asr}__{diar}: no RTTMs at {turns_dir}")
        return False

    out_root = data / "attrib" / f"{asr}__{diar}"
    manifest = Manifest(out_root / "manifest.jsonl")
    clips = sorted(p.stem for p in words_dir.glob("*.json"))

    pending = [c for c in clips if not manifest.done(c)]
    todo = pending[:limit] if limit else pending
    tag = f"{asr}__{diar}" + ("  [ORACLE -- diagnostic only]" if diar == ORACLE else "")
    print(f"\n[cond] {tag}")
    print(f"[plan] {len(clips)} clips: {len(clips) - len(pending)} done, "
          f"{len(pending)} pending, running {len(todo)} now")

    n_ok = n_fail = 0
    tot_words = tot_orphan = tot_overlap = tot_boundary = 0
    for clip_id in todo:
        try:
            src = json.loads((words_dir / f"{clip_id}.json").read_text(encoding="utf-8"))
            turns = load_turns(turns_dir / f"{clip_id}.rttm")
            if not turns:
                # No hypothesis for this clip is a real result, not a crash:
                # Stage 3 left sortformer short on the long clips. Record it as
                # a failure so it is retried if the RTTM appears, and so it can
                # never be mistaken for a clip with zero words.
                raise ValueError("no turns in RTTM")

            res = attribute_clip(src["words"], turns)
            write_json(out_root / f"{clip_id}.json", {
                "clip_id": clip_id,
                "asr": asr,
                "diar": diar,
                "oracle": diar == ORACLE,
                "duration": src.get("duration"),
                "lang": src.get("lang"),
                **res,
            })
            manifest.append({"clip_id": clip_id, "status": "ok",
                             "n_words": res["n_words"],
                             "n_orphan": res["n_orphan"],
                             "n_overlap_words": res["n_overlap_words"],
                             "n_boundary_words": res["n_boundary_words"],
                             "n_turns": res["n_turns"],
                             "n_speakers": len(res["speakers"])})
            n_ok += 1
            tot_words += res["n_words"]
            tot_orphan += res["n_orphan"]
            tot_overlap += res["n_overlap_words"]
            tot_boundary += res["n_boundary_words"]
        except Exception as exc:  # noqa: BLE001 -- one bad clip must not stop 98 good ones
            manifest.append({"clip_id": clip_id, "status": "fail",
                             "error": f"{type(exc).__name__}: {exc}"})
            n_fail += 1
            print(f"  fail: {clip_id[:40]}  {type(exc).__name__}: {exc}")

    if tot_words:
        print(f"[done] ok={n_ok} fail={n_fail}  {tot_words:,} words, "
              f"{tot_orphan:,} orphaned ({100 * tot_orphan / tot_words:.2f}%), "
              f"{tot_overlap:,} in overlapped speech "
              f"({100 * tot_overlap / tot_words:.2f}%), "
              f"{tot_boundary:,} on a turn boundary "
              f"({100 * tot_boundary / tot_words:.2f}%)")
    else:
        print(f"[done] ok={n_ok} fail={n_fail}")
    return n_fail == 0


# --------------------------------------------------------------------------

def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--asr", nargs="+", required=True,
                    help="ASR systems under data/asr/ (e.g. whisper indicconformer)")
    ap.add_argument("--diar", nargs="+", required=True,
                    help=f"diarization systems under data/hyp/, plus '{ORACLE}' "
                         f"for the oracle condition")
    ap.add_argument("--data", default="data", help="pipeline root")
    ap.add_argument("--limit", type=int, default=None, help="smoke test: first N clips")
    args = ap.parse_args()

    data = Path(args.data)
    clean = True
    for asr in args.asr:
        for diar in args.diar:
            clean &= run_condition(asr, diar, data, args.limit)

    if ORACLE in args.diar:
        print(f"\n[note] the '{ORACLE}' condition used the reference RTTM. It is a "
              f"diagnostic upper bound -- label it 'oracle' in every table.")
    return 0 if clean else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile stage4_fallback.py
#!/usr/bin/env python3
"""
Stage 4c -- language-ID fallback: IndicConformer, or Whisper when IndicConformer
does not know what language it is hearing (CPU, seconds, no model runs).

    python stage4_fallback.py --data data
    python stage4_attribute.py --asr ic_lid_fallback --diar pyannote31 sortformer_stream sortformer ref

Writes data/asr/ic_lid_fallback/words/<clip_id>.json in exactly the Stage 4a
format, so attribution, Stage 5 and scoring treat it as one more ASR system.


THE FAILURE IT TARGETS
----------------------
IndicConformer decodes a clip inside ONE language's vocabulary block, chosen by
a frame vote over the model's own logits (stage4_asr.py, `lang_lock`). On 13 of
99 clips that vote lands on a language this task does not serve: 11 Urdu, 2
Nepali. The decode is then spelled in the wrong script for the whole clip --
Perso-Arabic against a Devanagari reference -- and scores 100% WER no matter how
well the acoustics were recognised. Hindi and Urdu are close to one spoken
language written in two scripts, so this vote is near a coin flip on acoustics
alone; it is not a bug to be fixed inside the decoder.

Whisper is not immune to the same confusion, but its language decision comes
from a separate model trained on far more Hindi, and on these clips it is right
far more often.


THE RULE
--------
    if IndicConformer's detected language is in TARGET_LANGS:  keep IndicConformer
    else:                                                      use Whisper's words

Clip-level, all or nothing. Mixing the two systems' words inside a clip would
need a word alignment between hypotheses (ROVER-style) and is out of scope.


WHAT IT DOES AND DOES NOT USE
-----------------------------
Inputs are the two ASR outputs and a fixed list of languages. No reference
transcript, no reference diarization, no per-clip metadata.

TARGET_LANGS is the one judgement call, and it should be stated as such: it is
the set of languages the system is deployed for -- here the nine scripts this
corpus was collected in, fixed once for the whole run. That is task
configuration, the same thing that decides which model is loaded, not a label on
any clip. It was NOT tuned by trying language subsets against the scores: the
rule is "outside the served set", and the set is the task's.

The per-clip consequences are checked, not assumed: stage6_report.py confirms
every kept clip scores identically to `indicconformer` and every switched clip
identically to `whisper`, so the gain cannot come from anything but the switch.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

from stage4_attribute import Manifest, write_json

NAME = "ic_lid_fallback"
PRIMARY = "indicconformer"
FALLBACK = "whisper"

# IndicConformer's language codes for the nine scripts the task serves.
# Devanagari covers both Hindi and Marathi; everything else is one-to-one.
TARGET_LANGS = frozenset({
    "hi", "mr",   # Devanagari
    "bn",         # Bengali
    "gu",         # Gujarati
    "kn",         # Kannada
    "ml",         # Malayalam
    "or",         # Odia
    "pa",         # Punjabi (Gurmukhi)
    "ta",         # Tamil
    "te",         # Telugu
})


def run(data: Path) -> bool:
    src = {s: data / "asr" / s / "words" for s in (PRIMARY, FALLBACK)}
    for s, d in src.items():
        if not d.is_dir():
            print(f"[!] no {s} words at {d} -- run stage4_asr.py --system {s}")
            return False

    out_root = data / "asr" / NAME
    # Rebuilt from scratch every run rather than resumed: it takes seconds, and
    # a stale file from an earlier TARGET_LANGS would otherwise survive.
    manifest_path = out_root / "manifest.jsonl"
    if manifest_path.exists():
        manifest_path.unlink()
    manifest = Manifest(manifest_path)

    clips = sorted(p.stem for p in src[PRIMARY].glob("*.json"))
    switched, missing = [], []
    for clip_id in clips:
        prim = json.loads((src[PRIMARY] / f"{clip_id}.json").read_text(encoding="utf-8"))
        lang = prim.get("lang")
        use_fallback = lang not in TARGET_LANGS

        if use_fallback:
            fb_path = src[FALLBACK] / f"{clip_id}.json"
            if not fb_path.exists():
                # Nothing to fall back to: keep the primary rather than drop the
                # clip, and say so. Dropping would shrink the scored set.
                missing.append(clip_id)
                use_fallback = False
            else:
                chosen = json.loads(fb_path.read_text(encoding="utf-8"))
                switched.append((clip_id, lang, chosen.get("lang")))
        if not use_fallback:
            chosen = prim

        source = FALLBACK if use_fallback else PRIMARY
        write_json(out_root / "words" / f"{clip_id}.json", {
            **chosen,
            "system": NAME,
            "source": source,
            "primary_lang": lang,
        })
        manifest.append({"clip_id": clip_id, "status": "ok",
                         "n_words": len(chosen["words"]),
                         "lang": chosen.get("lang"),
                         "duration": chosen.get("duration"),
                         "source": source,
                         "primary_lang": lang})

    print(f"[fallback] {len(clips)} clips: {len(clips) - len(switched)} kept "
          f"{PRIMARY}, {len(switched)} switched to {FALLBACK}")
    print(f"  served languages: {' '.join(sorted(TARGET_LANGS))}")
    for clip_id, lang, fb_lang in switched:
        print(f"  {clip_id}  {PRIMARY} lang={lang!s:<4} -> {FALLBACK} lang={fb_lang}")
    if missing:
        print(f"  [!] {len(missing)} clip(s) needed a fallback with no {FALLBACK} "
              f"words; kept {PRIMARY}: {missing}")
    print(f"wrote -> {out_root}")
    return not missing


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    ap.add_argument("--data", type=Path, default=Path("data"))
    args = ap.parse_args()
    return 0 if run(args.data) else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
run("stage 4b: LID fallback", ["stage4_fallback.py", "--data", "data"])

ASR = ["indicconformer", "indicconformer_free", "whisper", "ic_lid_fallback"]
DIAR = list(DIAR_SYSTEMS)
run("stage 4b: attribute words", ["stage4_attribute.py", "--asr", *ASR,
                                  "--diar", *DIAR, "ref", "--data", "data"], may_fail=True)

In [ ]:
from collections import Counter

failures = {}
for m in sorted((DATA / "attrib").glob("*/manifest.jsonl")):
    last = {}
    for line in m.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            last[r["clip_id"]] = r
    bad = [r for r in last.values() if r["status"] != "ok"]
    if bad:
        failures[m.parent.name] = Counter(r["error"] for r in bad)
for cond, errs in failures.items():
    print(cond, dict(errs))
assert set(failures) == {f"{a}__sortformer" for a in ASR}, "unexpected attribution failures"
assert all(errs == {"ValueError: no turns in RTTM": 25} for errs in failures.values())
for row in REPORT:
    if row["step"] == "stage 4b: attribute words" and row["exit"] == 1:
        row["status"] = "ok (expected: 25 OOM sortformer clips)"
print(f"{len(list((DATA / 'attrib').iterdir()))} conditions; the only failures are the "
      "25 out-of-memory sortformer clips, as expected")

## Stage 5 — speaker relabelling from the transcript (built, measured, not adopted)

After DiarizationLM and lexical speaker error correction. Transcripts are cut into
same-speaker runs split at pauses over 0.5 s. A unit may only be moved to an existing
speaker, so **the text cannot change and WER must not move** — asserted per clip.

- **Ceiling first** (`--audit`, an oracle diagnostic that edits nothing): even after
  pause splitting, 37.87% of words sit in units spanning two true speakers, where no
  relabel can help.
- **Rule baseline** (CPU): a unit under 1 s joins its nearer neighbour.
- **LLM** (GPU): Qwen2.5-7B-Instruct in 4-bit sees 25 units at a time and returns
  JSON edits with confidences; below 0.7 is dropped, and a clip where it tries to edit
  over 30% of units is discarded. It made WDER worse (20.12 → 20.96).

In [ ]:
%%writefile stage5_correct.py
#!/usr/bin/env python3
"""
Stage 5 -- LLM speaker-label correction.

Takes one Stage 4b condition and rewrites speaker labels using an open-weights
LLM reading the transcript as discourse. Writes a new condition that Stage 4c
scores exactly like any other, so baseline and improved rows sit side by side.

    python stage5_correct.py --cond indicconformer__pyannote31 --method rule
    python stage5_correct.py --cond indicconformer__pyannote31 --method llm
    python stage5_correct.py --cond indicconformer__pyannote31 --audit

Output goes to data/attrib/<asr>__<diar>+<method>/. Stage 4c splits a condition
name on the first `__`, so the suffix rides along on the diarizer and every
variant sorts directly under its baseline: `pyannote31`, `pyannote31+rule`,
`pyannote31+llm` read down the results table in that order.

Methods live in the METHODS registry and share one signature,
`(units, speakers, lang, llm, min_conf) -> ([(unit, speaker, confidence)],
rejections)`. Adding the acoustic variant later is one function and one registry
entry; run(), the manifest and Stage 4c need no change. That is deliberate --
Rule, LLM and LLM+acoustic have to be scorable side by side, and they can only
stay comparable if they differ in the edit proposal and in nothing else.


WHAT IT TARGETS, AND WHY NOT cpWER
----------------------------------
Measured attribution_cost (cpWER - DI-cpWER) is 0.01-0.33 across all twelve
Stage 4 conditions. There is no cpWER headroom, and an "improvement" of that
size would be noise. Stage 5 targets WDER, where the spread is real: 20.33 for
pyannote31 against 39.53 for sortformer_stream on the same IndicConformer words.

This also matches the goal. False splits, false merges and speaker swaps are
all speaker-structure errors; none of them is a text error. Stage 5 never edits
a word.


THE EDIT SPACE IS RELABEL, AND ONLY RELABEL
-------------------------------------------
The model may say "unit 7 belongs to Speaker_A". It may not add a speaker,
move a boundary, or touch text.

That is not a limitation, it is the whole design:

  * cpWER, DI-cpWER and WDER are functions of the word -> speaker map alone, so
    relabel is a COMPLETE edit space for every metric this project reports.
  * WER ignores speakers entirely, so WER CANNOT MOVE. A Stage 5 run that
    changes WER by 0.01 is broken. That is a free tripwire and it is asserted
    here rather than left for someone to notice in the results table.

Units are contiguous same-speaker runs, split further at pauses longer than
GAP_SEC. The split matters: with maximal runs, relabel would fix false splits
and swaps but never a false merge, because two speakers inside one run cannot be
separated by relabelling the run. A real speaker change usually has a pause at
it, so splitting on pauses turns many false merges into two units that relabel
can then fix -- without asking the model to point at a word index inside 79%-WER
text, which it would do badly.

"Usually" is doing real work in that sentence, so `--audit` measures it rather
than asserting it. It reports how many units still span two reference speakers
after splitting -- the false merges relabel-only correction can never reach --
against how many splitting rescued. That number is the ceiling on this whole
stage, and it belongs in the writeup next to any improvement claimed.


ABSTENTION
----------
Each edit carries a confidence and anything below --min-conf is discarded. A
wrong relabel costs WDER twice, removing a correct word from one speaker and
adding a wrong one to another, while an abstention costs only the improvement
forgone -- so at 79-87% WER the bias should be conservative and the default
threshold is high. A missing confidence is treated as a rejection, not as
certainty: a model that ignores the schema is exactly the one whose edits should
not bypass the threshold. Both kinds of abstention are counted and reported.


THE RISK, STATED UP FRONT
-------------------------
These transcripts are 79-87% WER. Whether an LLM can recover discourse structure
from text that damaged is genuinely unknown until it is measured. The design
fails safe: no parse, no edits, and the condition scores identically to its
baseline. Every way the model can be ignored is counted in the manifest, so
"the LLM was overruled on N clips" is itself a reportable result.

Generation is greedy (do_sample=False). Stage 4 was bitten once by an unseeded
sampler making a benchmark irreproducible; the same mistake is not available
here.


THE RULE BASELINE
-----------------
`--method rule` relabels any unit shorter than MIN_UNIT_SEC to the neighbour it
is closer to. It costs no GPU and captures the single most common diarization
artefact -- a sliver of a turn dropped inside someone else's speech. Without it
"the LLM improved WDER" is unfalsifiable: the comparison that matters is against
a dumb method, not against doing nothing.
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

from stage4_attribute import Manifest, write_json

# A unit break needs a pause this long. Below it, a label change is far more
# likely to be diarizer jitter than a real speaker change.
GAP_SEC = 0.5

# Units shorter than this are what --method rule rewrites.
MIN_UNIT_SEC = 1.0

# Above this fraction of units edited, the model is not correcting a transcript,
# it is rewriting one. Drop the whole clip's edits and record it.
#
# The floor matters as much as the fraction: on a 3-unit clip a single correct
# edit is 33% and would trip a bare percentage, so a clip has to exceed BOTH to
# count as rogue. Without it the guard fires hardest on the short clips where
# there is least to get wrong.
MAX_EDIT_FRAC = 0.30
MIN_ROGUE_EDITS = 3

# Units per prompt, and how many trailing units of the previous window are
# reshown as context. Context units are visible but not editable, so no unit is
# ever decided twice.
WINDOW = 25
CONTEXT = 4

# Per-unit character cap in the prompt. Long units are informative for a few
# words and then just spend budget.
UNIT_CHARS = 220

# An edit below this confidence is discarded. At 79-87% WER the model is reading
# heavily corrupted text, so the default is deliberately high: a wrong relabel
# costs WDER twice over -- it removes a correct word from one speaker and adds a
# wrong one to another -- while an abstention costs nothing beyond the
# improvement forgone. Conservative is the right bias here.
MIN_CONF = 0.7

MODEL = "Qwen/Qwen2.5-7B-Instruct"

LANG_NAME = {
    "hi": "Hindi", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "kn": "Kannada", "ml": "Malayalam", "ta": "Tamil", "te": "Telugu",
    "pa": "Punjabi", "or": "Odia", "ur": "Urdu", "ne": "Nepali", "en": "English",
}


# --------------------------------------------------------------------------
# units
# --------------------------------------------------------------------------

def build_units(words: list[dict], gap: float = GAP_SEC) -> list[dict]:
    """Attributed words -> editable units.

    A unit breaks on a speaker change, or on a pause longer than `gap` within
    one speaker. See the module docstring for why the pause break is load
    bearing rather than cosmetic.
    """
    units: list[dict] = []
    for i, w in enumerate(words):
        new = (
            not units
            or w["spk"] != units[-1]["spk"]
            or w["start"] - units[-1]["end"] > gap
        )
        if new:
            units.append({"spk": w["spk"], "start": w["start"], "end": w["end"],
                          "idx": [i]})
        else:
            units[-1]["end"] = w["end"]
            units[-1]["idx"].append(i)
    for u in units:
        u["text"] = " ".join(words[i]["w"] for i in u["idx"])
    return units


def render(units: list[dict], lo: int, hi: int, ctx: int) -> str:
    """Units [lo, hi) as prompt text, the first `ctx` of them marked context."""
    lines = []
    for i in range(lo, hi):
        u = units[i]
        text = u["text"][:UNIT_CHARS]
        mark = "  [context, do not edit]" if i < lo + ctx else ""
        lines.append(f'[{i}] {u["spk"]} ({u["start"]:.1f}-{u["end"]:.1f}s)'
                     f'{mark}: {text}')
    return "\n".join(lines)


PROMPT = """You are correcting speaker labels on an automatic transcript of a \
conversation in {lang}. The transcript came from a speech recogniser with a high \
error rate, so the words are unreliable -- judge by conversational structure, \
not by whether the text reads correctly.

The speakers present are: {speakers}

Each line is one unit: [index] speaker (start-end): text

{units}

Some units carry the wrong speaker. The three failures to look for:
- a false split: one person's continuous speech broken across two labels
- a false merge: one unit's label covering what are really two people
- a swap: two speakers' labels exchanged across a boundary

Signals that survive a bad transcript: a question and its answer are different \
speakers; a sentence continuing mid-clause across a label change is one speaker; \
a very short unit inside a long stretch of one speaker is usually that speaker.

Reply with JSON only, no other text:
{{"edits": [{{"unit": <index>, "speaker": "<one of the speakers above>", \
"confidence": <0.0 to 1.0>, "why": "<a few words>"}}]}}

Be conservative. Propose an edit only where the conversational structure makes \
the current label clearly wrong, and set confidence honestly -- below {minconf} \
if you are unsure, and the edit will be discarded rather than applied. Leaving a \
label alone costs nothing; a wrong change makes the transcript worse. If the \
labels look right, reply {{"edits": []}}.
"""


def build_prompt(units, lo, hi, ctx, speakers, lang, min_conf=MIN_CONF) -> str:
    return PROMPT.format(
        lang=LANG_NAME.get(lang or "", "an Indic language"),
        speakers=", ".join(speakers),
        units=render(units, lo, hi, ctx),
        minconf=f"{min_conf:.2f}",
    )


# --------------------------------------------------------------------------
# parsing -- every failure mode here is a silent no-op, never a crash
# --------------------------------------------------------------------------

def extract_json(text: str) -> dict | None:
    """First JSON object in a model reply, fenced or bare."""
    fence = re.search(r"```(?:json)?\s*(.+?)```", text, re.S)
    if fence:
        text = fence.group(1)
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    obj = json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
                return obj if isinstance(obj, dict) else None

    # Unbalanced: the reply hit max_new_tokens mid-array. Every edit before the
    # cut is still well-formed and independently valid, so salvage them rather
    # than discarding the whole window -- otherwise the model's willingness to
    # explain itself is what loses its edits, which is a silly failure mode.
    tail = text.rfind("}")
    if tail > start:
        try:
            obj = json.loads(text[start:tail + 1] + "]}")
        except json.JSONDecodeError:
            return None
        return obj if isinstance(obj, dict) else None
    return None


REJECTIONS = ("parse_fail", "bad_unit", "bad_speaker", "dup",
              "low_conf", "no_conf", "oom")


def parse_edits(reply: str, editable: range, speakers: set[str],
                min_conf: float = MIN_CONF) -> tuple[list, dict]:
    """Model reply -> [(unit, speaker, conf)], plus every way it was rejected.

    Confidence is required, not optional. A missing or unparseable score is
    counted as `no_conf` and the edit is dropped: treating it as high confidence
    would let a model that ignores the schema bypass the threshold entirely,
    which is precisely the model whose edits are least worth trusting. If
    `no_conf` dominates a run, the model is not following the format and the
    summary says so out loud rather than reporting a quiet zero.
    """
    bad = {k: 0 for k in REJECTIONS}
    obj = extract_json(reply)
    if obj is None or not isinstance(obj.get("edits"), list):
        bad["parse_fail"] = 1
        return [], bad

    seen: set[int] = set()
    out: list[tuple[int, str, float]] = []
    for e in obj["edits"]:
        if not isinstance(e, dict):
            bad["bad_unit"] += 1
            continue
        u, s = e.get("unit"), e.get("speaker")
        if not isinstance(u, int) or u not in editable:
            bad["bad_unit"] += 1
            continue
        if s not in speakers:
            # The model inventing a speaker is the failure that would quietly
            # wreck cpWER, so it is counted separately from a bad index.
            bad["bad_speaker"] += 1
            continue
        try:
            conf = float(e["confidence"])
        except (KeyError, TypeError, ValueError):
            bad["no_conf"] += 1
            continue
        if conf < min_conf:
            bad["low_conf"] += 1
            continue
        if u in seen:
            bad["dup"] += 1
            continue
        seen.add(u)
        out.append((u, s, conf))
    return out, bad


# --------------------------------------------------------------------------
# methods
# --------------------------------------------------------------------------

def method_rule(units, speakers, lang, llm, min_conf):
    """Short units go to the neighbour they are closer to in time.

    No model, no GPU, no confidence -- a rule that is certain by construction
    reports 1.0 so the downstream shape matches the LLM's.
    """
    edits = []
    for i, u in enumerate(units):
        if u["end"] - u["start"] >= MIN_UNIT_SEC:
            continue
        prev_u, next_u = (units[i - 1] if i else None,
                          units[i + 1] if i + 1 < len(units) else None)
        cands = []
        if prev_u:
            cands.append((u["start"] - prev_u["end"], prev_u["spk"]))
        if next_u:
            cands.append((next_u["start"] - u["end"], next_u["spk"]))
        cands = [c for c in cands if c[1] != u["spk"]]
        if cands:
            edits.append((i, min(cands)[1], 1.0))
    return edits, {k: 0 for k in REJECTIONS}


class LLM:
    def __init__(self, model_name: str, device: str = "cuda"):
        import os

        # Long Indic prompts produce large, variable attention buffers, and a
        # fragmented allocator fails to serve them even when the free total is
        # sufficient. Must be set before the first CUDA allocation.
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF",
                              "expandable_segments:True")

        import torch
        from transformers import (AutoModelForCausalLM, AutoTokenizer,
                                  BitsAndBytesConfig)

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=quant, device_map=device,
            torch_dtype=torch.float16,
        )
        self.model.eval()
        print(f"[env ] {model_name} (4-bit nf4) on {device}")

    def free(self) -> None:
        """Drop cached blocks after an OOM, keeping the model loaded."""
        import gc

        import torch

        gc.collect()
        torch.cuda.empty_cache()

    def close(self) -> None:
        """Release the weights.

        A notebook cell that loads the model holds ~8.8 GiB for the life of the
        kernel, and the next `!python stage5_correct.py` is a SEPARATE process
        that then OOMs on a 14.6 GiB T4. Any in-process use must call this.
        """
        import gc

        import torch

        self.model = None
        self.tok = None
        gc.collect()
        torch.cuda.empty_cache()

    def __call__(self, prompt: str, max_new_tokens: int = 900) -> str:
        import torch

        msgs = [{"role": "user", "content": prompt}]
        text = self.tok.apply_chat_template(msgs, tokenize=False,
                                            add_generation_prompt=True)
        enc = self.tok(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy: a benchmark must reproduce
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=self.tok.eos_token_id,
            )
        return self.tok.decode(out[0][enc["input_ids"].shape[1]:],
                               skip_special_tokens=True)


def _is_oom(exc: BaseException) -> bool:
    return "out of memory" in str(exc).lower()


def method_llm(units, speakers, lang, llm, min_conf):
    """Windowed pass over one clip's units.

    A window that will not fit in VRAM is halved and retried rather than
    allowed to kill the clip. Indic scripts tokenize far worse than Latin --
    often near one token per character -- so WINDOW units of text is a much
    bigger prompt here than the unit count suggests, and the long clips are
    exactly the ones worth correcting. Each OOM is counted, so a run that only
    survived by shrinking says so instead of looking clean.
    """
    edits: list[tuple[int, str, float]] = []
    bad = {k: 0 for k in REJECTIONS}
    lo = 0
    while lo < len(units):
        ctx = CONTEXT if lo else 0
        span = min(WINDOW, len(units) - lo)
        reply = None
        while True:
            try:
                reply = llm(build_prompt(units, lo, lo + span, ctx,
                                         speakers, lang, min_conf))
                break
            except Exception as exc:                       # noqa: BLE001
                if not _is_oom(exc):
                    raise
                bad["oom"] += 1
                llm.free()
                if span <= ctx + 2:
                    # Cannot shrink further. Skip the window rather than the
                    # clip: the units in it keep their baseline labels.
                    break
                span = max(ctx + 2, span // 2)

        hi = lo + span
        if reply is not None:
            got, b = parse_edits(reply, range(lo + ctx, hi), set(speakers),
                                 min_conf)
            edits += got
            for k in bad:
                bad[k] += b[k]
        if hi >= len(units):
            break
        lo = hi - CONTEXT
    return edits, bad


# Every method takes (units, speakers, lang, llm, min_conf) and returns
# ([(unit, speaker, confidence)], rejection counters). Adding the acoustic
# variant later means adding one function and one entry here -- nothing in
# run(), the manifest, or Stage 4c needs to know about it.
METHODS = {
    "rule": method_rule,
    "llm": method_llm,
}
NEEDS_GPU = {"llm"}


# --------------------------------------------------------------------------
# apply
# --------------------------------------------------------------------------

def apply_edits(words: list[dict], units: list[dict],
                edits: list[tuple[int, str, float]]) -> list[dict]:
    """New word list with relabelled units. Text is untouched, by construction."""
    out = [dict(w) for w in words]
    for u_i, spk, conf in edits:
        for w_i in units[u_i]["idx"]:
            out[w_i]["spk"] = spk
            out[w_i]["relabelled"] = True
            out[w_i]["conf"] = conf
    return out


def check_text_unchanged(before: list[dict], after: list[dict]) -> None:
    """The invariant that makes WER a tripwire rather than a hope."""
    if len(before) != len(after):
        raise AssertionError(f"word count changed: {len(before)} -> {len(after)}")
    for a, b in zip(before, after):
        if a["w"] != b["w"] or a["start"] != b["start"] or a["end"] != b["end"]:
            raise AssertionError(f"word altered: {a!r} -> {b!r}")


# --------------------------------------------------------------------------
# false-merge audit -- DIAGNOSTIC ONLY, never touches the correction path
# --------------------------------------------------------------------------

def audit_clip(words, ref_turns, gap: float):
    """How much of this clip's false merging is reachable by relabel at all?

    A unit whose words belong to two different REFERENCE speakers cannot be
    fixed by relabelling it -- whichever label it gets, half its words are
    wrong. Splitting units at pauses is what rescues some of them, and this
    measures exactly how many:

      merged_words_maximal  words in a unit spanning >1 true speaker, with NO
                            pause splitting
      merged_words_split    the same after splitting at `gap` -- the residue
                            relabel-only correction can never reach
      freed                 the difference: words that splitting released into
                            separately-labellable units

    Measured in WORDS, and that is not a detail. Splitting a multi-speaker unit
    often yields two units that are each still multi-speaker, so the unit count
    can RISE while the situation improves: the two segmentations have different
    denominators and their unit counts are not comparable. Words are conserved
    under splitting, so they are.

    This reads the reference RTTM and is therefore an ORACLE measurement. It
    exists so the writeup can state the ceiling on relabel-only correction
    instead of implying there is none. Nothing it computes is fed to any model
    or used to choose an edit.
    """
    from stage4_attribute import attribute_word

    starts = [t[0] for t in ref_turns]
    max_dur = max((e - s for s, e, _ in ref_turns), default=0.0)
    truth = []
    for w in words:
        spk, _ov, _spans = attribute_word(w["start"], w["end"], ref_turns,
                                          starts, max_dur)
        truth.append(spk)

    out = {}
    for name, g in (("maximal", float("inf")), ("split", gap)):
        units = build_units(words, g)
        n_multi = n_words = 0
        for u in units:
            spks = {truth[i] for i in u["idx"] if truth[i] is not None}
            if len(spks) > 1:
                n_multi += 1
                n_words += len(u["idx"])
        out[f"merged_{name}"] = n_multi
        out[f"merged_words_{name}"] = n_words
        out[f"units_{name}"] = len(units)
    out["n_words"] = len(words)
    return out


def run_audit(cond: str, data: Path, gap: float, limit: int | None) -> bool:
    from stage4_attribute import load_turns

    src_root = data / "attrib" / cond
    ref_dir = data / "ref" / "rttm"
    if not src_root.is_dir():
        print(f"[skip] no condition at {src_root}")
        return False
    if not ref_dir.is_dir():
        print(f"[skip] no reference RTTMs at {ref_dir} -- the audit is scored "
              f"against ground truth and cannot run without it")
        return False

    clips = sorted(p.stem for p in src_root.glob("*.json"))
    clips = clips[:limit] if limit else clips
    tot = {}
    n = 0
    for clip_id in clips:
        turns = load_turns(ref_dir / f"{clip_id}.rttm")
        if not turns:
            continue
        src = json.loads((src_root / f"{clip_id}.json").read_text(encoding="utf-8"))
        a = audit_clip(src["words"], turns, gap)
        for k, v in a.items():
            tot[k] = tot.get(k, 0) + v
        n += 1

    if not n:
        print(f"[audit] {cond}: nothing to audit")
        return False

    wm, ws = tot["merged_words_maximal"], tot["merged_words_split"]
    freed = wm - ws
    pct = 100.0 * freed / max(wm, 1)
    nw = max(tot["n_words"], 1)
    print(f"\n[audit] {cond}  ({n} clips)  ORACLE DIAGNOSTIC, not a system result")
    print(f"  units    : {tot['units_maximal']:,} unsplit -> "
          f"{tot['units_split']:,} split at {gap:.1f}s")
    print(f"  words trapped in a unit spanning >1 true speaker:")
    print(f"    unsplit: {wm:>7,}  ({100.0 * wm / nw:5.2f}% of all words)")
    print(f"    split  : {ws:>7,}  ({100.0 * ws / nw:5.2f}% of all words)")
    print(f"    freed  : {freed:>7,}  ({pct:.1f}% of the unsplit total)")
    print(f"  CEILING  : {100.0 * ws / nw:.2f}% of words sit in a unit that no "
          f"relabel can fix")
    print(f"  (measured in words: unit counts are NOT comparable across the two "
          f"segmentations, since splitting a multi-speaker unit can yield two "
          f"of them)")
    return True


# --------------------------------------------------------------------------
# run
# --------------------------------------------------------------------------

def run(cond: str, method: str, data: Path, model_name: str,
        limit: int | None, min_conf: float = MIN_CONF,
        shard: tuple[int, int] | None = None) -> bool:
    src_root = data / "attrib" / cond
    if not src_root.is_dir():
        print(f"[skip] no condition at {src_root}")
        return False
    if "+" in cond:
        print(f"[skip] {cond} is already a Stage 5 output; correcting a "
              f"correction is not a condition anyone can interpret")
        return False

    out_root = data / "attrib" / f"{cond}+{method}"
    manifest = Manifest(out_root / "manifest.jsonl")
    clips = sorted(p.stem for p in src_root.glob("*.json"))
    if shard:
        # Sharded BEFORE the done-filter, so each worker owns a fixed set of
        # clips no matter when it starts or how far the others have got. Two
        # workers must never be handed the same clip: they would race on the
        # same output path and double-count in the manifest.
        i, n_sh = shard
        clips = [c for k, c in enumerate(clips) if k % n_sh == i]
    pending = [c for c in clips if not manifest.done(c)]
    todo = pending[:limit] if limit else pending

    tag = f"{cond}+{method}" + (f"  [shard {shard[0]}/{shard[1]}]" if shard else "")
    print(f"\n[cond] {cond} -> {tag}")
    print(f"[plan] {len(clips)} clips: {len(clips) - len(pending)} done, "
          f"{len(pending)} pending, running {len(todo)} now")
    if not todo:
        return True

    llm = LLM(model_name) if method in NEEDS_GPU else None
    fn = METHODS[method]

    n_ok = n_fail = 0
    tot = {"units": 0, "proposed": 0, "applied": 0, "rogue": 0,
           **{k: 0 for k in REJECTIONS}}
    for n, clip_id in enumerate(todo, 1):
        try:
            src = json.loads((src_root / f"{clip_id}.json").read_text(encoding="utf-8"))
            words = src["words"]
            units = build_units(words)
            speakers = sorted({u["spk"] for u in units})

            edits, bad = fn(units, speakers, src.get("lang"), llm, min_conf)

            proposed = len(edits)
            # The guard exists to catch a MODEL that has stopped following
            # instructions. A deterministic rule cannot do that, and applying
            # it there silently cripples the baseline on exactly the noisiest
            # conditions -- 30 of 99 sortformer_stream clips were being dropped,
            # which is the comparison the rule is supposed to provide.
            rogue = (method in NEEDS_GPU
                     and proposed > MIN_ROGUE_EDITS
                     and proposed > MAX_EDIT_FRAC * len(units))
            if rogue:
                # Not a correction pass any more. Keep the baseline labels and
                # say so; a clip silently rewritten would be indistinguishable
                # from one the model genuinely improved.
                edits = []

            new_words = apply_edits(words, units, edits)
            check_text_unchanged(words, new_words)

            rec = dict(src)
            rec["words"] = new_words
            rec["stage5"] = {
                "method": method,
                "model": model_name if method in NEEDS_GPU else None,
                "min_conf": min_conf,
                "n_units": len(units),
                "n_proposed": proposed,
                "n_applied": len(edits),
                "n_words_relabelled": sum(1 for w in new_words if w.get("relabelled")),
                "rogue": rogue,
                **bad,
            }
            write_json(out_root / f"{clip_id}.json", rec)

            tot["units"] += len(units)
            tot["proposed"] += proposed
            tot["applied"] += len(edits)
            tot["rogue"] += int(rogue)
            for k in REJECTIONS:
                tot[k] += bad[k]

            manifest.append({"clip_id": clip_id, "status": "ok",
                             **rec["stage5"]})
            n_ok += 1
            print(f"[{n:3d}/{len(todo)}] {clip_id:44s} ok  "
                  f"{len(units):4d} units  {len(edits):3d} edits"
                  f"{'  ROGUE' if rogue else ''}")
        except Exception as exc:                       # noqa: BLE001
            n_fail += 1
            manifest.append({"clip_id": clip_id, "status": "fail",
                             "error": f"{type(exc).__name__}: {exc}"})
            print(f"[{n:3d}/{len(todo)}] {clip_id:44s} FAIL  "
                  f"{type(exc).__name__}: {exc}")

    pct = 100.0 * tot["applied"] / max(tot["units"], 1)
    print(f"\n[done] ok={n_ok} fail={n_fail}  {tot['units']:,} units, "
          f"{tot['applied']:,} relabelled ({pct:.2f}%), "
          f"{tot['rogue']} clips rogue")
    if method in NEEDS_GPU:
        print(f"[done] abstained: {tot['low_conf']} below conf {min_conf}, "
              f"{tot['no_conf']} with no confidence given")
        if tot["oom"]:
            print(f"[done] {tot['oom']} windows hit OOM and were retried at half "
                  f"width -- results are still valid, but the GPU is the "
                  f"binding constraint on the long clips")
        print(f"[done] rejected: {tot['parse_fail']} unparseable replies, "
              f"{tot['bad_unit']} bad index, {tot['bad_speaker']} invented "
              f"speaker, {tot['dup']} duplicate")
        offered = tot["applied"] + tot["low_conf"] + tot["no_conf"]
        if tot["no_conf"] > max(tot["applied"], 5):
            # Not a conservative model -- a model ignoring the schema. Said
            # loudly, because the run otherwise looks like a clean abstention.
            print("[WARN] most edits arrived without a confidence score, so "
                  "they were dropped. The model is not following the reply "
                  "format; fix the prompt before reading anything into this "
                  "run's WDER.")
        elif offered and tot["applied"] == 0:
            print("[WARN] every proposed edit was abstained away. This scores "
                  "identically to the baseline by construction -- lower "
                  "--min-conf or accept that the model has no usable signal.")
    return n_fail == 0


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    ap.add_argument("--cond", nargs="+", required=True,
                    help="Stage 4b condition(s), e.g. indicconformer__pyannote31")
    ap.add_argument("--method", choices=tuple(METHODS), default="llm")
    ap.add_argument("--model", default=MODEL)
    ap.add_argument("--data", type=Path, default=Path("data"))
    ap.add_argument("--limit", type=int)
    ap.add_argument("--shard", metavar="I/N",
                    help="process only clips where index %% N == I. Run one "
                         "worker per GPU with CUDA_VISIBLE_DEVICES to halve "
                         "wall-clock: the shards are disjoint, so the two "
                         "workers never touch the same output file.")
    ap.add_argument("--min-conf", type=float, default=MIN_CONF,
                    help=f"discard edits below this confidence "
                         f"(default {MIN_CONF})")
    ap.add_argument("--audit", action="store_true",
                    help="ORACLE DIAGNOSTIC: report how much false merging "
                         "pause splitting recovers, and how much no relabel "
                         "can reach. Reads the reference RTTM, writes no "
                         "condition, and never influences an edit.")
    args = ap.parse_args()

    shard = None
    if args.shard:
        i, n = (int(x) for x in args.shard.split("/"))
        if not 0 <= i < n:
            raise SystemExit(f"--shard {args.shard}: need 0 <= I < N")
        shard = (i, n)

    clean = True
    for c in args.cond:
        if c.endswith("__ref"):
            # Correcting oracle labels would improve a diagnostic that is
            # perfect by construction. It cannot mean anything.
            print(f"[skip] {c} is the oracle condition")
            continue
        if args.audit:
            clean &= run_audit(c, args.data, GAP_SEC, args.limit)
        else:
            clean &= run(c, args.method, args.data, args.model, args.limit,
                         args.min_conf, shard)
    return 0 if clean else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
BASE = [f"{a}__{d}" for a in ASR for d in DIAR]
run("stage 5: relabel audit (oracle)", ["stage5_correct.py", "--cond", "indicconformer__pyannote31",
                                        "indicconformer__sortformer_stream", "--audit", "--data", "data"])
run("stage 5: rule relabel", ["stage5_correct.py", "--method", "rule", "--data", "data", "--cond", *BASE])

In [ ]:
LLM_COND = "indicconformer__pyannote31"
label = "stage 5: LLM relabel"
if not HAVE_GPU:
    skip(label, "no GPU")
elif (LLM_PY := gpu_env("llm", "bitsandbytes", "accelerate")) is None:
    skip(label, "environment install failed")
else:
    if not FULL_GPU_RUN:
        # The smoke pass reads the attribution just computed, in its own root.
        shutil.copytree(DATA / "attrib" / LLM_COND, WORKDIR / "smoke" / "attrib" / LLM_COND,
                        dirs_exist_ok=True, ignore=shutil.ignore_patterns("manifest.jsonl"))
    run(label, ["stage5_correct.py", "--cond", LLM_COND, "--method", "llm",
                "--data", GPU_ROOT] + GPU_LIMIT, python=LLM_PY, may_fail=GPU_MAY_FAIL)

if not FULL_GPU_RUN:
    live_dir = WORKDIR / "smoke" / "attrib" / f"{LLM_COND}+llm"
    for f in sorted(live_dir.glob("*.json")) if live_dir.is_dir() else []:
        live = json.loads(f.read_text(encoding="utf-8"))
        ref = json.loads((CACHE / "attrib" / f"{LLM_COND}+llm" / f.name).read_text(encoding="utf-8"))
        same = sum(a["spk"] == b["spk"] for a, b in zip(live["words"], ref["words"]))
        print(f"{f.stem}  {live['stage5']['n_applied']} edits live vs {ref['stage5']['n_applied']} "
              f"in the reference run; {100 * same / len(ref['words']):.1f}% of word labels identical")
    restore(f"attrib/{LLM_COND}+llm")

## Stage 4c — cpWER, WDER (and WER, DI-cpWER)

- **WER** ignores speakers, so it must be identical across every diarizer and every
  relabel of one ASR system — the tripwire for a leak between stages.
- **cpWER** (MeetEval): the best speaker permutation, then WER.
- **WDER**: hand-rolled after Shafey et al., since MeetEval has none.
- Corpus rates are **error-weighted** (total errors / total reference words).

In [ ]:
%%writefile stage4_score.py
#!/usr/bin/env python3
"""
Stage 4c -- ASR + attribution scoring (CPU-only).

Scores every (asr, diar) condition from Stage 4b against the Stage 2 reference
transcripts and writes:

    data/results/asr_per_clip.csv
    data/results/asr_summary.csv
    data/results/asr_summary.md

Four metrics, chosen so that the errors decompose rather than pile into one
number:

  WER       speaker-agnostic, reference text vs hypothesis text. Pure ASR
            quality; identical for every diar condition of the same ASR, which
            is a useful invariant to eyeball -- if it moves, something leaked.
  cpWER     permutation-optimal per-speaker WER. The headline number.
  DI-cpWER  diarization-invariant cpWER. meeteval 0.4.3 ships only the greedy
            approximation, whose error count is an upper bound on the optimal,
            so `cpWER - DI-cpWER` is a LOWER bound on the attribution cost.
  WDER      word diarization error rate, hand-rolled: meeteval has no WDER.

  cpWER - DI-cpWER  isolates what wrong speaker attribution cost, which is
                    exactly the quantity Stage 5 is trying to reduce.

CORPUS NUMBERS ARE ERROR-WEIGHTED: sum(errors) / sum(reference words), not the
mean of per-clip rates. The unweighted mean is reported beside it because it is
the one people publish by accident -- it lets a 50 s clip outweigh a 30 min one.

TWO SUBSETS, ALWAYS BOTH. `sortformer` has 74 of 99 clips and the 25 it is
missing are all long ones, with more speakers and more turn changes than
average. A corpus number over 74 clips is not comparable to one over 99, so
every table reports `all` (each condition over what it has) and `common` (the
clips every condition produced), and only `common` is a fair cross-system
comparison.

    python stage4_score.py --data data

Normalisation is defined in one place, `normalise()`, and applied identically to
reference and hypothesis. WER is extremely sensitive to it, so it is a stated
policy rather than an accident: Unicode NFC, bracketed annotations removed,
punctuation stripped (including the Devanagari danda), case folded, whitespace
collapsed.
"""

from __future__ import annotations

import argparse
import json
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

try:
    from rapidfuzz.distance import Levenshtein
except ImportError:  # checked in main(), so the message arrives in second one
    Levenshtein = None

ORACLE = "ref"

# Bracketed annotations: "(inaudible)", "[laughs]", "<laughter>". Reference-only
# conventions that the ASR cannot produce, so scoring them would be scoring the
# annotator. The angle form matters: PUNCT strips "<" and ">" as punctuation, so
# without this the tag survives as a bare word. 666 of them across the corpus --
# <unintelligible> 263, <noise> 163, <laughter> 146, <vocalization> 81,
# <background_speech> 12, <uhhh> 1 -- each an unmatchable reference token.
BRACKETS = re.compile(r"\([^)]*\)|\[[^\]]*\]|\{[^}]*\}|<[^>]*>")

# Punctuation across the scripts in this corpus, plus the Devanagari danda and
# its double form, which are sentence terminators rather than words.
PUNCT = re.compile(r"[!-/:-@\[-`{-~।॥‐-‧‰-⁞¡-¿]")


def normalise(text: str) -> list[str]:
    """Text -> comparable word list. The single definition of 'a word' here."""
    text = unicodedata.normalize("NFC", text)
    text = BRACKETS.sub(" ", text)
    text = PUNCT.sub(" ", text)
    return text.casefold().split()


# --------------------------------------------------------------------------
# meeteval, resolved defensively
# --------------------------------------------------------------------------

def resolve_meeteval():
    """Return (cpwer_fn, di_cpwer_fn, siso_fn).

    meeteval has moved these between `meeteval.wer` and `meeteval.wer.wer.*`
    across releases, and DI-cpWER is recent. Rather than fail with an
    AttributeError three hours into a session, try the known spellings and, if
    none match, say exactly what the installed version does expose.
    """
    try:
        import meeteval.wer as mw
    except ImportError:
        raise SystemExit("meeteval is not installed:  pip install meeteval")

    def pick(*names):
        for n in names:
            fn = getattr(mw, n, None)
            if fn is not None:
                return fn
        return None

    cp = pick("cp_word_error_rate", "cpwer")
    siso = pick("siso_word_error_rate", "wer", "word_error_rate")
    # 0.4.3 ships DI-cpWER only in the greedy form. Greedy assigns words without
    # searching every permutation, so its error count is an upper bound on the
    # optimal DI-cpWER -- which makes `cpWER - DI-cpWER` a LOWER bound on the
    # cost of wrong attribution: the real cost is at least this, never less.
    # Stated because Stage 5's entire claim is a reduction in that quantity.
    di = pick("di_cp_word_error_rate", "dicpwer", "di_cpwer",
              "greedy_di_cp_word_error_rate", "greedy_dicpwer")

    missing = [n for n, f in (("cpWER", cp), ("DI-cpWER", di), ("WER", siso)) if f is None]
    if missing:
        avail = sorted(n for n in dir(mw) if "error_rate" in n or n.endswith("wer"))
        raise SystemExit(f"meeteval is missing {missing}. Installed version exposes: {avail}")

    di_name = getattr(di, "__name__", "?")
    print(f"[env ] meeteval bindings: cpWER={getattr(cp, '__name__', '?')}, "
          f"DI-cpWER={di_name}, WER={getattr(siso, '__name__', '?')}")
    if "greedy" in di_name:
        print("[env ] DI-cpWER is the greedy approximation, so attribution_cost "
              "(cpWER - DI-cpWER) is a LOWER bound on the true attribution cost.")
    return cp, di, siso


# meeteval's entry points do not all take the same input shape: cpWER accepts a
# {speaker: text} mapping, while greedy DI-cpWER wants SegLST (a list of segment
# dicts) and raises TypeError on a mapping. Rather than hard-code which is which
# -- it has changed between releases -- try the mapping, fall back to SegLST, and
# remember the answer per function so the probe costs one call, not 99.
_CALL_STYLE: dict[str, str] = {}


def _seglst(mapping: dict[str, str], clip_id: str) -> list[dict]:
    return [{"session_id": clip_id, "speaker": spk, "words": text}
            for spk, text in mapping.items()]


def call_wer(fn, ref: dict[str, str], hyp: dict[str, str], clip_id: str):
    name = getattr(fn, "__name__", repr(fn))
    style = _CALL_STYLE.get(name)

    if style in (None, "mapping"):
        try:
            out = fn(ref, hyp)
            _CALL_STYLE[name] = "mapping"
            return out
        except (TypeError, AttributeError, KeyError):
            if style == "mapping":
                raise

    out = fn(_seglst(ref, clip_id), _seglst(hyp, clip_id))
    if _CALL_STYLE.get(name) != "seglst":
        _CALL_STYLE[name] = "seglst"
        print(f"[env ] {name} takes SegLST, not a speaker mapping")
    return out


def rate(err) -> tuple[int, int]:
    """meeteval ErrorRate -> (errors, reference length), version-tolerantly."""
    errors = getattr(err, "errors", None)
    length = getattr(err, "length", None)
    if errors is None:  # older releases expose only the ratio
        errors, length = round(err.error_rate * err.length), err.length
    return int(errors), int(length)


# --------------------------------------------------------------------------
# WDER -- hand-rolled, because meeteval has no WDER
# --------------------------------------------------------------------------

def align_pairs(ref_words: list[str], hyp_words: list[str]):
    """Yield (ref_index, hyp_index) for every correct or substituted word.

    Insertions and deletions are skipped: WDER is defined over words that exist
    on both sides, since a word the ASR never produced has no speaker to be
    wrong about. Uses rapidfuzz's Levenshtein opcodes (C++); a pure-Python DP
    would be minutes per clip at 3000 words a side.
    """
    for op in Levenshtein.opcodes(ref_words, hyp_words):
        if op.tag == "equal":
            for k in range(op.src_end - op.src_start):
                yield op.src_start + k, op.dest_start + k
        elif op.tag == "replace":
            # Pair positionally; the ragged tail is insertions or deletions.
            for k in range(min(op.src_end - op.src_start, op.dest_end - op.dest_start)):
                yield op.src_start + k, op.dest_start + k


def speaker_mapping(ref_by_spk: dict[str, list[str]],
                    hyp_by_spk: dict[str, list[str]],
                    cp_result) -> dict[str, str]:
    """hyp speaker -> ref speaker, under cpWER's optimal permutation.

    Prefers the assignment meeteval already computed, so WDER and cpWER agree on
    who is who. Falls back to a Hungarian match on shared word counts if the
    installed version does not expose it -- reported when it happens, because a
    different permutation makes the two metrics tell slightly different stories.
    """
    assignment = getattr(cp_result, "assignment", None)
    if assignment:
        out = {}
        for pair in assignment:
            # meeteval orders these (reference, hypothesis).
            r, h = pair[0], pair[1]
            if h is not None and r is not None:
                out[str(h)] = str(r)
        if out:
            return out

    from collections import Counter

    import numpy as np
    from scipy.optimize import linear_sum_assignment

    refs, hyps = sorted(ref_by_spk), sorted(hyp_by_spk)
    gain = np.zeros((len(refs), len(hyps)))
    for i, r in enumerate(refs):
        rc = Counter(ref_by_spk[r])
        for j, h in enumerate(hyps):
            hc = Counter(hyp_by_spk[h])
            gain[i, j] = sum((rc & hc).values())
    ri, hj = linear_sum_assignment(-gain)
    return {hyps[j]: refs[i] for i, j in zip(ri, hj)}


def wder(ref_seq, hyp_seq, mapping: dict[str, str]) -> tuple[int, int]:
    """(mis-attributed words, scorable words).

    WDER = (S_IS + C_IS) / (S + C): among words that align -- correct or
    substituted -- the fraction whose speaker is wrong. A hypothesis speaker
    with no counterpart in the mapping counts as wrong, which is the honest
    reading: the word was attributed to somebody who does not exist.
    """
    ref_words = [w for w, _ in ref_seq]
    hyp_words = [w for w, _ in hyp_seq]
    wrong = total = 0
    for i, j in align_pairs(ref_words, hyp_words):
        total += 1
        if mapping.get(hyp_seq[j][1]) != ref_seq[i][1]:
            wrong += 1
    return wrong, total


# --------------------------------------------------------------------------
# loading
# --------------------------------------------------------------------------

def load_reference(path: Path):
    """ref/segments/<clip>.json -> (by_speaker words, time-ordered (word, spk))."""
    d = json.loads(path.read_text(encoding="utf-8"))
    by_spk: dict[str, list[str]] = {}
    seq: list[tuple[str, str]] = []
    for seg in sorted(d["segments"], key=lambda s: (s["start"], s["index"])):
        spk = str(seg["speaker"])
        words = normalise(seg["text"])
        by_spk.setdefault(spk, []).extend(words)
        seq.extend((w, spk) for w in words)
    return by_spk, seq


def load_hypothesis(path: Path):
    """attrib/<cond>/<clip>.json -> (by_speaker words, time-ordered (word, spk))."""
    d = json.loads(path.read_text(encoding="utf-8"))
    by_spk: dict[str, list[str]] = {}
    seq: list[tuple[str, str]] = []
    for w in d["words"]:
        spk = str(w["spk"])
        for token in normalise(w["w"]):  # a "word" may normalise to 0 or 2
            by_spk.setdefault(spk, []).append(token)
            seq.append((token, spk))
    return by_spk, seq, d


# --------------------------------------------------------------------------

def score_clip(ref_by_spk, ref_seq, hyp_by_spk, hyp_seq, fns, clip_id="clip") -> dict:
    cp_fn, di_fn, siso_fn = fns

    ref_txt = {k: " ".join(v) for k, v in ref_by_spk.items()}
    hyp_txt = {k: " ".join(v) for k, v in hyp_by_spk.items()} or {"spk0": ""}

    cp = call_wer(cp_fn, ref_txt, hyp_txt, clip_id)
    di = call_wer(di_fn, ref_txt, hyp_txt, clip_id)
    siso = siso_fn(" ".join(w for w, _ in ref_seq), " ".join(w for w, _ in hyp_seq))

    cp_e, cp_n = rate(cp)
    di_e, di_n = rate(di)
    si_e, si_n = rate(siso)

    mapping = speaker_mapping(ref_by_spk, hyp_by_spk, cp)
    wd_e, wd_n = wder(ref_seq, hyp_seq, mapping)

    return {"cp_errors": cp_e, "cp_len": cp_n,
            "di_errors": di_e, "di_len": di_n,
            "wer_errors": si_e, "wer_len": si_n,
            "wder_errors": wd_e, "wder_len": wd_n,
            "ref_words": len(ref_seq), "hyp_words": len(hyp_seq),
            "ref_spk": len(ref_by_spk), "hyp_spk": len(hyp_by_spk)}


def aggregate(df: pd.DataFrame, label: str) -> dict:
    def ew(e, n):  # error-weighted
        return round(100 * df[e].sum() / max(df[n].sum(), 1), 2)

    def mean(e, n):  # unweighted mean of per-clip rates
        return round(100 * (df[e] / df[n].clip(lower=1)).mean(), 2)

    cp, di = ew("cp_errors", "cp_len"), ew("di_errors", "di_len")
    return {"subset": label, "clips": len(df),
            "WER": ew("wer_errors", "wer_len"),
            "cpWER": cp, "DI_cpWER": di,
            "attribution_cost": round(cp - di, 2),
            "WDER": ew("wder_errors", "wder_len"),
            "cpWER_unweighted_mean": mean("cp_errors", "cp_len"),
            "ref_words": int(df["ref_words"].sum())}


def _as_markdown(df: pd.DataFrame) -> str:
    """to_markdown needs `tabulate`, which is not always installed. A missing
    optional dependency must not discard a completed scoring run."""
    try:
        return df.to_markdown(index=False)
    except ImportError:
        fence = "```"
        return fence + "\n" + df.to_string(index=False) + "\n" + fence


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default="data")
    ap.add_argument("--limit", type=int, default=None, help="smoke test: first N clips")
    args = ap.parse_args()

    data = Path(args.data)
    # Both dependencies are checked before a single clip is scored: discovering
    # a missing one after cpWER has run on 99 clips wastes the whole pass.
    if Levenshtein is None:
        raise SystemExit("WDER needs rapidfuzz for the word alignment:  "
                         "pip install rapidfuzz")
    fns = resolve_meeteval()
    ref_dir = data / "ref" / "segments"
    conds = sorted(p for p in (data / "attrib").glob("*") if p.is_dir())
    if not conds:
        raise SystemExit(f"no conditions under {data / 'attrib'} -- run stage4_attribute.py")

    # Stage 4b needs only ref/rttm, so a dataset can carry the RTTMs and not the
    # transcripts and everything upstream still looks healthy. Scoring needs the
    # text. Say so here rather than skipping all 99 clips and reporting the
    # uninformative "nothing scored".
    if not ref_dir.is_dir():
        raise SystemExit(
            f"no reference transcripts at {ref_dir}. Stage 4b only needed "
            f"ref/rttm/, so this is easy to miss: attach the Stage 2 dataset "
            f"that carries ref/segments/ (one JSON per clip, with the text)."
        )
    n_ref = len(list(ref_dir.glob("*.json")))
    print(f"[env ] {n_ref} reference transcripts, {len(conds)} conditions")
    if n_ref == 0:
        raise SystemExit(f"{ref_dir} exists but holds no *.json")

    rows, skipped = [], []
    for cond in conds:
        asr, diar = cond.name.split("__", 1)
        clips = sorted(p.stem for p in cond.glob("*.json"))
        clips = clips[:args.limit] if args.limit else clips
        print(f"[cond] {cond.name}: {len(clips)} clips")

        for clip_id in clips:
            ref_path = ref_dir / f"{clip_id}.json"
            if not ref_path.exists():
                skipped.append((cond.name, clip_id, "no reference"))
                continue
            ref_by_spk, ref_seq = load_reference(ref_path)
            if not ref_seq:
                # Nothing to score against: an empty reference makes WER either
                # 0/0 or infinite depending on convention, and neither is a
                # result. Excluded and listed rather than silently counted.
                skipped.append((cond.name, clip_id, "empty reference"))
                continue
            hyp_by_spk, hyp_seq, _ = load_hypothesis(cond / f"{clip_id}.json")
            r = score_clip(ref_by_spk, ref_seq, hyp_by_spk, hyp_seq, fns, clip_id)
            rows.append({"asr": asr, "diar": diar, "oracle": diar == ORACLE,
                         "clip_id": clip_id, **r})

    per_clip = pd.DataFrame(rows)
    if per_clip.empty:
        why = {}
        for _, _, reason in skipped:
            why[reason] = why.get(reason, 0) + 1
        raise SystemExit(
            f"nothing scored. Skipped {len(skipped)} (condition, clip) pairs: {why}. "
            f"'no reference' in bulk means the clip ids in data/attrib do not match "
            f"the filenames in {ref_dir}."
        )

    # The common subset: clips every condition produced. This is the only fair
    # cross-system comparison, since sortformer is missing the long clips.
    per_cond = per_clip.groupby(["asr", "diar"])["clip_id"].apply(set)
    common = set.intersection(*per_cond) if len(per_cond) else set()

    summary = []
    for (asr, diar), g in per_clip.groupby(["asr", "diar"]):
        for label, sub in (("all", g), ("common", g[g.clip_id.isin(common)])):
            if sub.empty:
                continue
            summary.append({"asr": asr,
                            "diar": diar + (" (oracle)" if diar == ORACLE else ""),
                            **aggregate(sub, label)})
    summary = pd.DataFrame(summary).sort_values(["subset", "asr", "diar"])

    # DI-cpWER relaxes cpWER's speaker constraint, so it can never be larger.
    # If it is, the resolver bound the wrong meeteval function or the inputs are
    # shaped wrongly -- either way the attribution-cost column is meaningless
    # and must not be quietly written to a results table.
    bad = summary[summary["attribution_cost"] < -0.01]
    if not bad.empty:
        print("\n[WARN] DI-cpWER exceeds cpWER. With the optimal DI-cpWER that is "
              "impossible and means the binding or the input shape is wrong; with "
              "the greedy approximation a small excess is possible on hard clips. "
              "Either way attribution_cost is not trustworthy here:",
              file=sys.stderr)
        print(bad.to_string(index=False), file=sys.stderr)

    out = data / "results"
    out.mkdir(parents=True, exist_ok=True)
    per_clip.to_csv(out / "asr_per_clip.csv", index=False, encoding="utf-8")
    summary.to_csv(out / "asr_summary.csv", index=False, encoding="utf-8")

    md = ["# Stage 4 -- ASR and attribution results", "",
          f"Common subset: {len(common)} of "
          f"{per_clip.clip_id.nunique()} clips scored by every condition.", "",
          "All rates are percentages, error-weighted "
          "(sum of errors / sum of reference words).", "",
          "`ref` is an ORACLE condition using the reference diarization. It is a "
          "diagnostic floor, never a system result.", "",
          _as_markdown(summary), ""]
    if skipped:
        md += ["## Excluded", ""] + [f"- `{c}` / `{k}`: {why}" for c, k, why in skipped] + [""]
    (out / "asr_summary.md").write_text("\n".join(md), encoding="utf-8")

    print()
    print(summary.to_string(index=False))
    if skipped:
        print(f"\nexcluded {len(skipped)} (condition, clip) pairs; see asr_summary.md")
    print(f"\nwrote {out / 'asr_summary.md'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
run("stage 4c: score ASR", ["stage4_score.py", "--data", "data"])

## Stage 5b — DER / JER for the word-level conditions

DER is measured over time, but Stage 5 and the fallback change word labels. Each
condition's word labels are projected back onto **the diarizer's own turns**
(duration-weighted majority), so boundaries never move and missed speech and false
alarm are identical to the baseline — any change is speaker confusion alone.

The projection is lossy by itself: with **zero edits** it raises pyannote's DER from
27.34 to 29.03. So the unedited conditions are projected too, and serve as the control
(`DER_ctrl`) that every corrected DER is read against.

In [ ]:
%%writefile stage5_to_rttm.py
#!/usr/bin/env python3
"""
Stage 5d -- corrected word labels back to RTTM, so DER/JER can be re-scored.

    python stage5_to_rttm.py --cond indicconformer__pyannote31+rule --data data
    python stage3_score.py --data data --systems pyannote31 pyannote31+rule

Stage 5 relabels WORDS. DER and JER are defined over time, not words, so the
corrected transcript cannot be scored against them directly -- which leaves the
"baseline vs improved DER/JER" cell of the results table empty. This closes it.


WHY RELABEL TURNS INSTEAD OF BUILDING THEM FROM WORDS
-----------------------------------------------------
The obvious approach -- emit one turn per unit, using the first and last word
times -- is wrong here, and quietly so.

A diarizer's turn spans continuous speech including the pauses inside it. Word
spans do not: they stop at the last word and resume at the next, and they omit
leading/trailing silence within a turn entirely. Turns rebuilt from words are
therefore systematically SHORTER than the turns they replace, so hypothesis
speech time drops, missed speech rises, and DER moves for a reason that has
nothing to do with Stage 5. The improvement would be measuring the conversion.

So the baseline turn boundaries are kept EXACTLY as the diarizer produced them,
and only the label changes: each turn takes the majority label of the corrected
words inside it, weighted by word duration so a long word counts for more than
a filler. Same segmentation, different names. The DER delta then isolates
speaker confusion, which is the only thing a relabel-only stage can affect --
and missed speech, false alarm and total speech time are unchanged by
construction, which the summary asserts.

A turn containing no words keeps its original label: there is no evidence to
revise it, and inventing one would be noise.
"""

from __future__ import annotations

import argparse
import bisect
import json
import sys
from collections import defaultdict
from pathlib import Path

from stage4_attribute import load_turns, rttm_dir


def relabel_turns(turns, words):
    """[(start, end, spk)] + corrected words -> [(start, end, new_spk)].

    Assignment is by duration-weighted majority of the words overlapping each
    turn. Overlap, not containment: a word straddling a boundary should vote in
    both turns it touches rather than in neither.
    """
    starts = [t[0] for t in turns]
    max_dur = max((e - s for s, e, _ in turns), default=0.0)

    votes: dict[int, dict[str, float]] = defaultdict(lambda: defaultdict(float))
    for w in words:
        # Every turn that could overlap this word: bisect to the first turn
        # starting at or after (word start - longest turn), then walk forward
        # until turns start after the word ends.
        i = bisect.bisect_left(starts, w["start"] - max_dur)
        while i < len(turns) and turns[i][0] < w["end"]:
            s, e, _ = turns[i]
            ov = min(w["end"], e) - max(w["start"], s)
            if ov > 0:
                votes[i][w["spk"]] += ov
            i += 1

    out = []
    for i, (s, e, spk) in enumerate(turns):
        v = votes.get(i)
        if v:
            # max() on (weight, speaker) so ties break on the label, not on
            # dict order -- the output has to be reproducible.
            spk = max(sorted(v.items()), key=lambda kv: kv[1])[0]
        out.append((s, e, spk))
    return out


def write_rttm(path: Path, clip_id: str, turns) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        f"SPEAKER {clip_id} 1 {s:.3f} {e - s:.3f} <NA> <NA> {spk} <NA> <NA>"
        for s, e, spk in turns if e > s
    ]
    path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def run(cond: str, data: Path, limit: int | None) -> bool:
    src_root = data / "attrib" / cond
    if not src_root.is_dir():
        print(f"[skip] no condition at {src_root}")
        return False

    # `asr__diar+method` -> the diarizer whose RTTMs supplied the boundaries.
    _asr, rest = cond.split("__", 1)
    diar = rest.split("+")[0]
    # Layout is data/hyp/<system>/rttm/<clip>.rttm -- that extra `rttm` level is
    # what Stage 3 writes and what stage3_score.py reads, so borrow Stage 4b's
    # resolver rather than rebuilding the path and getting it subtly wrong.
    base_dir = rttm_dir(data, diar)
    if not base_dir.is_dir():
        print(f"[skip] no baseline RTTMs at {base_dir}")
        return False

    # Named for the FULL condition, not just the diarizer. Three ASR systems
    # share each diarizer, and each produces a different corrected RTTM from
    # the same baseline turns -- naming these `pyannote31+rule` would have all
    # three overwrite one another, and the survivor would depend on argument
    # order. That the same diarizer repairs differently under different ASR is
    # itself a result; it needs three rows, not one.
    out_name = cond
    out_dir = data / "hyp" / out_name / "rttm"

    clips = sorted(p.stem for p in src_root.glob("*.json"))
    clips = clips[:limit] if limit else clips

    n_ok = n_skip = 0
    n_turns = n_changed = 0
    dur_total = dur_changed = 0.0
    for clip_id in clips:
        turns = load_turns(base_dir / f"{clip_id}.rttm")
        if not turns:
            n_skip += 1
            continue
        rec = json.loads((src_root / f"{clip_id}.json").read_text(encoding="utf-8"))
        new = relabel_turns(turns, rec["words"])
        write_rttm(out_dir / f"{clip_id}.rttm", clip_id, new)
        for (s, e, a), (_s, _e, b) in zip(turns, new):
            n_turns += 1
            dur_total += e - s
            if a != b:
                n_changed += 1
                dur_changed += e - s
        n_ok += 1

    print(f"\n[rttm] {cond} -> hyp/{out_name}")
    print(f"  clips     : {n_ok} written, {n_skip} skipped (no baseline RTTM)")
    if n_ok == 0:
        # A directory that exists but yields nothing is a path bug, not a
        # result. Fail loudly rather than reporting a tidy row of zeroes.
        print(f"  [!] nothing written -- {base_dir} exists but held no *.rttm "
              f"matching these clip ids. Check the path, not the data.")
        return False
    print(f"  turns     : {n_turns:,}, {n_changed:,} relabelled "
          f"({100.0 * n_changed / max(n_turns, 1):.2f}%)")
    print(f"  speech    : {dur_total / 3600:.2f} h, {dur_changed / 3600:.2f} h "
          f"relabelled ({100.0 * dur_changed / max(dur_total, 1e-9):.2f}%)")
    print(f"  boundaries are unchanged, so missed speech, false alarm and total "
          f"speech time are identical to `{diar}` -- any DER/JER delta is "
          f"speaker confusion alone")
    return n_ok > 0


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    ap.add_argument("--cond", nargs="+", required=True,
                    help="Stage 5 condition(s), e.g. indicconformer__pyannote31+rule")
    ap.add_argument("--data", type=Path, default=Path("data"))
    ap.add_argument("--limit", type=int)
    args = ap.parse_args()

    clean = True
    for c in args.cond:
        clean &= run(c, args.data, args.limit)
    return 0 if clean else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
corrected = sorted(p.name for p in (DATA / "attrib").iterdir() if p.is_dir() and "+" in p.name)
run("stage 5b: project to RTTM", ["stage5_to_rttm.py", "--data", "data", "--cond", *corrected, *BASE])

# No --diagnostic here: it only prints, after the summaries are written, and on
# 28 systems it is the slowest CPU step in the notebook.
systems = sorted(p.name for p in (DATA / "hyp").iterdir() if p.is_dir())
run("stage 5b: score DER/JER", ["stage3_score.py", "--data", "data", "--systems", *systems])

## Stage 6 — results table: baseline vs improved

Regroups the per-clip error counts from Stages 3–5; runs no model. Before writing, it
refuses to proceed unless WER is identical across relabelling methods on every clip,
missed speech and false alarm are identical across baseline, control and corrected
RTTMs, every `ic_lid_fallback` row reproduces the system it took its words from, and
every corpus figure re-derives to the Stage 3 and Stage 4 summaries.

Writes `data/results/results_table.md`, `results_per_video.csv` and
`results_per_video.xlsx`.

In [ ]:
%%writefile stage6_report.py
#!/usr/bin/env python3
"""
Stage 6 -- the results table: baseline vs improved, per model, per video.

    python stage6_report.py --data data

Nothing here runs a model. Every number is a regrouping of per-clip error
counts that Stages 3-5 already wrote to data/results/, so this reruns in
seconds whenever anything upstream changes -- and it cannot quietly disagree
with the stage that produced a number, because it re-derives the corpus figures
and checks them against the Stage 3 and Stage 4 summaries before writing.

Writes to data/results/:
    results_per_video.csv    long: one row per clip x ASR x diarizer x method
    results_per_video.xlsx   summaries, the improvement track per video, and
                             one wide sheet per ASR x diarizer
    results_table.md         the improvement track, corpus table, consistency,
                             breakdowns, and the per-video table


MODEL AND METHOD
----------------
A "model" is an ASR x diarizer pair. A "method" is what ran on top of it:
    baseline   Stage 4 output: ASR words assigned to the diarizer's turns
    rule       Stage 5 heuristic: a sub-second unit takes its nearer neighbour
    llm        Stage 5 Qwen2.5-7B relabel (IndicConformer x pyannote31 only)

ASR systems:
    indicconformer        language-locked decode of the multisoftmax CTC head
    indicconformer_free   naive argmax over every language head -- the ablation
    whisper               faster-whisper large-v3, greedy
    ic_lid_fallback       indicconformer, or whisper on clips where
                          IndicConformer's language ID falls outside the served
                          languages (stage4_fallback.py)


THE IMPROVEMENT TRACK
---------------------
The goal is an improvement on top of the best benchmarked combination,
IndicConformer x pyannote31. TRACK lists the systems built on it, in the order
they were tried, so every table leads with the same five rows.


WHY THERE IS A SECOND DER COLUMN
--------------------------------
Stage 5 relabels words; DER is measured over time. stage5_to_rttm.py projects
word labels back onto the diarizer's own turns, and that projection is lossy on
its own: with ZERO edits it relabels 17% of pyannote31 turns under IndicConformer
words and 31% under Whisper words.

So every model carries two:
    DER        baseline: the diarizer's RTTM exactly as produced
               rule/llm: the relabelled RTTM
    DER_ctrl   the same round trip with no edits -- projection, no correction

A method's DER effect is DER - DER_ctrl. Comparing against the baseline DER
instead would charge the method for the projection. Boundaries never move, so
missed speech and false alarm are identical across all three; that is asserted.

An ASR change (decode, fallback) cannot move the raw diarizer's DER at all: the
diarizer never sees the words.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import pandas as pd

NAIVE, LOCKED, WHISPER = "indicconformer_free", "indicconformer", "whisper"
FALLBACK = "ic_lid_fallback"

ASR_ORDER = [LOCKED, FALLBACK, NAIVE, WHISPER]
DIAR_ORDER = ["pyannote31", "sortformer_stream", "sortformer"]
METHOD_ORDER = ["baseline", "rule", "llm"]
BEST_DIAR = "pyannote31"

TRACK = [
    (LOCKED, "baseline", "IC"),
    (LOCKED, "rule", "IC +rule"),
    (LOCKED, "llm", "IC +llm"),
    (FALLBACK, "baseline", "IC +LID fallback"),
    (FALLBACK, "rule", "IC +LID fallback +rule"),
]

# Excel caps sheet names at 31 characters; `indicconformer_free__sortformer_stream`
# is 38.
SHORT = {LOCKED: "IC", NAIVE: "IC-naive", WHISPER: "Whisper", FALLBACK: "IC-fb",
         "pyannote31": "pyannote", "sortformer": "sortformer",
         "sortformer_stream": "sf-stream"}

KEY = ["asr", "diar", "method", "clip_id"]
ASR_COLS = ["wer_errors", "wer_len", "cp_errors", "cp_len",
            "wder_errors", "wder_len"]
DIA_COLS = ["der", "jer", "err_miss_sec", "err_fa_sec", "err_conf_sec",
            "ref_speech_sec", "hyp_missing"]
EDIT_COLS = ["n_units", "n_applied", "n_words_relabelled", "rogue"]

# Missed speech / false alarm may differ by RTTM rounding (3 decimal places),
# never by more. Anything larger means a boundary moved.
BOUNDARY_TOL_SEC = 0.01


def rate(errors: pd.Series, length: pd.Series) -> float:
    """Error-weighted rate in percent: total errors over total reference."""
    n = length.sum()
    return 100.0 * errors.sum() / n if n else float("nan")


def der_of(g: pd.DataFrame, sfx: str = "") -> float:
    return rate(g[f"err_miss_sec{sfx}"] + g[f"err_fa_sec{sfx}"]
                + g[f"err_conf_sec{sfx}"], g[f"ref_speech_sec{sfx}"])


def pick(df: pd.DataFrame, asr: str, method: str, diar: str | None = None):
    m = (df.asr == asr) & (df.method == method)
    if diar is not None:
        m &= df.diar == diar
    return df[m]


def ordered(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by ASR, diarizer, method in reading order rather than alphabet."""
    df = df.copy()
    rank = {c: {v: i for i, v in enumerate(o)} for c, o in
            (("asr", ASR_ORDER), ("diar", DIAR_ORDER), ("method", METHOD_ORDER))}
    cols = [c for c in ("asr", "diar", "method") if c in df]
    for c in cols:
        df[f"_{c}"] = df[c].map(rank[c]).fillna(99)
    by = [f"_{c}" for c in cols] + (["clip_id"] if "clip_id" in df else [])
    return (df.sort_values(by).drop(columns=[f"_{c}" for c in cols])
              .reset_index(drop=True))


def read_manifest(path: Path) -> dict[str, dict]:
    """Last record per clip wins, matching the append-only manifests."""
    last = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            last[rec["clip_id"]] = rec
    return last


# --------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------

def load_meta(data: Path) -> pd.DataFrame:
    meta = pd.read_csv(data / "ref" / "clip_meta.csv")
    meta = meta[meta.has_audio.astype(str).str.lower() == "true"].copy()
    meta["speakers"] = pd.Categorical(
        meta.n_speakers.map(lambda n: "5+" if n >= 5 else str(int(n))),
        ["2", "3", "4", "5+"], ordered=True)
    # Terciles over clips, not over rows of the long table: every clip counts
    # once regardless of how many models scored it.
    _, edges = pd.qcut(meta.overlap_frac_of_speech, 3, retbins=True)
    e1, e2 = edges[1], edges[2]
    meta["overlap"] = pd.qcut(
        meta.overlap_frac_of_speech, 3,
        labels=[f"low (<={e1:.1%})", f"mid ({e1:.1%}-{e2:.1%}]",
                f"high (>{e2:.1%})"])
    return meta


def load_edits(data: Path, models: pd.DataFrame) -> pd.DataFrame:
    """Per-clip edit counts from the Stage 5 manifests."""
    rows = []
    for a, d, m in models[models.method != "baseline"].itertuples(index=False):
        path = data / "attrib" / f"{a}__{d}+{m}" / "manifest.jsonl"
        if not path.is_file():
            continue
        for rec in read_manifest(path).values():
            if rec.get("status") == "ok":
                rows.append({"asr": a, "diar": d, "method": m,
                             "clip_id": rec["clip_id"],
                             **{c: rec.get(c) for c in EDIT_COLS}})
    return pd.DataFrame(rows, columns=KEY + EDIT_COLS)


def build_long(data: Path) -> pd.DataFrame:
    res = data / "results"
    meta = load_meta(data)

    asr = pd.read_csv(res / "asr_per_clip.csv")
    # The `ref` condition assigns words using the REFERENCE diarization. It is
    # an oracle diagnostic, never a system result, so it has no row here.
    asr = asr[asr.oracle.astype(str).str.lower() != "true"].copy()
    parts = asr.diar.str.split("+", n=1, expand=True)
    asr["diar"] = parts[0]
    asr["method"] = parts[1].fillna("baseline") if 1 in parts else "baseline"

    # Every model x every clip, not just the clips a model produced words for:
    # Sortformer emitted nothing on 25 clips, and "per model per video" should
    # show those as scored-as-total-miss rows rather than silently omit them.
    models = asr[["asr", "diar", "method"]].drop_duplicates()
    df = models.merge(meta, how="cross")
    df = df.merge(asr[KEY + ASR_COLS], on=KEY, how="left")

    df["sys"] = [d if m == "baseline" else f"{a}__{d}+{m}"
                 for a, d, m in zip(df.asr, df.diar, df.method)]
    df["sys_ctrl"] = df.asr + "__" + df.diar

    dia = pd.read_csv(res / "diarization_per_clip.csv")
    missing = sorted((set(df.sys) | set(df.sys_ctrl)) - set(dia.system))
    if missing:
        raise SystemExit(
            f"no diarization scores for {missing}.\n"
            f"Run stage5_to_rttm.py for the Stage 5 conditions and their bare "
            f"`asr__diar` controls, then stage3_score.py over every system in "
            f"data/hyp, then rerun this.")
    dia = dia.set_index(["system", "clip_id"])[DIA_COLS]
    df = df.join(dia, on=["sys", "clip_id"])
    df = df.join(dia.add_suffix("_ctrl"), on=["sys_ctrl", "clip_id"])

    df["WER"] = 100 * df.wer_errors / df.wer_len
    df["cpWER"] = 100 * df.cp_errors / df.cp_len
    df["WDER"] = 100 * df.wder_errors / df.wder_len
    df["DER"] = 100 * df.der
    df["JER"] = 100 * df.jer
    df["DER_ctrl"] = 100 * df.der_ctrl
    df["JER_ctrl"] = 100 * df.jer_ctrl

    df = df.merge(load_edits(data, models), on=KEY, how="left")
    return ordered(df)


# --------------------------------------------------------------------------
# Checks -- run before anything is written
# --------------------------------------------------------------------------

def check_invariants(df: pd.DataFrame) -> float:
    # Stage 5 may only relabel. If WER differs between methods on any clip,
    # text changed, and every number in that row is void.
    scored = df.dropna(subset=["wer_len"])
    n = scored.groupby(["asr", "diar", "clip_id"])[["wer_errors", "wer_len"]].nunique()
    bad = n[(n > 1).any(axis=1)]
    if len(bad):
        raise SystemExit(f"TEXT INVARIANT BROKEN on {len(bad)} clip(s):\n{bad.head()}")

    # Boundaries never move, so miss/FA must match the raw diarizer in both
    # the corrected RTTM and the control.
    base = (df[df.method == "baseline"]
            .set_index(["asr", "diar", "clip_id"])[["err_miss_sec", "err_fa_sec"]]
            .add_suffix("_base"))
    j = df.join(base, on=["asr", "diar", "clip_id"])
    dev = max((j[f"{c}{s}"] - j[f"{c}_base"]).abs().max()
              for c in ("err_miss_sec", "err_fa_sec") for s in ("", "_ctrl"))
    if dev > BOUNDARY_TOL_SEC:
        raise SystemExit(f"BOUNDARY INVARIANT BROKEN: miss/FA moved by {dev:.3f} s")
    return float(dev)


def check_fallback(df: pd.DataFrame, data: Path) -> tuple[int, int] | None:
    """Every fallback row must score EXACTLY like the system it took words from.

    The fallback only chooses between two existing outputs, and attribution and
    the rule are deterministic, so a switched clip must reproduce Whisper's
    per-clip numbers and a kept clip IndicConformer's -- in every metric, under
    every diarizer and method, including the relabelled RTTMs. Any difference
    means something other than the switch moved the score.
    """
    path = data / "asr" / FALLBACK / "manifest.jsonl"
    if FALLBACK not in set(df.asr) or not path.is_file():
        return None
    source = {c: r["source"] for c, r in read_manifest(path).items()}
    cols = ASR_COLS + ["der", "jer", "der_ctrl", "jer_ctrl"]
    idx = df.set_index(KEY)
    n_rows = 0
    for r in df[df.asr == FALLBACK].itertuples(index=False):
        twin = (source[r.clip_id], r.diar, r.method, r.clip_id)
        if twin not in idx.index:
            continue
        t = idx.loc[twin]
        for c in cols:
            mine, theirs = getattr(r, c), t[c]
            if pd.isna(mine) and pd.isna(theirs):
                continue
            if pd.isna(mine) or pd.isna(theirs) or abs(mine - theirs) > 1e-9:
                raise SystemExit(
                    f"FALLBACK INVARIANT BROKEN: {r.clip_id} {r.diar} {r.method} "
                    f"{c}: {mine} here vs {theirs} in {source[r.clip_id]}")
        n_rows += 1
    return n_rows, sum(s != LOCKED for s in source.values())


def check_against_upstream(corp: pd.DataFrame, res: Path) -> None:
    """Re-derived corpus figures must equal what Stages 3 and 4 reported."""
    asum = pd.read_csv(res / "asr_summary.csv")
    asum = asum[(asum.subset == "all") & ~asum.diar.str.contains("oracle")]
    got = corp.assign(diar_full=[d if m == "baseline" else f"{d}+{m}"
                                 for d, m in zip(corp.diar, corp.method)])
    m = asum.rename(columns={"diar": "diar_full"}).merge(
        got, on=["asr", "diar_full"], suffixes=("_up", ""))
    if len(m) != len(asum):
        raise SystemExit(f"{len(asum) - len(m)} Stage 4 summary row(s) have no "
                         f"counterpart here")
    worst = max((m[f"{c}_up"] - m[c].round(2)).abs().max()
                for c in ("WER", "cpWER", "WDER"))
    if worst > 0.011:
        raise SystemExit(f"ASR figures disagree with asr_summary.csv by {worst:.3f}")

    dsum = pd.read_csv(res / "diarization_summary.csv").set_index("system").DER * 100
    worst = max((corp.DER - corp.sys.map(dsum)).abs().max(),
                (corp.DER_ctrl - corp.sys_ctrl.map(dsum)).abs().max())
    if not worst < 1e-6:
        raise SystemExit(f"DER disagrees with diarization_summary.csv by {worst}")


# --------------------------------------------------------------------------
# Tables
# --------------------------------------------------------------------------

def corpus_table(df: pd.DataFrame, res: Path) -> pd.DataFrame:
    jer = pd.read_csv(res / "diarization_summary.csv").set_index("system")
    jer = jer.JER_pyannote_accum * 100
    rows = []
    for (a, d, m), g in df.groupby(["asr", "diar", "method"], sort=False):
        rows.append({
            "asr": a, "diar": d, "method": m,
            "sys": g.sys.iloc[0], "sys_ctrl": g.sys_ctrl.iloc[0],
            "clips": int(g.wer_len.notna().sum()),
            "WER": rate(g.wer_errors, g.wer_len),
            "cpWER": rate(g.cp_errors, g.cp_len),
            "WDER": rate(g.wder_errors, g.wder_len),
            # DER over all 99 clips, hypothesis-less ones scored as total
            # miss -- the same convention as the Stage 3 table.
            "DER": der_of(g),
            "JER": jer[g.sys.iloc[0]],
            "DER_ctrl": der_of(g, "_ctrl"),
            "JER_ctrl": jer[g.sys_ctrl.iloc[0]],
        })
    out = ordered(pd.DataFrame(rows))
    base = out[out.method == "baseline"].set_index(["asr", "diar"])
    idx = pd.MultiIndex.from_frame(out[["asr", "diar"]])
    is_base = out.method == "baseline"
    for c in ("cpWER", "WDER"):
        out[f"d_{c}"] = (out[c].values - base[c].reindex(idx).values)
        out.loc[is_base, f"d_{c}"] = float("nan")
    out["d_DER_vs_ctrl"] = (out.DER - out.DER_ctrl).where(~is_base)
    # On the baseline row, control minus raw diarizer is the price of the
    # projection itself, before any correction.
    out["projection_cost"] = (out.DER_ctrl - out.DER).where(is_base)
    return out


def track_table(corp: pd.DataFrame) -> pd.DataFrame:
    """The improvement track on every diarizer, deltas against plain IC."""
    rows = []
    for d in DIAR_ORDER:
        base = pick(corp, TRACK[0][0], TRACK[0][1], d)
        if base.empty:
            continue
        b = base.iloc[0]
        for i, (a, m, label) in enumerate(TRACK):
            r = pick(corp, a, m, d)
            if r.empty:
                continue
            r = r.iloc[0]
            first = i == 0
            rows.append({
                "diar": d, "system": label, "clips": r.clips,
                "WER": r.WER, "cpWER": r.cpWER, "WDER": r.WDER,
                "d_WER": None if first else r.WER - b.WER,
                "d_cpWER": None if first else r.cpWER - b.cpWER,
                "d_WDER": None if first else r.WDER - b.WDER,
                "DER": r.DER, "JER": r.JER, "DER_ctrl": r.DER_ctrl,
                "d_DER_vs_ctrl": None if m == "baseline" else r.DER - r.DER_ctrl,
            })
    return pd.DataFrame(rows)


def comparisons(df: pd.DataFrame) -> list[tuple[str, str, str, str]]:
    """(new asr, new method, old asr, old method) pairs worth a win/loss count."""
    have = set(map(tuple, df[["asr", "method"]].drop_duplicates().values))
    out = [(LOCKED, "baseline", NAIVE, "baseline"),
           (FALLBACK, "baseline", LOCKED, "baseline"),
           (FALLBACK, "rule", LOCKED, "baseline")]
    out = [c for c in out if c[:2] in have and c[2:] in have]
    for a in ASR_ORDER:
        for m in METHOD_ORDER[1:]:
            if (a, m) in have:
                out.append((a, m, a, "baseline"))
    return out


def consistency_table(df: pd.DataFrame) -> pd.DataFrame:
    """Per clip: did the change make it better, leave it, or make it worse?

    Same-ASR changes (a Stage 5 method) are compared on DER against their own
    projection control. Cross-ASR changes cannot move the raw diarizer's DER,
    so they get no DER columns rather than a column of trivial "same".
    """
    idx = df.set_index(KEY)
    rows = []
    for a_new, m_new, a_old, m_old in comparisons(df):
        label = (f"{a_new}: {m_old} -> {m_new}" if a_new == a_old
                 else f"{a_old} -> {a_new}" + ("" if m_new == "baseline" else f" +{m_new}"))
        for d in DIAR_ORDER:
            new = pick(df, a_new, m_new, d)
            new = new[new.wer_len.notna()]      # clips the change actually ran on
            keys = [(a_old, d, m_old, c) for c in new.clip_id]
            if new.empty or not all(k in idx.index for k in keys):
                continue
            old = idx.loc[keys]
            row = {"change": label, "diar": d, "clips": len(new)}
            pairs = [("WER", new.WER.values, old.WER.values),
                     ("cpWER", new.cpWER.values, old.cpWER.values),
                     ("WDER", new.WDER.values, old.WDER.values)]
            if a_new == a_old:
                pairs.append(("DER vs ctrl", new.DER.values, new.DER_ctrl.values))
            for name, nv, ov in pairs:
                delta = pd.Series(nv - ov)
                row[f"{name}: better"] = int((delta < -1e-9).sum())
                row[f"{name}: same"] = int((delta.abs() <= 1e-9).sum())
                row[f"{name}: worse"] = int((delta > 1e-9).sum())
            rows.append(row)
    return pd.DataFrame(rows)


def asr_change_table(corp: pd.DataFrame, old: str, new: str) -> pd.DataFrame:
    """One ASR swapped for another under every diarizer, baseline method."""
    b = corp[corp.method == "baseline"].set_index(["asr", "diar"])
    rows = []
    for d in DIAR_ORDER:
        if (old, d) not in b.index or (new, d) not in b.index:
            continue
        o, n = b.loc[(old, d)], b.loc[(new, d)]
        row = {"diar": d, "clips": int(n.clips)}
        for c in ("WER", "cpWER", "WDER"):
            row[f"{c} {SHORT[old]}"] = o[c]
            row[f"{c} {SHORT[new]}"] = n[c]
            row[f"d_{c}"] = n[c] - o[c]
        rows.append(row)
    return pd.DataFrame(rows)


def track_breakdown(df: pd.DataFrame, by: str) -> pd.DataFrame:
    """The improvement track on the best diarizer, grouped by one clip property."""
    h = df[df.diar == BEST_DIAR]
    rows = []
    for key, g in h.groupby(by, observed=True, sort=True):
        ic = pick(g, TRACK[0][0], TRACK[0][1])
        row = {by: key, "clips": ic.clip_id.nunique(),
               "hours": ic.duration.sum() / 3600}
        steps = [(label, pick(g, a, m)) for a, m, label in TRACK]
        steps = [(label, s) for label, s in steps if len(s)]
        for label, s in steps:
            if s.method.iloc[0] == "baseline":
                row[f"WER {label}"] = rate(s.wer_errors, s.wer_len)
        for metric, (e, n) in (("cpWER", ("cp_errors", "cp_len")),
                               ("WDER", ("wder_errors", "wder_len"))):
            for label, s in steps:
                row[f"{metric} {label}"] = rate(s[e], s[n])
        row["DER"] = der_of(ic)
        row["JER (clip mean)"] = ic.JER.mean()
        rows.append(row)
    out = pd.DataFrame(rows)
    if by == "language":
        out = out.sort_values(["clips", "language"], ascending=[False, True])
    return out.reset_index(drop=True)


def asr_by_language(df: pd.DataFrame) -> pd.DataFrame:
    """WER per language for each ASR. WER ignores speakers, so any diarizer
    gives the same number; pyannote31 is used because it covers all 99 clips."""
    b = df[(df.method == "baseline") & (df.diar == BEST_DIAR)]
    rows = []
    for lang, g in b.groupby("language"):
        row = {"language": lang, "clips": g.clip_id.nunique()}
        for a in ASR_ORDER:
            s = g[g.asr == a]
            if len(s):
                row[f"WER {a}"] = rate(s.wer_errors, s.wer_len)
        if f"WER {NAIVE}" in row and f"WER {LOCKED}" in row:
            row["decode gain"] = row[f"WER {NAIVE}"] - row[f"WER {LOCKED}"]
        if f"WER {FALLBACK}" in row and f"WER {LOCKED}" in row:
            row["fallback gain"] = row[f"WER {LOCKED}"] - row[f"WER {FALLBACK}"]
        rows.append(row)
    return (pd.DataFrame(rows)
              .sort_values(["clips", "language"], ascending=[False, True])
              .reset_index(drop=True))


def track_per_video(df: pd.DataFrame, diar: str = BEST_DIAR) -> pd.DataFrame:
    """The improvement track, one row per clip."""
    ic = pick(df, TRACK[0][0], TRACK[0][1], diar).set_index("clip_id")
    out = ic[["language", "duration", "n_speakers", "hyp_missing"]].copy()
    out["overlap_%"] = 100 * ic.overlap_frac_of_speech
    out["DER"] = ic.DER
    out["JER"] = ic.JER
    steps = [(label, m, pick(df, a, m, diar).set_index("clip_id"))
             for a, m, label in TRACK]
    steps = [(label, m, s) for label, m, s in steps if len(s)]
    for label, m, s in steps:
        if m == "baseline":
            out[f"WER {label}"] = s.WER
    for metric in ("cpWER", "WDER"):
        for label, m, s in steps:
            out[f"{metric} {label}"] = s[metric]
    for label, m, s in steps:
        if m == "baseline":
            out[f"DER ctrl {label}"] = s.DER_ctrl
        else:
            out[f"DER {label}"] = s.DER
            out[f"JER {label}"] = s.JER
    fb = [s for label, m, s in steps if label == "IC +LID fallback"]
    if fb:
        out["WER source"] = [LOCKED if a == b else WHISPER for a, b in
                             zip(fb[0].wer_errors.reindex(out.index), ic.wer_errors)]
    return out.reset_index()


def per_video(df: pd.DataFrame, a: str, d: str) -> pd.DataFrame:
    """One model, one row per clip, methods side by side."""
    g = df[(df.asr == a) & (df.diar == d)]
    methods = [m for m in METHOD_ORDER if m in set(g.method)]
    wide = g.pivot(index="clip_id", columns="method",
                   values=["DER", "JER", "cpWER", "WDER"])
    base = g[g.method == "baseline"].set_index("clip_id")

    out = base[["language", "duration", "n_speakers", "hyp_missing"]].copy()
    out["overlap_%"] = 100 * base.overlap_frac_of_speech
    out["WER"] = base.WER
    for metric in ("DER", "JER"):
        out[f"{metric} baseline"] = wide[(metric, "baseline")]
        out[f"{metric} ctrl"] = base[f"{metric}_ctrl"]
        for m in methods[1:]:
            out[f"{metric} {m}"] = wide[(metric, m)]
    for metric in ("cpWER", "WDER"):
        for m in methods:
            out[f"{metric} {m}"] = wide[(metric, m)]
    for m in methods[1:]:
        out[f"d_DER {m} vs ctrl"] = wide[("DER", m)] - base.DER_ctrl
        out[f"d_cpWER {m}"] = wide[("cpWER", m)] - wide[("cpWER", "baseline")]
        out[f"d_WDER {m}"] = wide[("WDER", m)] - wide[("WDER", "baseline")]
    return out.reset_index()


# --------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------

def md(df: pd.DataFrame) -> str:
    # NaN -> None so tabulate prints a dash instead of "nan".
    clean = df.astype(object).where(df.notna(), None)
    return clean.to_markdown(index=False, floatfmt=".2f", missingval="–")


CORPUS_COLS = ["asr", "diar", "method", "clips", "WER", "cpWER", "WDER",
               "d_cpWER", "d_WDER", "DER", "JER", "DER_ctrl", "JER_ctrl",
               "projection_cost", "d_DER_vs_ctrl"]

# The per-video table in the markdown is read, not filtered, so it carries the
# brief's metrics for each track step; DER/JER of relabelled RTTMs and the
# controls stay in the xlsx.
TRACK_MD_COLS = ["clip_id", "language", "duration", "n_speakers", "overlap_%",
                 "DER", "JER", "WER IC", "WER IC +LID fallback",
                 "cpWER IC", "cpWER IC +LID fallback",
                 "cpWER IC +LID fallback +rule", "WDER IC", "WDER IC +rule",
                 "WDER IC +llm", "WDER IC +LID fallback",
                 "WDER IC +LID fallback +rule"]


def write_markdown(path: Path, meta: pd.DataFrame, t: dict, max_dev: float,
                   fb_check: tuple[int, int] | None) -> None:
    hours = meta.duration.sum() / 3600
    ov = meta.overlap_sec.sum() / meta.speech_sec.sum()
    fb_line = ("" if fb_check is None else
               f" every `{FALLBACK}` row ({fb_check[0]:,} of them; "
               f"{fb_check[1]} clips switched) scores identically to the system "
               f"it took its words from;")
    L = [
        "# Results -- baseline vs improved",
        "",
        f"{len(meta)} clips, {hours:.2f} h, {meta.language.nunique()} scripts/"
        f"languages, {ov:.2%} of reference speech overlapped.",
        "",
        "**Metric policy.** DER/JER: `collar=0.0`, `skip_overlap=False`, UEM = "
        "full clip; overlap is scored. ASR rates are error-weighted (total "
        "errors / total reference words), DER is duration-weighted. JER is "
        "pyannote's accumulated JER in corpus tables and a per-clip mean in "
        "breakdowns.",
        "",
        "**Systems.** `indicconformer` (IC) decodes within the detected "
        "language's head; `indicconformer_free` takes a naive argmax over all "
        "heads; `whisper` is large-v3, greedy; `ic_lid_fallback` is IC, except "
        "on clips where IC's own language ID lands outside the ten served "
        "language codes, which take Whisper's words. `rule` and `llm` are the "
        "Stage 5 relabelling methods. The oracle (reference-diarization) "
        "condition is a diagnostic and is excluded.",
        "",
        "**DER_ctrl.** Stage 5 edits word labels; DER needs turns. Projecting "
        "words back onto the diarizer's turns with *no* edits already changes "
        "DER, so a method's DER effect is `DER - DER_ctrl`, and on a baseline "
        "row `projection_cost = DER_ctrl - DER` is the price of the projection "
        "alone. ASR changes never move the raw diarizer's DER.",
        "",
        "**Checks passed before writing:** WER identical across methods on every "
        f"clip (labels changed, text did not); missed speech and false alarm "
        f"identical across baseline, control and corrected RTTMs (max deviation "
        f"{max_dev:.3f} s);{fb_line} every corpus figure re-derived from "
        f"per-clip counts and matched against the Stage 3 and Stage 4 summaries.",
        "",
        "## 1. Baseline vs improved: the best combination and what was built on it",
        "",
        "Baseline is `IC` (IndicConformer x the diarizer). `d_*` are against it.",
        "",
        md(t["track"]),
        "",
        "## 2. ASR changes",
        "",
        "### Language-ID fallback: IC -> IC with Whisper on out-of-set clips",
        "",
        md(t["fallback"]),
        "",
        "### Decode: naive argmax -> language-locked",
        "",
        "Same model, same weights, same audio; only the decode differs.",
        "",
        md(t["decode"]),
        "",
        "### WER by language (pyannote31, baseline)",
        "",
        md(t["asr_lang"]),
        "",
        "## 3. Consistency: per-clip wins and losses",
        "",
        "Clips where the change lowered, left unchanged, or raised each metric. "
        "Stage 5 methods are compared on DER against their own projection "
        "control; ASR swaps cannot change DER and have no DER columns.",
        "",
        md(t["consistency"]),
        "",
        "## 4. Every model, every method",
        "",
        "Sortformer produced no output on 25 clips: its ASR metrics cover the "
        "74 it did, its DER counts the other 25 as total miss.",
        "",
        md(t["corpus"][CORPUS_COLS]),
        "",
        f"## 5. Where the improvements help and hurt (`{BEST_DIAR}`)",
        "",
        "### By language",
        "",
        md(t["by_language"]),
        "",
        "### By reference speaker count",
        "",
        md(t["by_speakers"]),
        "",
        "### By overlap (tercile of overlapped-speech fraction)",
        "",
        md(t["by_overlap"]),
        "",
        f"## 6. Per video: the improvement track (`{BEST_DIAR}`)",
        "",
        "Relabelled-RTTM DER/JER, projection controls, and every model's own "
        "per-video table are sheets in `results_per_video.xlsx`; all rows are in "
        "`results_per_video.csv`.",
        "",
        md(t["track_video"][[c for c in TRACK_MD_COLS if c in t["track_video"]]]),
        "",
    ]
    path.write_text("\n".join(L), encoding="utf-8")


def write_excel(path: Path, df: pd.DataFrame, t: dict) -> bool:
    try:
        import openpyxl  # noqa: F401
    except ImportError:
        print("[skip] openpyxl not installed -- no .xlsx (CSV and MD still written)")
        return False
    sheets = {"track": t["track"], f"track per video ({SHORT[BEST_DIAR]})":
              t["track_video"], "consistency": t["consistency"],
              "corpus": t["corpus"][CORPUS_COLS], "asr_fallback": t["fallback"],
              "asr_decode": t["decode"], "asr_by_language": t["asr_lang"],
              "by_language": t["by_language"], "by_speakers": t["by_speakers"],
              "by_overlap": t["by_overlap"]}
    for a, d in t["corpus"][["asr", "diar"]].drop_duplicates().itertuples(index=False):
        sheets[f"{SHORT.get(a, a)} x {SHORT.get(d, d)}"] = per_video(df, a, d)
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for name, table in sheets.items():
            table.round(2).to_excel(xw, sheet_name=name[:31], index=False)
    return True


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    ap.add_argument("--data", type=Path, default=Path("data"))
    args = ap.parse_args()
    res = args.data / "results"

    meta = load_meta(args.data)
    df = build_long(args.data)
    max_dev = check_invariants(df)
    fb_check = check_fallback(df, args.data)
    corp = corpus_table(df, res)
    check_against_upstream(corp, res)

    t = {
        "corpus": corp,
        "track": track_table(corp),
        "fallback": asr_change_table(corp, LOCKED, FALLBACK),
        "decode": asr_change_table(corp, NAIVE, LOCKED),
        "asr_lang": asr_by_language(df),
        "consistency": consistency_table(df),
        "by_language": track_breakdown(df, "language"),
        "by_speakers": track_breakdown(df, "speakers"),
        "by_overlap": track_breakdown(df, "overlap"),
        "track_video": track_per_video(df),
    }

    long_cols = (KEY + ["language", "duration", "n_speakers", "speakers",
                        "overlap_frac_of_speech", "overlap", "hyp_missing",
                        "WER", "cpWER", "WDER", "DER", "JER", "DER_ctrl",
                        "JER_ctrl"] + EDIT_COLS + ASR_COLS
                 + ["err_miss_sec", "err_fa_sec", "err_conf_sec",
                    "ref_speech_sec", "sys", "sys_ctrl"])
    df[long_cols].to_csv(res / "results_per_video.csv", index=False,
                         encoding="utf-8", float_format="%.4f")
    write_markdown(res / "results_table.md", meta, t, max_dev, fb_check)
    xlsx = write_excel(res / "results_per_video.xlsx", df, t)

    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 40)
    print("=" * 78)
    print("STAGE 6 -- BASELINE VS IMPROVED")
    print("=" * 78)
    print(t["track"].round(2).to_string(index=False))
    print("\nLanguage-ID fallback:")
    print(t["fallback"].round(2).to_string(index=False))
    print("\nPer-clip consistency:")
    print(t["consistency"].to_string(index=False))
    print(f"\nchecks: text invariant ok; boundary deviation {max_dev:.3f} s; "
          + ("" if fb_check is None else
             f"fallback rows match their source ({fb_check[0]:,} rows, "
             f"{fb_check[1]} clips switched); ")
          + "corpus figures match Stage 3/4 summaries")
    print(f"wrote -> {res / 'results_per_video.csv'}  ({len(df):,} rows)")
    print(f"wrote -> {res / 'results_table.md'}")
    if xlsx:
        print(f"wrote -> {res / 'results_per_video.xlsx'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
run("stage 6: results table", ["stage6_report.py", "--data", "data"])

In [ ]:
from IPython.display import Markdown, display

table = (DATA / "results" / "results_table.md").read_text(encoding="utf-8")
# The headline section: the best combination and what was built on it.
display(Markdown(table.split("## 2.")[0]))

In [ ]:
# Smoke mode: the CPU stages ran on exactly the reference run's GPU outputs, so
# every table must match the committed one. A full GPU run is compared, not asserted.
run("check: results vs committed tables",
    ["nb_checks.py", "expected", "--data", "data", "--expected", EXPECTED]
    + ([] if FULL_GPU_RUN else ["--strict"]))

## Run report

Every step this session ran, skipped or failed. In smoke mode a failed GPU stage
does not invalidate the tables above — they are built from the cached full outputs —
but it does mean that stage's code was not demonstrated in this environment.

In [ ]:
width = max(len(r["step"]) for r in REPORT)
print(f"{'step':<{width}}  {'status':<42} {'exit':>4} {'min':>6}")
for r in REPORT:
    ex = "" if r["exit"] is None else r["exit"]
    print(f"{r['step']:<{width}}  {r['status']:<42} {ex:>4} {r['min']:6.1f}")